# IFMIR medication-error incident-report dataset: audit and preparation
Run cells in order in Google Colab. This notebook audits the uploaded workbooks, validates offsets, prepares report-level JSONL files, and runs a modest TF-IDF report-classification baseline only if a suitable report-level target is present.

**Important:** this corpus is incident reports, not MIMIC EHR notes. Do not retain MIMIC cohort descriptions or old performance figures. The CrossValid workbook may already have a prescribed role/fold structure; the random split below is exploratory only until the dataset protocol is checked. No model-performance claims are fabricated.

## 1. Install and import

In [1]:
!pip -q install pandas openpyxl scikit-learn matplotlib
import os,re,json,zipfile,hashlib
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
pd.set_option("display.max_columns",100)
print("Ready")

Ready


## 2. Upload and extract `23504922.zip` (or upload individual XLSX files)

In [2]:
from google.colab import files
ROOT=Path("/content/med_error_data"); ROOT.mkdir(exist_ok=True)
zip_path=Path("/content/23504922.zip")
if not zip_path.exists():
    up=files.upload()
    z=[k for k in up if k.lower().endswith(".zip")]
    if z: zip_path=Path(z[0])
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as f: f.extractall(ROOT)
else:
    for k in up:
        if k.lower().endswith(".xlsx"):
            import shutil; shutil.copy2(k,ROOT/Path(k).name)
workbook_paths=sorted(ROOT.rglob("*.xlsx"))
if not workbook_paths: raise FileNotFoundError("No ZIP/XLSX found. Upload 23504922.zip or its Excel files.")
print("\n".join(str(x) for x in workbook_paths))

Saving AnnotatedDataset.xlsx to AnnotatedDataset.xlsx
/content/med_error_data/AnnotatedDataset.xlsx


## 3. Load all sheets and inspect schema

In [3]:
books={}
for path in workbook_paths:
    for sh in pd.ExcelFile(path).sheet_names:
        df=pd.read_excel(path,sheet_name=sh)
        books[(path.name,sh)]=df
        print(f"\n{path.name} | {sh} | {df.shape}\n{list(df.columns)}")


AnnotatedDataset.xlsx | Sheet1 | (478176, 13)
['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']


## 4. Identify CrossValid workbook and normalize columns

In [4]:
def norm(x): return re.sub(r"[^a-z0-9]","",str(x).lower())
def role(k):
    s=(" ".join(k)).lower()
    if "crossvalid" in s: return "crossvalid"
    if "manual" in s or "goldstandard" in s: return "manual"
    if "error" in s: return "error"
    return "other"
inventory=pd.DataFrame([{"file":k[0],"sheet":k[1],"role":role(k),"rows":len(v),"columns":len(v.columns)} for k,v in books.items()])
display(inventory)
keys=[k for k in books if role(k)=="crossvalid"]
if not keys: raise ValueError("No CrossValid workbook found; inspect inventory and set CROSS_KEY manually.")
CROSS_KEY=keys[0]; raw=books[CROSS_KEY].copy()
def col(aliases):
    d={norm(c):c for c in raw.columns}
    return next((d[norm(a)] for a in aliases if norm(a) in d),None)
C={"id":col(["id","report_id"]),"text":col(["report","reports","original_report"]),
   "translation":col(["translated_report","english_report"]),"entity":col(["namedentity","named_entity"]),
   "translation_entity":col(["translated_namedentity","translated_entity"]),
   "start":col(["startidx","start_idx","start"]),"end":col(["endidx","end_idx","end"]),
   "type":col(["entitytype","entity_type","label"]),"error":col(["errorlabel","error_label"])}
print("Selected",CROSS_KEY,"\nColumn map",C)
need=["id","text","entity","start","end","type"]
if any(C[x] is None for x in need): raise ValueError("Required columns missing. Inspect printed schema and amend C.")

,file,sheet,role,rows,columns
0,AnnotatedDataset.xlsx,Sheet1,other,478176,13


ValueError: No CrossValid workbook found; inspect inventory and set CROSS_KEY manually.

In [5]:

# Select the uploaded annotated dataset for inspection

print("Available workbook keys:")
for k in books:
    print(k, books[k].shape, list(books[k].columns))

# Select the large annotated dataset
CROSS_KEY = next(
    (k for k in books if "AnnotatedDataset.xlsx" in str(k)),
    None
)

if CROSS_KEY is None:
    raise ValueError(
        "AnnotatedDataset.xlsx was not found. "
        "Check the available workbook keys above."
    )

raw = books[CROSS_KEY].copy()

print("\nSelected workbook:", CROSS_KEY)
print("Shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())

print("\nFirst 5 rows:")
display(raw.head())

print("\nData types:")
print(raw.dtypes)

Available workbook keys:
('AnnotatedDataset.xlsx', 'Sheet1') (478176, 13) ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']

Selected workbook: ('AnnotatedDataset.xlsx', 'Sheet1')
Shape: (478176, 13)

Columns:
['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']

First 5 rows:


,no,id,year,report,namedentity,startidx,endidx,entitytype,errorlabel,relation,incidenttypes,translated_report,translated_namedentity
0,0,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ノルアドレナリン,1,9,Drug,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,Noradrenaline
1,1,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,0.1,12,15,Strength_amount,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,0.1
2,2,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,3ml/h,18,23,Strength_rate,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,3ml/h
3,3,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,13:55,46,51,Timing,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,13:55
4,4,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,シ,52,53,Route,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,Si



Data types:
no                          int64
id                         object
year                        int64
report                     object
namedentity                object
startidx                    int64
endidx                      int64
entitytype                 object
errorlabel                 object
relation                  float64
incidenttypes              object
translated_report          object
translated_namedentity     object
dtype: object


## 5. Dataset counts, missingness and labels

In [7]:


# CELL 5 — Dataset inventory and annotation audit

import pandas as pd
import numpy as np

# Use the already loaded AnnotatedDataset
df = raw.copy()

# Explicit column mapping based on the actual workbook
C = {
    "id": "id",
    "year": "year",
    "text": "report",
    "entity": "namedentity",
    "start": "startidx",
    "end": "endidx",
    "type": "entitytype",
    "error": "errorlabel",
    "relation": "relation",
    "incident": "incidenttypes",
    "translation": "translated_report",
    "translation_entity": "translated_namedentity"
}

# Check required columns
required = list(C.values())
missing_columns = [col for col in required if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# 1. Dataset size
ids = df[C["id"]].astype(str).str.strip()

print("=" * 60)
print("1. DATASET SIZE")
print("=" * 60)
print("Annotation rows:", len(df))
print("Unique report IDs:", ids.nunique())
print("Unique original report texts:", df[C["text"]].nunique())
print("Unique translated report texts:", df[C["translation"]].nunique())

# 2. Missing values
print("\n2. MISSING VALUES")
display(df.isna().sum().rename("missing").to_frame())

# 3. Entity type distribution
print("\n3. ENTITY TYPE DISTRIBUTION")
label_counts = (
    df[C["type"]]
    .fillna("[MISSING]")
    .astype(str)
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="annotation_rows")
)
display(label_counts)

# 4. Error label distribution
print("\n4. ERROR LABEL DISTRIBUTION")
display(
    df[C["error"]]
    .fillna("[MISSING]")
    .astype(str)
    .value_counts()
    .rename_axis("error_label")
    .reset_index(name="annotation_rows")
)

# 5. Incident type distribution
print("\n5. INCIDENT TYPE DISTRIBUTION")
display(
    df[C["incident"]]
    .fillna("[MISSING]")
    .astype(str)
    .value_counts()
    .head(50)
    .rename_axis("incident_type")
    .reset_index(name="annotation_rows")
)

# 6. Relation distribution
print("\n6. RELATION DISTRIBUTION")
display(
    df[C["relation"]]
    .fillna("[MISSING]")
    .value_counts()
    .rename_axis("relation")
    .reset_index(name="annotation_rows")
)

# 7. Annotations per report
print("\n7. ANNOTATIONS PER REPORT")
annotations_per_report = df.groupby(C["id"]).size()
display(annotations_per_report.describe().to_frame("annotations_per_report"))

print("Reports with one annotation:", (annotations_per_report == 1).sum())
print("Reports with more than 10 annotations:",
      (annotations_per_report > 10).sum())

# 8. Report ID consistency
print("\n8. REPORT ID CONSISTENCY")
consistency = df.groupby(C["id"]).agg(
    unique_original_texts=(C["text"], "nunique"),
    unique_translated_texts=(C["translation"], "nunique"),
    unique_years=(C["year"], "nunique")
)

print("IDs with multiple original texts:",
      (consistency["unique_original_texts"] > 1).sum())
print("IDs with multiple translated texts:",
      (consistency["unique_translated_texts"] > 1).sum())
print("IDs with multiple years:",
      (consistency["unique_years"] > 1).sum())

# 9. Duplicate annotations
print("\n9. DUPLICATE ANNOTATIONS")
duplicate_columns = [
    C["id"], C["entity"], C["start"], C["end"],
    C["type"], C["error"], C["relation"], C["incident"]
]
print("Exact duplicate annotation rows:",
      df.duplicated(subset=duplicate_columns).sum())

# 10. Offset validation against original Japanese text
print("\n10. OFFSET VALIDATION — ORIGINAL TEXT")

original_match = (
    df[C["text"]].fillna("").astype(str).str.slice(
        df[C["start"]], df[C["end"]]
    )
    == df[C["entity"]].fillna("").astype(str)
)

print("Exact span matches:", int(original_match.sum()))
print("Total rows:", len(df))
print("Exact match percentage:",
      round(original_match.mean() * 100, 3), "%")
print("Mismatches:", int((~original_match).sum()))

if (~original_match).any():
    print("\nFirst 10 mismatches:")
    display(df.loc[~original_match, [
        C["id"], C["text"], C["entity"],
        C["start"], C["end"], C["type"]
    ]].head(10))

# 11. Offset validation against translated English text
print("\n11. OFFSET VALIDATION — TRANSLATED TEXT")

translated_match = (
    df[C["translation"]].fillna("").astype(str).str.slice(
        df[C["start"]], df[C["end"]]
    )
    == df[C["translation_entity"]].fillna("").astype(str)
)

print("Exact translated span matches:",
      int(translated_match.sum()))
print("Translated match percentage:",
      round(translated_match.mean() * 100, 3), "%")

print("\nAUDIT CELL COMPLETED")

1. DATASET SIZE
Annotation rows: 478176
Unique report IDs: 58389
Unique original report texts: 58340
Unique translated report texts: 58315

2. MISSING VALUES


,missing
no,0
id,0
year,0
report,0
namedentity,0
startidx,0
endidx,0
entitytype,0
errorlabel,71909
relation,776



3. ENTITY TYPE DISTRIBUTION


,entity_type,annotation_rows
0,Drug,168633
1,Timing,90986
2,Strength_amount,59493
3,Date,40617
4,Route,36496
5,Form_form,22859
6,Dosage,22731
7,Duration,15127
8,Strength_rate,8084
9,Frequency,7289



4. ERROR LABEL DISTRIBUTION


,error_label,annotation_rows
0,IA,348088
1,[MISSING],71909
2,IN,57939
3,0,240


... (5 outputs left out)

,annotations_per_report
count,58389.000000
mean,8.189488
std,5.914924
min,1.000000
25%,4.000000
50%,7.000000
75%,11.000000
max,92.000000


Reports with one annotation: 2672
Reports with more than 10 annotations: 15440

8. REPORT ID CONSISTENCY
IDs with multiple original texts: 0
IDs with multiple translated texts: 2
IDs with multiple years: 0

9. DUPLICATE ANNOTATIONS
Exact duplicate annotation rows: 6

10. OFFSET VALIDATION — ORIGINAL TEXT
Exact span matches: 0
Total rows: 478176
Exact match percentage: 0.0 %
Mismatches: 478176

First 10 mismatches:


,id,report,namedentity,startidx,endidx,entitytype
0,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ノルアドレナリン,1,9,Drug
1,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,0.1,12,15,Strength_amount
2,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,3ml/h,18,23,Strength_rate
3,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,13:55,46,51,Timing
4,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,シ,52,53,Route
5,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ノルアドレナリン,76,84,Drug
6,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,67/37,142,147,Date
7,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,14,148,150,Timing
8,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,14:02,182,187,Timing
9,A001502FDA86DD17F,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,14:20,213,218,Timing



11. OFFSET VALIDATION — TRANSLATED TEXT
Exact translated span matches: 0
Translated match percentage: 0.0 %

AUDIT CELL COMPLETED


In [8]:

# CELL 6 — Robust offset convention audit

import pandas as pd
import unicodedata

df = raw.copy()

def get_span(text, start, end, convention):
    text = str(text)
    start = int(start)
    end = int(end)

    if convention == "zero_based_end_exclusive":
        return text[start:end]

    elif convention == "one_based_end_exclusive":
        return text[start-1:end-1]

    elif convention == "one_based_end_inclusive":
        return text[start-1:end]

    elif convention == "zero_based_end_inclusive":
        return text[start:end+1]

    raise ValueError("Unknown convention")

conventions = [
    "zero_based_end_exclusive",
    "one_based_end_exclusive",
    "one_based_end_inclusive",
    "zero_based_end_inclusive"
]

entities = df["namedentity"].fillna("").astype(str).tolist()
texts = df["report"].fillna("").astype(str).tolist()
starts = df["startidx"].tolist()
ends = df["endidx"].tolist()

results = []

for convention in conventions:
    matches = []

    for text, entity, start, end in zip(
        texts, entities, starts, ends
    ):
        extracted = get_span(text, start, end, convention)
        matches.append(extracted == entity)

    results.append({
        "convention": convention,
        "exact_matches": sum(matches),
        "total": len(matches),
        "match_percentage": round(
            sum(matches) / len(matches) * 100, 4
        )
    })

offset_results = pd.DataFrame(results)
print("OFFSET CONVENTION COMPARISON")
display(offset_results)

# Inspect the first 15 annotations using all conventions
print("\nFIRST 15 ROWS — MANUAL INSPECTION")

inspection = []

for i in range(min(15, len(df))):
    row = df.iloc[i]

    item = {
        "row": i,
        "id": row["id"],
        "entity": row["namedentity"],
        "start": row["startidx"],
        "end": row["endidx"],
        "original_report_start": row["report"][:100]
    }

    for convention in conventions:
        item[convention] = get_span(
            row["report"],
            row["startidx"],
            row["endidx"],
            convention
        )

    inspection.append(item)

display(pd.DataFrame(inspection))

# Check exact duplicate report texts across different IDs
print("\nDUPLICATE REPORT TEXT AUDIT")

text_id_counts = df.groupby("report")["id"].nunique()

print("Texts associated with multiple IDs:",
      int((text_id_counts > 1).sum()))

print("Maximum IDs associated with one text:",
      int(text_id_counts.max()))

# Check report IDs with multiple translated texts
print("\nREPORTS WITH MULTIPLE TRANSLATIONS")

translation_counts = df.groupby("id")["translated_report"].nunique()

multi_translation_ids = translation_counts[
    translation_counts > 1
]

print("Number of affected IDs:", len(multi_translation_ids))

if len(multi_translation_ids) > 0:
    display(
        df[df["id"].isin(multi_translation_ids.index)][
            ["id", "report", "translated_report"]
        ].drop_duplicates().head(10)
    )

# Inspect the unusual error label value 0
print("\nROWS WITH ERROR LABEL 0")

zero_label = df[
    df["errorlabel"].astype(str).str.strip() == "0"
]

print("Rows:", len(zero_label))
display(
    zero_label[
        ["id", "namedentity", "entitytype",
         "errorlabel", "incidenttypes", "relation"]
    ].head(20)
)

OFFSET CONVENTION COMPARISON


,convention,exact_matches,total,match_percentage
0,zero_based_end_exclusive,49,478176,0.0102
1,one_based_end_exclusive,456751,478176,95.5194
2,one_based_end_inclusive,0,478176,0.0000
3,zero_based_end_inclusive,0,478176,0.0000



FIRST 15 ROWS — MANUAL INSPECTION


,row,id,entity,start,end,original_report_start,zero_based_end_exclusive,one_based_end_exclusive,one_based_end_inclusive,zero_based_end_inclusive
0,0,A001502FDA86DD17F,ノルアドレナリン,1,9,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ルアドレナリンを,ノルアドレナリン,ノルアドレナリンを,ルアドレナリンを時
1,1,A001502FDA86DD17F,0.1,12,15,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,.15,0.1,0.15,.15γ
2,2,A001502FDA86DD17F,3ml/h,18,23,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ml/h),3ml/h,3ml/h),ml/h)で
3,3,A001502FDA86DD17F,13:55,46,51,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,3:55、,13:55,13:55、,3:55、シ
4,4,A001502FDA86DD17F,シ,52,53,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,リ,シ,シリ,リン
5,5,A001502FDA86DD17F,ノルアドレナリン,76,84,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ルアドレナリンが,ノルアドレナリン,ノルアドレナリンが,ルアドレナリンが残
6,6,A001502FDA86DD17F,67/37,142,147,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,7/37。,67/37,67/37。,7/37。1
7,7,A001502FDA86DD17F,14,148,150,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,4:,14,14:,4:0
8,8,A001502FDA86DD17F,14:02,182,187,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,4:02、,14:02,14:02、,4:02、一
9,9,A001502FDA86DD17F,14:20,213,218,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,4:20、,14:20,14:20、,4:20、再



DUPLICATE REPORT TEXT AUDIT
Texts associated with multiple IDs: 26
Maximum IDs associated with one text: 9

REPORTS WITH MULTIPLE TRANSLATIONS
Number of affected IDs: 2


,id,report,translated_report
159996,H48718EB22B9EE128,胸腔鏡下左肺上大区切除術後2日目の患者。朝食後より術後中止していたメトグルコ錠が再開予定であ...,A patient 2 days after thoracoscopic left uppe...
159997,H48718EB22B9EE128,胸腔鏡下左肺上大区切除術後2日目の患者。朝食後より術後中止していたメトグルコ錠が再開予定であ...,A patient 3 days after thoracoscopic left uppe...
159998,H48718EB22B9EE128,胸腔鏡下左肺上大区切除術後2日目の患者。朝食後より術後中止していたメトグルコ錠が再開予定であ...,A patient 4 days after thoracoscopic left uppe...
239994,H7626A967A64E04AC,眠剤にフルニトラゼパムを2錠内服していた患者に、薬剤を1錠しか準備しておらず、1錠しか内服し...,A patient who was taking 2 tablets of flunitra...
239995,H7626A967A64E04AC,眠剤にフルニトラゼパムを2錠内服していた患者に、薬剤を1錠しか準備しておらず、1錠しか内服し...,A patient who was taking 2 tablets of flunitra...
239996,H7626A967A64E04AC,眠剤にフルニトラゼパムを2錠内服していた患者に、薬剤を1錠しか準備しておらず、1錠しか内服し...,A patient who was taking 2 tablets of flunitra...
239997,H7626A967A64E04AC,眠剤にフルニトラゼパムを2錠内服していた患者に、薬剤を1錠しか準備しておらず、1錠しか内服し...,A patient who was taking 2 tablets of flunitra...
239998,H7626A967A64E04AC,眠剤にフルニトラゼパムを2錠内服していた患者に、薬剤を1錠しか準備しておらず、1錠しか内服し...,A patient who was taking 2 tablets of flunitra...



ROWS WITH ERROR LABEL 0
Rows: 240


,id,namedentity,entitytype,errorlabel,incidenttypes,relation
1077,A0758DCA95E871D49,徐々に,Route,0,"Wrong Strength_amount,Wrong Route,Wrong Date,W...",2.0
2151,A0F6CE18A5D1DDE5F,div,Date,0,"Wrong Strength_amount,Wrong Dosage,Wrong Date,...",NaN
2320,A107305DDDBEF780A,発注,Route,0,"Wrong Strength_amount,Wrong Dosage,Wrong Route",NaN
2331,A107305DDDBEF780A,発注,Route,0,"Wrong Strength_amount,Wrong Dosage,Wrong Route",NaN
2414,A1172BA43266C7779,er,Form_mode,0,"Wrong Strength_amount,Wrong Form_mode",NaN
2806,A14D1AB26B652DE7C,1,Date,0,"Wrong Strength_rate,Wrong Dosage,Wrong Duratio...",NaN
2808,A14D1AB26B652DE7C,0.5,Date,0,"Wrong Strength_rate,Wrong Dosage,Wrong Duratio...",NaN
3572,A1C2B26FAD017656B,同,Date,0,Wrong Strength_concentration,1.0
3644,A1C875C023A7C654B,発注,Route,0,Others,NaN
4212,A213093A0C052068F,d,Form_mode,0,"Wrong Form_form,Wrong Date,Wrong Timing,Wrong ...",NaN


In [9]:
import pandas as pd
import numpy as np
import unicodedata
from collections import Counter

# --------------------------------------------------
# 1. Apply the best-supported offset convention
# --------------------------------------------------
d = df.copy()

def safe_extract(row):
    text = row["report"]
    start = row["startidx"]
    end = row["endidx"]

    if pd.isna(text) or pd.isna(start) or pd.isna(end):
        return None

    try:
        return str(text)[int(start)-1:int(end)-1]
    except Exception:
        return None

d["extracted"] = d.apply(safe_extract, axis=1)
d["exact_match"] = (
    d["extracted"].fillna("") == d["namedentity"].fillna("").astype(str)
)

print("EXACT MATCH:", int(d["exact_match"].sum()), "/", len(d))
print("MISMATCH:", int((~d["exact_match"]).sum()))

# --------------------------------------------------
# 2. Check whether mismatched entities occur elsewhere
#    in the same original report
# --------------------------------------------------
def occurrence_count(row):
    text = str(row["report"])
    entity = str(row["namedentity"])

    if not entity or entity == "nan":
        return 0

    return text.count(entity)

d["entity_occurrences_in_report"] = d.apply(occurrence_count, axis=1)

m = d[~d["exact_match"]].copy()

print("\nMISMATCH OCCURRENCE ANALYSIS")
print(m["entity_occurrences_in_report"].value_counts().sort_index())

# --------------------------------------------------
# 3. Check Unicode normalization
# --------------------------------------------------
def normalize_unicode(x, form="NFC"):
    if pd.isna(x):
        return ""
    return unicodedata.normalize(form, str(x))

for form in ["NFC", "NFKC"]:
    extracted_norm = d["extracted"].apply(
        lambda x: normalize_unicode(x, form)
    )
    entity_norm = d["namedentity"].apply(
        lambda x: normalize_unicode(x, form)
    )
    matches = extracted_norm == entity_norm

    print(
        f"{form} normalized matches:",
        int(matches.sum()),
        f"({matches.mean()*100:.3f}%)"
    )

# --------------------------------------------------
# 4. Match rate by entity type
# --------------------------------------------------
type_audit = (
    d.groupby("entitytype", dropna=False)
     .agg(
         annotations=("exact_match", "size"),
         exact_matches=("exact_match", "sum")
     )
)

type_audit["match_percentage"] = (
    type_audit["exact_matches"] /
    type_audit["annotations"] * 100
).round(2)

print("\nMATCH RATE BY ENTITY TYPE")
display(type_audit.sort_values("match_percentage"))

# --------------------------------------------------
# 5. Inspect 30 mismatches
# --------------------------------------------------
cols = [
    "id", "entitytype", "namedentity",
    "startidx", "endidx", "extracted",
    "entity_occurrences_in_report", "report"
]

print("\nFIRST 30 MISMATCHES")
display(m[cols].head(30))

# --------------------------------------------------
# 6. Inspect duplicate original reports across IDs
# --------------------------------------------------
report_id_counts = (
    d[["id", "report"]]
    .drop_duplicates()
    .groupby("report")["id"]
    .nunique()
)

duplicate_texts = report_id_counts[report_id_counts > 1]

print("\nDUPLICATE REPORT TEXTS")
print("Number of texts linked to multiple IDs:", len(duplicate_texts))
print("Maximum IDs for one identical text:",
      int(duplicate_texts.max()) if len(duplicate_texts) else 0)

# Show a few duplicate groups
if len(duplicate_texts):
    duplicate_examples = (
        d[d["report"].isin(duplicate_texts.index)]
        [["id", "report"]]
        .drop_duplicates()
        .groupby("report")
        .agg(ids=("id", lambda x: list(x)))
        .head(10)
        .reset_index()
    )
    display(duplicate_examples)

# --------------------------------------------------
# 7. Check IDs with multiple translated versions
# --------------------------------------------------
translation_audit = (
    d.groupby("id")
     .agg(
         original_versions=("report", "nunique"),
         translation_versions=("translated_report", "nunique"),
         annotation_rows=("id", "size")
     )
)

print("\nIDs WITH MULTIPLE TRANSLATIONS")
display(
    translation_audit[
        translation_audit["translation_versions"] > 1
    ].head(20)
)

# --------------------------------------------------
# 8. Inspect unusual errorlabel values
# --------------------------------------------------
print("\nERROR LABEL VALUES")
print(d["errorlabel"].value_counts(dropna=False).head(20))

print("\nROWS WHERE errorlabel IS ZERO")
display(
    d[d["errorlabel"].astype(str).str.strip().isin(["0", "0.0"])]
    [["id", "namedentity", "entitytype", "errorlabel",
      "incidenttypes", "relation"]]
    .head(30)
)

EXACT MATCH: 456751 / 478176
MISMATCH: 21425

MISMATCH OCCURRENCE ANALYSIS
entity_occurrences_in_report
0     10278
1      5520
2      2435
3      1348
4       761
5       430
6       291
7       141
8       107
9        21
10       15
11       15
12        5
13       17
14       13
15        8
16        6
17        1
18        1
21        9
22        2
25        1
Name: count, dtype: int64
NFC normalized matches: 456751 (95.519%)
NFKC normalized matches: 456751 (95.519%)

MATCH RATE BY ENTITY TYPE


,annotations,exact_matches,match_percentage
entitytype,,,
Date,40617,35619,87.69
Timing,90986,84510,92.88
Duration,15127,14487,95.77
Strength_concentration,4041,3885,96.14
Frequency,7289,7075,97.06
Route,36496,35446,97.12
Strength_amount,59493,57786,97.13
Drug,168633,163899,97.19
Form_form,22859,22219,97.20



FIRST 30 MISMATCHES


,id,entitytype,namedentity,startidx,endidx,extracted,entity_occurrences_in_report,report
21,A0015DA1427930D33,Route,坐薬,163,165,視で,6,腰痛に対しボルタレン坐薬使用、鎮痛剤点滴、mri撮影の指示あり、外来処置室ベッド上左側臥位に...
22,A0015DA1427930D33,Route,坐薬,187,189,大き,6,腰痛に対しボルタレン坐薬使用、鎮痛剤点滴、mri撮影の指示あり、外来処置室ベッド上左側臥位に...
143,A0111289E37E07E2C,Drug,▁オピ,2,5,ピスタ,0,オピスタンを静脈注射するところをサイレースを注射した。看護師がサイレースを台に準備しており、...
250,A01DE352B2048F6AB,Drug,▁高カロリー輸液,2,10,カロリー輸液によ,0,高カロリー輸液による高血糖予防のため、ヒューマリンr4単位をエルネオパ2号1500mlに混注...
471,A038D4E4CAFF5930E,Timing,▁16時に,3,8,時に内服す,0,16時に内服する指示のリタリン10mg1錠を内服させ忘れた。
475,A038D823C48FFF4D3,Timing,朝の,49,51,ェノ,1,患者は、症候性てんかんがあり、側彎症術後で経口摂取が困難となり胃瘻増設していた。朝の内服指示...
476,A038D823C48FFF4D3,Drug,フェノバールエリキシル,56,67,キシル4ml、デパケン,2,患者は、症候性てんかんがあり、側彎症術後で経口摂取が困難となり胃瘻増設していた。朝の内服指示...
477,A038D823C48FFF4D3,Strength_amount,4ml,67,70,シロッ,1,患者は、症候性てんかんがあり、側彎症術後で経口摂取が困難となり胃瘻増設していた。朝の内服指示...
478,A038D823C48FFF4D3,Drug,デパケンシロップ,71,79,7mlがあった。,2,患者は、症候性てんかんがあり、側彎症術後で経口摂取が困難となり胃瘻増設していた。朝の内服指示...
479,A038D823C48FFF4D3,Strength_amount,7ml,79,82,注入用,1,患者は、症候性てんかんがあり、側彎症術後で経口摂取が困難となり胃瘻増設していた。朝の内服指示...



DUPLICATE REPORT TEXTS
Number of texts linked to multiple IDs: 26
Maximum IDs for one identical text: 9


,report,ids
0,オメプラール注用20mg「np」+生食100mlを調剤すべきところ、タケプロン注30mg+生...,"[H53E62C4B6FC394BD, HF64A5EDBA2675FF9]"
1,・ジェニナック1回2錠内服のところ、1錠しか投与しなかった。,"[H120A5E3FC1B8FF17, H94C66D516E70AA19]"
2,・昼食後薬の投薬を忘れた。,"[H70A7B7D2B920659E, H9AD15DA28ECA6A64]"
3,処方単位間違い,"[H59019892ED862C3F, H964992DAF471C0E8, HB31D91..."
4,処方薬剤間違い,"[H259052FF8625863B, H39D615DC813E1647, H882F63..."
5,処方薬間違い,"[H07FCCEC2A1DF734F, H4BE6BF5064A0D90C]"
6,単位間違い,"[H03452DF8F75A50F1, H40CD7645B78418D7, H9B0741..."
7,夕食後薬を配薬し忘れた。,"[HE5A8322E801825FF, HFAED519B16C0D0EB]"
8,投与時間・日付間違い,"[H0284F79B10B32622, H07965A65D9F82CEB, H10937D..."
9,投薬時間・日付間違い,"[H7329049EF9709F34, H9BC216FC9A808ADE]"



IDs WITH MULTIPLE TRANSLATIONS


,original_versions,translation_versions,annotation_rows
id,,,
H48718EB22B9EE128,1,3,8
H7626A967A64E04AC,1,5,7



ERROR LABEL VALUES
errorlabel
IA     348088
NaN     71909
IN      57939
0         240
Name: count, dtype: int64

ROWS WHERE errorlabel IS ZERO


,id,namedentity,entitytype,errorlabel,incidenttypes,relation
1077,A0758DCA95E871D49,徐々に,Route,0,"Wrong Strength_amount,Wrong Route,Wrong Date,W...",2.0
2151,A0F6CE18A5D1DDE5F,div,Date,0,"Wrong Strength_amount,Wrong Dosage,Wrong Date,...",NaN
2320,A107305DDDBEF780A,発注,Route,0,"Wrong Strength_amount,Wrong Dosage,Wrong Route",NaN
2331,A107305DDDBEF780A,発注,Route,0,"Wrong Strength_amount,Wrong Dosage,Wrong Route",NaN
2414,A1172BA43266C7779,er,Form_mode,0,"Wrong Strength_amount,Wrong Form_mode",NaN
2806,A14D1AB26B652DE7C,1,Date,0,"Wrong Strength_rate,Wrong Dosage,Wrong Duratio...",NaN
2808,A14D1AB26B652DE7C,0.5,Date,0,"Wrong Strength_rate,Wrong Dosage,Wrong Duratio...",NaN
3572,A1C2B26FAD017656B,同,Date,0,Wrong Strength_concentration,1.0
3644,A1C875C023A7C654B,発注,Route,0,Others,NaN
4212,A213093A0C052068F,d,Form_mode,0,"Wrong Form_form,Wrong Date,Wrong Timing,Wrong ...",NaN


In [10]:
import pandas as pd
import numpy as np

# Use the existing audit dataframe: d
m = d[~d["exact_match"]].copy()

# Find all occurrences of an entity in a report
def find_positions(text, entity):
    text = str(text)
    entity = str(entity)

    if not entity or entity == "nan":
        return []

    positions = []
    start = 0

    while True:
        pos = text.find(entity, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1

    return positions

# Analyze each mismatch
def diagnose(row):
    text = str(row["report"])
    entity = str(row["namedentity"])
    start = int(row["startidx"]) - 1
    end = int(row["endidx"]) - 1
    extracted = text[start:end]

    # Test whether removing SentencePiece-like marker helps
    cleaned_entity = entity.replace("▁", "").strip()
    cleaned_positions = find_positions(text, cleaned_entity)

    exact_positions = find_positions(text, entity)

    # Check if the annotated span is close to an actual occurrence
    nearest_distance = (
        min(abs(p - start) for p in cleaned_positions)
        if cleaned_positions else None
    )

    return pd.Series({
        "entity": entity,
        "cleaned_entity": cleaned_entity,
        "extracted": extracted,
        "entity_length": len(entity),
        "extracted_length": len(extracted),
        "exact_occurrences": len(exact_positions),
        "cleaned_occurrences": len(cleaned_positions),
        "occurrence_positions": cleaned_positions[:10],
        "nearest_start_distance": nearest_distance,
        "start": start,
        "end": end,
        "report_length": len(text),
        "context_at_offset": text[max(0,start-15):min(len(text),end+15)]
    })

diagnostics = m.apply(diagnose, axis=1)
m_diag = pd.concat(
    [m[["id", "entitytype", "errorlabel", "relation"]].reset_index(drop=True),
     diagnostics.reset_index(drop=True)],
    axis=1
)

# --------------------------------------------------
# A. Overall diagnostic categories
# --------------------------------------------------
m_diag["diagnostic_category"] = np.select(
    [
        m_diag["cleaned_occurrences"] > 0,
        m_diag["exact_occurrences"] > 0,
        m_diag["cleaned_occurrences"] == 0
    ],
    [
        "Found after removing ▁ / trimming",
        "Exact entity found elsewhere",
        "Entity not found in original report"
    ],
    default="Other"
)

print("DIAGNOSTIC CATEGORY COUNTS")
display(m_diag["diagnostic_category"].value_counts())

# --------------------------------------------------
# B. Inspect mismatches where marker removal helps
# --------------------------------------------------
print("\nEXAMPLES: ENTITY FOUND AFTER CLEANING")
display(
    m_diag[
        m_diag["cleaned_occurrences"] > 0
    ][[
        "id", "entitytype", "entity", "cleaned_entity",
        "extracted", "occurrence_positions",
        "start", "nearest_start_distance"
    ]].head(25)
)

# --------------------------------------------------
# C. Inspect entities found elsewhere in the report
# --------------------------------------------------
print("\nEXAMPLES: ENTITY FOUND ELSEWHERE")
display(
    m_diag[
        (m_diag["cleaned_occurrences"] > 0) &
        (m_diag["nearest_start_distance"] > 0)
    ][[
        "id", "entitytype", "entity", "extracted",
        "occurrence_positions", "start",
        "nearest_start_distance"
    ]].head(25)
)

# --------------------------------------------------
# D. Inspect entities not found in the original report
# --------------------------------------------------
print("\nEXAMPLES: ENTITY NOT FOUND")
display(
    m_diag[
        m_diag["cleaned_occurrences"] == 0
    ][[
        "id", "entitytype", "entity", "extracted",
        "start", "end", "context_at_offset",
        "report_length"
    ]].head(25)
)

# --------------------------------------------------
# E. Check whether the entity matches translated report
# --------------------------------------------------
def translated_match(row):
    translated = str(row["translated_report"])
    entity = str(row["translated_namedentity"])

    if not entity or entity == "nan":
        return False

    return entity in translated

m["translated_entity_found"] = m.apply(translated_match, axis=1)

print("\nTRANSLATED ENTITY PRESENCE AMONG MISMATCHES")
print(m["translated_entity_found"].value_counts(dropna=False))

# --------------------------------------------------
# F. Summary by entity type
# --------------------------------------------------
print("\nDIAGNOSTIC CATEGORIES BY ENTITY TYPE")
display(
    pd.crosstab(
        m_diag["entitytype"],
        m_diag["diagnostic_category"],
        margins=True
    )
)

DIAGNOSTIC CATEGORY COUNTS


,count
diagnostic_category,
Found after removing ▁ / trimming,20582
Entity not found in original report,843



EXAMPLES: ENTITY FOUND AFTER CLEANING


,id,entitytype,entity,cleaned_entity,extracted,occurrence_positions,start,nearest_start_distance
0,A0015DA1427930D33,Route,坐薬,坐薬,視で,"[10, 47, 89, 106, 158, 178]",162,4.0
1,A0015DA1427930D33,Route,坐薬,坐薬,大き,"[10, 47, 89, 106, 158, 178]",186,8.0
2,A0111289E37E07E2C,Drug,▁オピ,オピ,ピスタ,[0],1,1.0
3,A01DE352B2048F6AB,Drug,▁高カロリー輸液,高カロリー輸液,カロリー輸液によ,[0],1,1.0
4,A038D4E4CAFF5930E,Timing,▁16時に,16時に,時に内服す,[0],2,2.0
5,A038D823C48FFF4D3,Timing,朝の,朝の,ェノ,[40],48,8.0
6,A038D823C48FFF4D3,Drug,フェノバールエリキシル,フェノバールエリキシル,キシル4ml、デパケン,"[47, 244]",55,8.0
7,A038D823C48FFF4D3,Strength_amount,4ml,4ml,シロッ,[58],66,8.0
8,A038D823C48FFF4D3,Drug,デパケンシロップ,デパケンシロップ,7mlがあった。,"[62, 181]",70,8.0
9,A038D823C48FFF4D3,Strength_amount,7ml,7ml,注入用,[70],78,8.0



EXAMPLES: ENTITY FOUND ELSEWHERE


,id,entitytype,entity,extracted,occurrence_positions,start,nearest_start_distance
0,A0015DA1427930D33,Route,坐薬,視で,"[10, 47, 89, 106, 158, 178]",162,4.0
1,A0015DA1427930D33,Route,坐薬,大き,"[10, 47, 89, 106, 158, 178]",186,8.0
2,A0111289E37E07E2C,Drug,▁オピ,ピスタ,[0],1,1.0
3,A01DE352B2048F6AB,Drug,▁高カロリー輸液,カロリー輸液によ,[0],1,1.0
4,A038D4E4CAFF5930E,Timing,▁16時に,時に内服す,[0],2,2.0
5,A038D823C48FFF4D3,Timing,朝の,ェノ,[40],48,8.0
6,A038D823C48FFF4D3,Drug,フェノバールエリキシル,キシル4ml、デパケン,"[47, 244]",55,8.0
7,A038D823C48FFF4D3,Strength_amount,4ml,シロッ,[58],66,8.0
8,A038D823C48FFF4D3,Drug,デパケンシロップ,7mlがあった。,"[62, 181]",70,8.0
9,A038D823C48FFF4D3,Strength_amount,7ml,注入用,[70],78,8.0



EXAMPLES: ENTITY NOT FOUND


,id,entitytype,entity,extracted,start,end,context_at_offset,report_length
52,A0A5B2C4622CE17AF,Drug,リン1mg▁2錠調,リン1mg 2錠調,179,188,参してもらったところ、ワーファリン1mg 2錠調剤されており、トータル2.75,243
53,A0AAAD1EC192566A3,Form_form,48ml▁2,48ml 2,118,124,キナーゼ24万単位+生理食塩液48ml 2ml/h」を左右のラインから投,213
188,A12B1ADC88C9C1AC4,Drug,グリセリン<unk>腸,ン浣腸150ml実施特,113,124,0処置室にて左側臥位でグリセリン浣腸150ml実施特に挿入時抵抗なし。ゼリー塗布時,509
201,A13AF77C41F559A86,Timing,16:30▁b,16:30 b,92,99,o2 82%低下酸素5l開始。16:30 bp80/70mmhgと低下。ソ,293
334,A212C89EB3E7C741C,Strength_amount,250mg▁3錠、ピラ,250mg 3錠、ピラ,287,298,、リファジン3cp、エブトール250mg 3錠、ピラマイド1.2g、ピドキサール1,395
335,A212C89EB3E7C741C,Strength_amount,mg▁2錠、アムロ,mg 2錠、アムロ,314,323,イド1.2g、ピドキサール10mg 2錠、アムロジン2.5mg 2錠、アジルバ,395
336,A212C89EB3E7C741C,Drug,ジン2.5mg▁2錠、アジルバ錠20mg▁,ジン2.5mg 2錠、アジルバ錠20mg,323,344,キサール10mg 2錠、アムロジン2.5mg 2錠、アジルバ錠20mg 0.5錠、ムコスタ錠...,395
338,A214B08AE9F974C56,Timing,分2▁8,分2 8,416,420,ーラル内用液10% 0.8ml分2 8:00/20:00 7日分」と,791
339,A214B08AE9F974C56,Timing,00/20:00▁7日,00/20:00 7日,421,432,液10% 0.8ml分2 8:00/20:00 7日分」と手書き処方せんで処方し、,791
373,A229A57FF007876C6,Strength_concentration,40meq▁/20ml)生理,40meq /20ml)生理,50,64,頸側管1 kcl点滴液15%(40meq /20ml)生理食塩液(20ml) 1日3回指,991



TRANSLATED ENTITY PRESENCE AMONG MISMATCHES
translated_entity_found
False    11926
True      9499
Name: count, dtype: int64

DIAGNOSTIC CATEGORIES BY ENTITY TYPE


diagnostic_category,Entity not found in original report,Found after removing ▁ / trimming,All
entitytype,,,
Date,133,4865,4998
Dosage,44,536,580
Drug,265,4469,4734
Duration,0,640,640
Form_form,62,578,640
Form_mode,0,44,44
Frequency,20,194,214
Route,41,1009,1050
Strength_amount,211,1496,1707


In [11]:
import pandas as pd
import numpy as np
from collections import Counter

# Use the existing dataframe if already loaded.
# Otherwise, load the workbook:
if 'df' not in globals():
    df = pd.read_excel('/content/AnnotatedDataset.xlsx')

required = ['id', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype']
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

def clean_entity(x):
    if pd.isna(x):
        return ''
    return str(x).replace('▁', '').strip()

def find_occurrences(text, entity):
    positions = []
    if not entity:
        return positions
    start = 0
    while True:
        pos = text.find(entity, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1
    return positions

records = []

for idx, row in df.iterrows():
    text = str(row['report'])
    entity_raw = str(row['namedentity'])
    entity = clean_entity(entity_raw)

    if not entity or entity_raw == 'nan':
        continue

    positions = find_occurrences(text, entity)

    if not positions:
        continue

    supplied_start = int(row['startidx'])

    nearest = min(positions, key=lambda p: abs(p - supplied_start))

    records.append({
        'row_index': idx,
        'id': row['id'],
        'entitytype': row['entitytype'],
        'entity': entity,
        'supplied_start': supplied_start,
        'nearest_occurrence': nearest,
        'difference_zero_based': nearest - supplied_start,
        'difference_one_based': nearest - (supplied_start - 1),
        'number_of_occurrences': len(positions),
        'all_occurrences': positions
    })

offset_audit = pd.DataFrame(records)

print("ENTITIES LOCATED IN ORIGINAL REPORT:", len(offset_audit))
print("")

for col in ['difference_zero_based', 'difference_one_based']:
    print(f"\n{col}")
    print("Exact:", (offset_audit[col] == 0).sum())
    print("Within 1 character:", (offset_audit[col].abs() <= 1).sum())
    print("Within 3 characters:", (offset_audit[col].abs() <= 3).sum())
    print("Within 5 characters:", (offset_audit[col].abs() <= 5).sum())
    print("Median absolute difference:", offset_audit[col].abs().median())

print("\nMOST COMMON OFFSET DIFFERENCES:")
print(offset_audit['difference_zero_based'].value_counts().head(20))

print("\nOFFSET DIFFERENCES BY ENTITY TYPE:")
display(
    offset_audit.groupby('entitytype')['difference_zero_based']
    .agg(
        n='count',
        exact=lambda x: (x == 0).sum(),
        median_abs_difference=lambda x: x.abs().median(),
        within_3=lambda x: (x.abs() <= 3).sum()
    )
    .sort_values('n', ascending=False)
)

print("\nEXAMPLES WITH LARGEST OFFSET DIFFERENCES:")
display(
    offset_audit.assign(
        abs_difference=offset_audit['difference_zero_based'].abs()
    )
    .sort_values('abs_difference', ascending=False)
    .head(30)
)

ENTITIES LOCATED IN ORIGINAL REPORT: 477333


difference_zero_based
Exact: 474
Within 1 character: 457247
Within 3 characters: 466267
Within 5 characters: 475776
Median absolute difference: 1.0

difference_one_based
Exact: 456743
Within 1 character: 462229
Within 3 characters: 466498
Within 5 characters: 475771
Median absolute difference: 0.0

MOST COMMON OFFSET DIFFERENCES:
difference_zero_based
-1     456743
-5       9210
-2       5012
-3       3952
-9       1139
 0        474
-4        260
-13       211
-17        54
 4         34
-8         32
 1         30
 3         29
 2         27
-21        24
-7         21
-12        20
-25        15
 8          7
 5          5
Name: count, dtype: int64

OFFSET DIFFERENCES BY ENTITY TYPE:


,n,exact,median_abs_difference,within_3
entitytype,,,,
Drug,168368,28,1.0,164554
Timing,90950,265,1.0,88973
Strength_amount,59282,84,1.0,57875
Date,40484,11,1.0,39447
Route,36455,9,1.0,35482
Form_form,22797,7,1.0,22271
Dosage,22687,16,1.0,22183
Duration,15127,22,1.0,14791
Strength_rate,8078,27,1.0,7931



EXAMPLES WITH LARGEST OFFSET DIFFERENCES:


,row_index,id,entitytype,entity,supplied_start,nearest_occurrence,difference_zero_based,difference_one_based,number_of_occurrences,all_occurrences,abs_difference
349067,349689,HB5AFD393541F543B,Strength_amount,150mg5,409,258,-151,-150,1,[258],151
75694,75844,H1857FEB525C599CE,Strength_amount,100mg2a,63,130,67,68,1,[130],67
344980,345594,HB39B1F10CBE92DEB,Strength_amount,g2,80,15,-65,-64,1,[15],65
299244,299781,H9941FBB48ACD3B19,Drug,ヒューマリンr3,39,91,52,53,2,"[91, 138]",52
317235,317802,HA36DBBF37B8FBC06,Dosage,1,249,216,-33,-32,3,"[60, 62, 216]",33
256952,257422,H80AED028DE809AA3,Strength_amount,30ml,140,107,-33,-32,1,[107],33
256953,257423,H80AED028DE809AA3,Strength_amount,20ml,159,126,-33,-32,1,[126],33
253943,254407,H7F085B9A74E46974,Strength_amount,v+ソル,139,108,-31,-30,1,[108],31
276312,276813,H8BD0ADDDF014808E,Form_form,200mg2錠,29,59,30,31,1,[59],30
317234,317800,HA36DBBF37B8FBC06,Date,3日目,226,197,-29,-28,1,[197],29


In [12]:
import pandas as pd
import numpy as np

def clean_entity(x):
    if pd.isna(x):
        return ''
    return str(x).replace('▁', '').strip()

def validate_span(row):
    text = str(row['report'])
    entity = clean_entity(row['namedentity'])

    # Candidate convention: one-based start, end-exclusive
    start = int(row['startidx']) - 1
    end = int(row['endidx']) - 1

    extracted = text[start:end]

    return pd.Series({
        'cleaned_entity': entity,
        'extracted_span': extracted,
        'exact_match': extracted == entity,
        'span_length': len(entity),
        'extracted_length': len(extracted),
        'length_match': len(entity) == len(extracted)
    })

span_results = df.apply(validate_span, axis=1)
audit = pd.concat([df[['id', 'entitytype', 'startidx', 'endidx']], span_results], axis=1)

print("TOTAL ANNOTATIONS:", len(audit))
print("\nEXACT SPAN MATCHES:")
print(audit['exact_match'].value_counts())

print("\nEXACT MATCH PERCENTAGE:")
print(round(audit['exact_match'].mean() * 100, 3), "%")

print("\nMATCH RATE BY ENTITY TYPE:")
display(
    audit.groupby('entitytype')['exact_match']
    .agg(total='count', exact_matches='sum', match_rate='mean')
    .assign(match_rate=lambda x: (x['match_rate'] * 100).round(2))
    .sort_values('total', ascending=False)
)

print("\nLENGTH MISMATCHES:")
print((~audit['length_match']).sum())

print("\nEXAMPLES OF FAILED SPAN VALIDATION:")
display(audit.loc[~audit['exact_match']].head(30))

TOTAL ANNOTATIONS: 478176

EXACT SPAN MATCHES:
exact_match
True     456751
False     21425
Name: count, dtype: int64

EXACT MATCH PERCENTAGE:
95.519 %

MATCH RATE BY ENTITY TYPE:


,total,exact_matches,match_rate
entitytype,,,
Drug,168633,163899,97.19
Timing,90986,84510,92.88
Strength_amount,59493,57786,97.13
Date,40617,35619,87.69
Route,36496,35446,97.12
Form_form,22859,22219,97.20
Dosage,22731,22151,97.45
Duration,15127,14487,95.77
Strength_rate,8084,7898,97.70



LENGTH MISMATCHES:
10091

EXAMPLES OF FAILED SPAN VALIDATION:


,id,entitytype,startidx,endidx,cleaned_entity,extracted_span,exact_match,span_length,extracted_length,length_match
21,A0015DA1427930D33,Route,163,165,坐薬,視で,False,2,2,True
22,A0015DA1427930D33,Route,187,189,坐薬,大き,False,2,2,True
143,A0111289E37E07E2C,Drug,2,5,オピ,ピスタ,False,2,3,False
250,A01DE352B2048F6AB,Drug,2,10,高カロリー輸液,カロリー輸液によ,False,7,8,False
471,A038D4E4CAFF5930E,Timing,3,8,16時に,時に内服す,False,4,5,False
475,A038D823C48FFF4D3,Timing,49,51,朝の,ェノ,False,2,2,True
476,A038D823C48FFF4D3,Drug,56,67,フェノバールエリキシル,キシル4ml、デパケン,False,11,11,True
477,A038D823C48FFF4D3,Strength_amount,67,70,4ml,シロッ,False,3,3,True
478,A038D823C48FFF4D3,Drug,71,79,デパケンシロップ,7mlがあった。,False,8,8,True
479,A038D823C48FFF4D3,Strength_amount,79,82,7ml,注入用,False,3,3,True


In [13]:
import pandas as pd
import numpy as np
from collections import Counter

def clean_entity(x):
    if pd.isna(x):
        return ''
    return str(x).replace('▁', '').strip()

def occurrences(text, entity):
    if not entity:
        return []
    positions = []
    pos = 0
    while True:
        pos = text.find(entity, pos)
        if pos == -1:
            break
        positions.append(pos)
        pos += 1
    return positions

mismatch_rows = []

for i, row in df.iterrows():
    text = str(row['report'])
    raw_entity = str(row['namedentity'])
    entity = clean_entity(raw_entity)

    start = int(row['startidx']) - 1
    end = int(row['endidx']) - 1
    extracted = text[start:end]

    if extracted == entity:
        continue

    positions = occurrences(text, entity)

    nearest_position = (
        min(positions, key=lambda p: abs(p - start))
        if positions else None
    )

    mismatch_rows.append({
        'row_index': i,
        'id': row['id'],
        'entitytype': row['entitytype'],
        'raw_entity': raw_entity,
        'entity': entity,
        'start': start,
        'end': end,
        'extracted': extracted,
        'entity_length': len(entity),
        'extracted_length': len(extracted),
        'length_difference': len(extracted) - len(entity),
        'occurrence_count': len(positions),
        'occurrence_positions': positions,
        'nearest_position': nearest_position,
        'nearest_distance': (
            abs(nearest_position - start)
            if nearest_position is not None else None
        ),
        'has_unk': '<unk>' in raw_entity,
        'has_boundary_marker': '▁' in raw_entity
    })

mismatches = pd.DataFrame(mismatch_rows)

print("TOTAL MISMATCHES:", len(mismatches))

print("\nOCCURRENCE CLASSIFICATION:")
print(
    mismatches['occurrence_count']
    .map(lambda x: 'Not found' if x == 0
         else 'Unique occurrence' if x == 1
         else 'Multiple occurrences')
    .value_counts()
)

print("\nLENGTH DIFFERENCE DISTRIBUTION:")
print(mismatches['length_difference'].value_counts().head(20))

print("\nMARKER COUNTS:")
print("Contains <unk>:", mismatches['has_unk'].sum())
print("Contains ▁:", mismatches['has_boundary_marker'].sum())

print("\nNEAREST OCCURRENCE DISTANCE:")
display(
    mismatches['nearest_distance']
    .describe()
    .to_frame('distance')
)

print("\nMISMATCHES BY ENTITY TYPE:")
display(
    mismatches.groupby('entitytype').agg(
        mismatches=('entity', 'size'),
        not_found=('occurrence_count', lambda x: (x == 0).sum()),
        unique_occurrence=('occurrence_count', lambda x: (x == 1).sum()),
        multiple_occurrences=('occurrence_count', lambda x: (x > 1).sum()),
        mean_length_difference=('length_difference', 'mean')
    ).sort_values('mismatches', ascending=False)
)

print("\nMISMATCHES WITH UNIQUE ENTITY OCCURRENCE:")
display(
    mismatches[mismatches['occurrence_count'] == 1][
        ['id', 'entitytype', 'entity', 'start', 'end',
         'extracted', 'occurrence_positions', 'nearest_distance']
    ].head(30)
)

print("\nMISMATCHES WITH NO ENTITY OCCURRENCE:")
display(
    mismatches[mismatches['occurrence_count'] == 0][
        ['id', 'entitytype', 'raw_entity', 'entity',
         'extracted', 'start', 'end']
    ].head(30)
)

TOTAL MISMATCHES: 21425

OCCURRENCE CLASSIFICATION:
occurrence_count
Unique occurrence       13648
Multiple occurrences     6934
Not found                 843
Name: count, dtype: int64

LENGTH DIFFERENCE DISTRIBUTION:
length_difference
 0     11334
 1     10010
 2        34
-1        16
-3         9
-2         8
-4         6
-5         2
 3         2
-7         1
 5         1
-11        1
-6         1
Name: count, dtype: int64

MARKER COUNTS:
Contains <unk>: 231
Contains ▁: 10047

NEAREST OCCURRENCE DISTANCE:


,distance
count,20582.000000
mean,3.217423
std,2.652455
min,0.000000
25%,1.000000
50%,4.000000
75%,4.000000
max,150.000000



MISMATCHES BY ENTITY TYPE:


,mismatches,not_found,unique_occurrence,multiple_occurrences,mean_length_difference
entitytype,,,,,
Timing,6476,36,5431,1009,0.692248
Date,4998,133,3851,1014,0.797919
Drug,4734,265,1807,2662,0.141107
Strength_amount,1707,211,934,562,0.178676
Route,1050,41,305,704,0.003810
Duration,640,0,543,97,0.468750
Form_form,640,62,172,406,0.082812
Dosage,580,44,262,274,0.117241
Frequency,214,20,132,62,0.158879



MISMATCHES WITH UNIQUE ENTITY OCCURRENCE:


,id,entitytype,entity,start,end,extracted,occurrence_positions,nearest_distance
2,A0111289E37E07E2C,Drug,オピ,1,4,ピスタ,[0],1.0
3,A01DE352B2048F6AB,Drug,高カロリー輸液,1,9,カロリー輸液によ,[0],1.0
4,A038D4E4CAFF5930E,Timing,16時に,2,7,時に内服す,[0],2.0
5,A038D823C48FFF4D3,Timing,朝の,48,50,ェノ,[40],8.0
7,A038D823C48FFF4D3,Strength_amount,4ml,66,69,シロッ,[58],8.0
9,A038D823C48FFF4D3,Strength_amount,7ml,78,81,注入用,[70],8.0
10,A038D823C48FFF4D3,Strength_amount,20ml,106,110,薬液ボト,[98],8.0
13,A038D823C48FFF4D3,Strength_amount,10ml,160,164,た。9:,[152],8.0
14,A038D823C48FFF4D3,Timing,9:30,170,174,ップシリ,[162],8.0
21,A038D823C48FFF4D3,Drug,蒸留水,300,303,ml程,[288],12.0



MISMATCHES WITH NO ENTITY OCCURRENCE:


,id,entitytype,raw_entity,entity,extracted,start,end
52,A0A5B2C4622CE17AF,Drug,リン1mg▁2錠調,リン1mg2錠調,リン1mg 2錠調,179,188
53,A0AAAD1EC192566A3,Form_form,48ml▁2,48ml2,48ml 2,118,124
188,A12B1ADC88C9C1AC4,Drug,グリセリン<unk>腸,グリセリン<unk>腸,ン浣腸150ml実施特,113,124
201,A13AF77C41F559A86,Timing,16:30▁b,16:30b,16:30 b,92,99
334,A212C89EB3E7C741C,Strength_amount,250mg▁3錠、ピラ,250mg3錠、ピラ,250mg 3錠、ピラ,287,298
335,A212C89EB3E7C741C,Strength_amount,mg▁2錠、アムロ,mg2錠、アムロ,mg 2錠、アムロ,314,323
336,A212C89EB3E7C741C,Drug,ジン2.5mg▁2錠、アジルバ錠20mg▁,ジン2.5mg2錠、アジルバ錠20mg,ジン2.5mg 2錠、アジルバ錠20mg,323,344
338,A214B08AE9F974C56,Timing,分2▁8,分28,分2 8,416,420
339,A214B08AE9F974C56,Timing,00/20:00▁7日,00/20:007日,00/20:00 7日,421,432
373,A229A57FF007876C6,Strength_concentration,40meq▁/20ml)生理,40meq/20ml)生理,40meq /20ml)生理,50,64


In [14]:
import pandas as pd
import re
import numpy as np

# Work on a copy; do not overwrite the original annotations.
q = d.copy()

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def normalize_spaces(x):
    x = safe_str(x).replace("\u3000", " ")
    return re.sub(r"\s+", " ", x).strip()

def normalize_marker_text(x):
    # Treat the tokenizer marker as a possible word/space boundary.
    return normalize_spaces(safe_str(x).replace("▁", " "))

# Reconstruct the supplied span using the candidate convention:
# one-based start, end-exclusive end.
q["original_text"] = q["report"].map(safe_str)
q["raw_entity"] = q["namedentity"].map(safe_str)
q["extracted_span"] = q.apply(
    lambda r: r["original_text"][int(r["startidx"])-1:int(r["endidx"])-1],
    axis=1
)

q["exact_match"] = q["extracted_span"] == q["raw_entity"]

m = q.loc[~q["exact_match"]].copy()

def find_occurrences(text, entity):
    if not entity or "<unk>" in entity:
        return []
    return [x.start() for x in re.finditer(re.escape(entity), text)]

records = []

for _, r in m.iterrows():
    text = r["original_text"]
    entity = r["raw_entity"]
    extracted = r["extracted_span"]

    # Possible interpretation of ▁ as a space.
    marker_entity = entity.replace("▁", " ")

    # Check whether the supplied span matches after whitespace normalization.
    marker_space_match = (
        normalize_spaces(extracted) == normalize_spaces(marker_entity)
    )

    # Check whether the entity can be found elsewhere in the original report.
    occurrences = find_occurrences(text, entity)
    marker_occurrences = (
        find_occurrences(text, marker_entity)
        if marker_entity != entity else []
    )

    # The expected zero-based start position under the candidate convention.
    expected_start = int(r["startidx"]) - 1

    all_positions = sorted(set(occurrences + marker_occurrences))
    nearest_positions = sorted(
        all_positions,
        key=lambda p: abs(p - expected_start)
    )

    if "<unk>" in entity:
        category = "Contains <unk> — unresolved"
    elif marker_space_match and "▁" in entity:
        category = "Likely tokenizer/whitespace artifact"
    elif marker_space_match:
        category = "Whitespace-normalization match"
    elif len(occurrences) == 1:
        category = "Entity occurs once elsewhere — offset review"
    elif len(occurrences) > 1:
        category = "Entity occurs multiple times — ambiguous offset"
    elif len(marker_occurrences) > 0:
        category = "Marker-adjusted occurrence — review"
    else:
        category = "Entity not found — annotation/text review"

    records.append({
        "id": r["id"],
        "year": r["year"] if "year" in r.index else None,
        "entitytype": r["entitytype"],
        "raw_entity": entity,
        "startidx": r["startidx"],
        "endidx": r["endidx"],
        "extracted_span": extracted,
        "category": category,
        "marker_space_match": marker_space_match,
        "exact_occurrences": len(occurrences),
        "marker_occurrences": len(marker_occurrences),
        "nearest_candidate_positions_zero_based": nearest_positions[:5],
        "nearest_distance": (
            abs(nearest_positions[0] - expected_start)
            if nearest_positions else None
        ),
        "report_text": text
    })

audit = pd.DataFrame(records)

print("MISMATCHES:", len(audit))
print("\nCLASSIFICATION:")
display(audit["category"].value_counts().to_frame("count"))

print("\nCLASSIFICATION BY ENTITY TYPE:")
display(pd.crosstab(audit["entitytype"], audit["category"]))

print("\nEXAMPLES OF TOKENIZER / WHITESPACE CASES:")
display(
    audit[audit["category"].str.contains("tokenizer|Whitespace", case=False)]
    [["id", "entitytype", "raw_entity", "extracted_span",
      "startidx", "endidx", "category"]]
    .head(30)
)

print("\nEXAMPLES OF UNRESOLVED CASES:")
display(
    audit[audit["category"].str.contains("unresolved|not found", case=False)]
    [["id", "entitytype", "raw_entity", "extracted_span",
      "startidx", "endidx", "category"]]
    .head(30)
)

# Export the complete mismatch audit for manual review.
audit.to_csv(
    "AnnotatedDataset_mismatch_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: AnnotatedDataset_mismatch_audit.csv")

MISMATCHES: 21425

CLASSIFICATION:


,count
category,
Entity not found — annotation/text review,8990
Entity occurs multiple times — ambiguous offset,5627
Entity occurs once elsewhere — offset review,5520
Likely tokenizer/whitespace artifact,1028
Contains <unk> — unresolved,231
Marker-adjusted occurrence — review,29



CLASSIFICATION BY ENTITY TYPE:


category,Contains <unk> — unresolved,Entity not found — annotation/text review,Entity occurs multiple times — ambiguous offset,Entity occurs once elsewhere — offset review,Likely tokenizer/whitespace artifact,Marker-adjusted occurrence — review
entitytype,,,,,,
Date,0,3850,305,704,137,2
Dosage,2,7,266,245,60,0
Drug,185,611,2450,1381,99,8
Duration,0,281,72,265,21,1
Form_form,1,0,406,171,62,0
Form_mode,0,0,32,12,0,0
Frequency,0,14,61,118,20,1
Route,41,13,700,292,4,0
Strength_amount,0,17,553,855,275,7



EXAMPLES OF TOKENIZER / WHITESPACE CASES:


,id,entitytype,raw_entity,extracted_span,startidx,endidx,category
23,A03AB2BE2AA38997A,Strength_rate,▁20ml/h」,20ml/h」,102,110,Likely tokenizer/whitespace artifact
52,A0A5B2C4622CE17AF,Drug,リン1mg▁2錠調,リン1mg 2錠調,180,189,Likely tokenizer/whitespace artifact
53,A0AAAD1EC192566A3,Form_form,48ml▁2,48ml 2,119,125,Likely tokenizer/whitespace artifact
201,A13AF77C41F559A86,Timing,16:30▁b,16:30 b,93,100,Likely tokenizer/whitespace artifact
246,A19E61D8EB16F5508,Dosage,▁10a+,10a+,60,65,Likely tokenizer/whitespace artifact
292,A1E9F3F9270052317,Strength_amount,▁10単位が入り,10単位が入り,30,38,Likely tokenizer/whitespace artifact
334,A212C89EB3E7C741C,Strength_amount,250mg▁3錠、ピラ,250mg 3錠、ピラ,288,299,Likely tokenizer/whitespace artifact
335,A212C89EB3E7C741C,Strength_amount,mg▁2錠、アムロ,mg 2錠、アムロ,315,324,Likely tokenizer/whitespace artifact
336,A212C89EB3E7C741C,Drug,ジン2.5mg▁2錠、アジルバ錠20mg▁,ジン2.5mg 2錠、アジルバ錠20mg,324,345,Likely tokenizer/whitespace artifact
337,A212C89EB3E7C741C,Drug,0.5錠、ムコスタ錠100mg▁,0.5錠、ムコスタ錠100mg,345,361,Likely tokenizer/whitespace artifact



EXAMPLES OF UNRESOLVED CASES:


,id,entitytype,raw_entity,extracted_span,startidx,endidx,category
2,A0111289E37E07E2C,Drug,▁オピ,ピスタ,2,5,Entity not found — annotation/text review
3,A01DE352B2048F6AB,Drug,▁高カロリー輸液,カロリー輸液によ,2,10,Entity not found — annotation/text review
4,A038D4E4CAFF5930E,Timing,▁16時に,時に内服す,3,8,Entity not found — annotation/text review
24,A04ABFC6AAF9CC49B,Timing,▁18:30,:30、患者,3,9,Entity not found — annotation/text review
25,A04BECA552503940C,Date,▁前日に,日に調製,2,6,Entity not found — annotation/text review
50,A08C7339686D69F61,Timing,▁9時15分頃,"時15分頃,a",2,9,Entity not found — annotation/text review
51,A08E2FCE4F1B815FA,Timing,▁10:00頃,:00頃、メイ,3,10,Entity not found — annotation/text review
54,A0B094A6A66FBBB8D,Timing,▁9時11分,時11分、左,2,8,Entity not found — annotation/text review
115,A0E3D0D582D5C564C,Timing,▁15時,時翌日の,3,7,Entity not found — annotation/text review
116,A0E722390E3214983,Timing,▁4:48,:48に与,2,7,Entity not found — annotation/text review



Saved: AnnotatedDataset_mismatch_audit.csv


In [15]:
import pandas as pd
import numpy as np

# Use the existing audit and original dataframe.
# Do not modify either source dataframe.

marker_cases = audit[
    audit["raw_entity"].str.contains("▁", na=False)
].copy()

print("TOTAL MARKER CASES:", len(marker_cases))

# Examine whether marker-bearing entities begin at the beginning
# of the report or appear elsewhere.
marker_cases["report_starts_with_entity"] = marker_cases.apply(
    lambda r: r["report_text"].startswith(
        r["raw_entity"].replace("▁", "")
    ),
    axis=1
)

print("\nREPORT STARTS WITH CLEANED ENTITY:")
print(marker_cases["report_starts_with_entity"].value_counts())

# Inspect the first 20 marker cases in detail.
display(
    marker_cases[
        [
            "id", "entitytype", "raw_entity",
            "startidx", "endidx", "extracted_span",
            "report_text", "nearest_candidate_positions_zero_based"
        ]
    ].head(40)
)

# Compare marker cases by entity type.
display(
    marker_cases.groupby("entitytype").agg(
        marker_cases=("id", "size"),
        unique_reports=("id", "nunique"),
        mean_startidx=("startidx", "mean"),
        median_startidx=("startidx", "median")
    ).sort_values("marker_cases", ascending=False)
)

TOTAL MARKER CASES: 10047

REPORT STARTS WITH CLEANED ENTITY:
report_starts_with_entity
True     8996
False    1051
Name: count, dtype: int64


,id,entitytype,raw_entity,startidx,endidx,extracted_span,report_text,nearest_candidate_positions_zero_based
2,A0111289E37E07E2C,Drug,▁オピ,2,5,ピスタ,オピスタンを静脈注射するところをサイレースを注射した。看護師がサイレースを台に準備しており、...,[]
3,A01DE352B2048F6AB,Drug,▁高カロリー輸液,2,10,カロリー輸液によ,高カロリー輸液による高血糖予防のため、ヒューマリンr4単位をエルネオパ2号1500mlに混注...,[]
4,A038D4E4CAFF5930E,Timing,▁16時に,3,8,時に内服す,16時に内服する指示のリタリン10mg1錠を内服させ忘れた。,[]
23,A03AB2BE2AA38997A,Strength_rate,▁20ml/h」,102,110,20ml/h」,患者は胸部症状を主訴に循環器内科を受診した。検査の結果、急性冠動脈症候群の疑いで緊急心臓カテ...,[101]
24,A04ABFC6AAF9CC49B,Timing,▁18:30,3,9,:30、患者,18:30、患者aと患者bのいずれかの食事介助をしようと配膳車へ向かう。患者aの食事介助を行...,[]
25,A04BECA552503940C,Date,▁前日に,2,6,日に調製,前日に調製した抗癌剤を投与した。(当日の抗癌剤は薬剤部で調製中であり、病棟には届いていなかった),[]
50,A08C7339686D69F61,Timing,▁9時15分頃,2,9,"時15分頃,a","9時15分頃,a医師より抗生剤を変更しました,お願いします,と注射指示箋を手渡された。指示内...",[]
51,A08E2FCE4F1B815FA,Timing,▁10:00頃,3,10,:00頃、メイ,10:00頃、メインルートからソルデム3a輸液500ml1袋+kcl注10meqキット10m...,[]
52,A0A5B2C4622CE17AF,Drug,リン1mg▁2錠調,180,189,リン1mg 2錠調,慢性心房細動の治療のため抗凝固療法中であった。ワーファリン錠1mg 1錠、ワーファリン錠0....,[179]
53,A0AAAD1EC192566A3,Form_form,48ml▁2,119,125,48ml 2,アンギオ室では血栓除去術後に医師の口頭指示で、「ウロキナーゼ12万単位+生理食塩液48ml ...,"[118, 41]"


,marker_cases,unique_reports,mean_startidx,median_startidx
entitytype,,,,
Timing,4489,4475,5.434395,3.0
Date,3989,3905,6.170970,2.0
Drug,718,704,20.034819,2.0
Duration,303,303,6.603960,2.0
Strength_amount,299,265,95.742475,61.0
Dosage,67,63,71.104478,43.0
Form_form,62,59,92.693548,67.0
Strength_rate,39,37,70.000000,45.0
Frequency,35,34,18.885714,15.0


In [16]:
import pandas as pd
import numpy as np

# Use the existing audit dataframe
df = audit.copy()

def safe_text(x):
    return "" if pd.isna(x) else str(x)

def evaluate_offset_convention(row, shift=0, one_based=True):
    text = safe_text(row["report_text"])
    entity = safe_text(row["raw_entity"])

    # Remove the visible marker only for this diagnostic comparison
    entity = entity.replace("▁", "")

    start = int(row["startidx"])
    end = int(row["endidx"])

    if one_based:
        start = start - 1
        end = end - 1

    start += shift
    end += shift

    if start < 0 or end > len(text) or start >= end:
        return False

    return text[start:end] == entity

# Evaluate alternative conventions on marker-bearing annotations
marker_df = df[
    df["raw_entity"].astype(str).str.contains("▁", regex=False)
].copy()

tests = {
    "one_based_end_exclusive": lambda r: evaluate_offset_convention(
        r, shift=0, one_based=True
    ),
    "zero_based_end_exclusive": lambda r: evaluate_offset_convention(
        r, shift=0, one_based=False
    ),
    "one_based_shift_minus1": lambda r: evaluate_offset_convention(
        r, shift=-1, one_based=True
    ),
    "one_based_shift_plus1": lambda r: evaluate_offset_convention(
        r, shift=1, one_based=True
    ),
}

results = []

for name, fn in tests.items():
    matches = marker_df.apply(fn, axis=1).sum()
    results.append({
        "Convention": name,
        "Exact matches": matches,
        "Total": len(marker_df),
        "Match percentage": round(matches / len(marker_df) * 100, 2)
    })

display(pd.DataFrame(results))

,Convention,Exact matches,Total,Match percentage
0,one_based_end_exclusive,0,10047,0.0
1,zero_based_end_exclusive,0,10047,0.0
2,one_based_shift_minus1,0,10047,0.0
3,one_based_shift_plus1,0,10047,0.0


In [17]:
# Inspect selected marker-bearing examples directly
# Do not modify or correct any annotations.

sample_ids = [
    "A03AB2BE2AA38997A",
    "A0111289E37E07E2C",
    "A038D4E4CAFF5930E",
    "A0AAAD1EC192566A3"
]

for rid in sample_ids:
    rows = audit[audit["id"].astype(str) == rid]

    for _, r in rows.iterrows():
        text = str(r["report_text"])
        raw = str(r["raw_entity"])
        cleaned = raw.replace("▁", "")

        s = int(r["startidx"])
        e = int(r["endidx"])

        print("\n" + "=" * 80)
        print("ID:", rid)
        print("Entity type:", r["entitytype"])
        print("Raw entity:", repr(raw))
        print("Cleaned entity:", repr(cleaned))
        print("Entity length:", len(cleaned))
        print("Offsets:", s, e)
        print("Report beginning:", repr(text[:80]))

        print("Original audit extracted span:",
              repr(r.get("extracted_span", None)))

        print("Slice [start-1:end-1]:",
              repr(text[s-1:e-1]))

        print("Slice [start-1:end]:",
              repr(text[s-1:e]))

        print("Slice [start:end]:",
              repr(text[s:e]))

        positions = []
        pos = text.find(cleaned)

        while pos != -1:
            positions.append(pos)
            pos = text.find(cleaned, pos + 1)

        print("All cleaned-entity positions:", positions)


ID: A03AB2BE2AA38997A
Entity type: Strength_rate
Raw entity: '▁20ml/h」'
Cleaned entity: '20ml/h」'
Entity length: 7
Offsets: 102 110
Report beginning: '患者は胸部症状を主訴に循環器内科を受診した。検査の結果、急性冠動脈症候群の疑いで緊急心臓カテーテル検査の予定となった。12時20分頃、主治医より点滴「生食500'
Original audit extracted span: ' 20ml/h」'
Slice [start-1:end-1]: ' 20ml/h」'
Slice [start-1:end]: ' 20ml/h」の'
Slice [start:end]: '20ml/h」の'
All cleaned-entity positions: [102]

ID: A0111289E37E07E2C
Entity type: Drug
Raw entity: '▁オピ'
Cleaned entity: 'オピ'
Entity length: 2
Offsets: 2 5
Report beginning: 'オピスタンを静脈注射するところをサイレースを注射した。看護師がサイレースを台に準備しており、医師は確認せずに投与した。'
Original audit extracted span: 'ピスタ'
Slice [start-1:end-1]: 'ピスタ'
Slice [start-1:end]: 'ピスタン'
Slice [start:end]: 'スタン'
All cleaned-entity positions: [0]

ID: A038D4E4CAFF5930E
Entity type: Timing
Raw entity: '▁16時に'
Cleaned entity: '16時に'
Entity length: 4
Offsets: 3 8
Report beginning: '16時に内服する指示のリタリン10mg1錠を内服させ忘れた。'
Original audit extracted span: '時に内服す'
Slice [start-1:end-1]: '時に内服す'
Slice [star

In [18]:
import pandas as pd
import numpy as np

marker_df = audit[
    audit["raw_entity"].astype(str).str.contains("▁", regex=False)
].copy()

def all_occurrences(text, entity):
    positions = []
    if not entity:
        return positions

    start = 0
    while True:
        pos = text.find(entity, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1

    return positions

def inspect_marker_row(row):
    text = str(row["report_text"])
    raw = str(row["raw_entity"])

    variants = {
        "marker_as_space": raw.replace("▁", " "),
        "marker_removed": raw.replace("▁", ""),
        "raw_marker": raw
    }

    # Candidate offsets: one-based start, end-exclusive end
    supplied_start = int(row["startidx"]) - 1
    supplied_end = int(row["endidx"]) - 1

    extracted = text[supplied_start:supplied_end]

    result = {
        "id": row["id"],
        "entitytype": row["entitytype"],
        "raw_entity": raw,
        "extracted_span": extracted,
        "supplied_start": supplied_start,
        "supplied_end": supplied_end
    }

    for name, entity in variants.items():
        positions = all_occurrences(text, entity)

        result[name + "_positions"] = positions
        result[name + "_found"] = len(positions) > 0
        result[name + "_offset_match"] = (
            extracted == entity
        )

    return pd.Series(result)

marker_results = marker_df.apply(inspect_marker_row, axis=1)

summary = []

for variant in [
    "marker_as_space",
    "marker_removed",
    "raw_marker"
]:
    summary.append({
        "Representation": variant,
        "Found anywhere": marker_results[
            variant + "_found"
        ].sum(),
        "Matches supplied offsets": marker_results[
            variant + "_offset_match"
        ].sum()
    })

display(pd.DataFrame(summary))

print("Examples:")
display(marker_results.head(30))

,Representation,Found anywhere,Matches supplied offsets
0,marker_as_space,1057,1028
1,marker_removed,9435,0
2,raw_marker,0,0


Examples:


,id,entitytype,raw_entity,extracted_span,supplied_start,supplied_end,marker_as_space_positions,marker_as_space_found,marker_as_space_offset_match,marker_removed_positions,marker_removed_found,marker_removed_offset_match,raw_marker_positions,raw_marker_found,raw_marker_offset_match
2,A0111289E37E07E2C,Drug,▁オピ,ピスタ,1,4,[],False,False,[0],True,False,[],False,False
3,A01DE352B2048F6AB,Drug,▁高カロリー輸液,カロリー輸液によ,1,9,[],False,False,[0],True,False,[],False,False
4,A038D4E4CAFF5930E,Timing,▁16時に,時に内服す,2,7,[],False,False,[0],True,False,[],False,False
23,A03AB2BE2AA38997A,Strength_rate,▁20ml/h」,20ml/h」,101,109,[101],True,True,[102],True,False,[],False,False
24,A04ABFC6AAF9CC49B,Timing,▁18:30,:30、患者,2,8,[],False,False,[0],True,False,[],False,False
25,A04BECA552503940C,Date,▁前日に,日に調製,1,5,[],False,False,[0],True,False,[],False,False
50,A08C7339686D69F61,Timing,▁9時15分頃,"時15分頃,a",1,8,[],False,False,[0],True,False,[],False,False
51,A08E2FCE4F1B815FA,Timing,▁10:00頃,:00頃、メイ,2,9,[],False,False,[0],True,False,[],False,False
52,A0A5B2C4622CE17AF,Drug,リン1mg▁2錠調,リン1mg 2錠調,179,188,[179],True,True,[],False,False,[],False,False
53,A0AAAD1EC192566A3,Form_form,48ml▁2,48ml 2,118,124,"[41, 118]",True,True,[],False,False,[],False,False


In [19]:
# CELL 1 — Upload and extract the four Excel workbooks

from google.colab import files
from pathlib import Path
import zipfile, os, shutil

import_dir = Path("/content/medication_revision")
extract_dir = import_dir / "zip_datasets"
extract_dir.mkdir(parents=True, exist_ok=True)

print("Please upload the ZIP archive containing the four Excel files.")
uploaded_zip = files.upload()

zip_files = [name for name in uploaded_zip if name.lower().endswith(".zip")]

if not zip_files:
    raise ValueError("No ZIP file was uploaded. Please upload your dataset ZIP.")

for zip_name in zip_files:
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(extract_dir)

print("\nExtraction completed.")
print("\nExcel files found:")

zip_excel_paths = sorted(extract_dir.rglob("*.xlsx"))

for p in zip_excel_paths:
    print(f" - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

if not zip_excel_paths:
    raise FileNotFoundError("No Excel workbooks were found inside the ZIP.")

Please upload the ZIP archive containing the four Excel files.


Saving 23504922-1.zip to 23504922-1.zip

Extraction completed.

Excel files found:
 - CrossValid_IFMIR_522.xlsx (282.6 KB)
 - Error_analysis_10.xlsx (15.1 KB)
 - ExternalValid_JQ2021_20.xlsx (35.8 KB)
 - InternalValid_JQ2010-20_40.xlsx (67.6 KB)


In [20]:
# CELL 2 — Detect the existing AnnotatedDataset DataFrame

import pandas as pd
import numpy as np
import re
import hashlib
import json
import warnings
from collections import Counter, defaultdict

def normalize_column_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def find_column(df, aliases):
    cols = {normalize_column_name(c): c for c in df.columns}
    for alias in aliases:
        key = normalize_column_name(alias)
        if key in cols:
            return cols[key]
    return None

required_aliases = {
    "id": ["id", "reportid"],
    "text": ["report", "reports"],
    "entity": ["namedentity", "entityname", "entity_name"],
    "start": ["startidx", "start_idx", "start"],
    "end": ["endidx", "end_idx", "end"],
    "type": ["entitytype", "entity", "label"]
}

def is_annotation_dataframe(obj):
    if not isinstance(obj, pd.DataFrame):
        return False
    return all(find_column(obj, aliases) is not None
               for aliases in required_aliases.values())

# Automatically search existing notebook variables.
existing_candidates = []

for variable_name, obj in list(globals().items()):
    try:
        if is_annotation_dataframe(obj):
            existing_candidates.append((variable_name, obj))
    except Exception:
        pass

print("Potential annotation DataFrames already in memory:")

for name, df in existing_candidates:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Prefer a DataFrame whose variable name includes "annotated".
preferred = [
    (name, df) for name, df in existing_candidates
    if "annotated" in name.lower()
]

if preferred:
    MAIN_DATASET_NAME, MAIN_DF = max(preferred, key=lambda x: len(x[1]))
elif existing_candidates:
    MAIN_DATASET_NAME, MAIN_DF = max(
        existing_candidates, key=lambda x: len(x[1])
    )
else:
    MAIN_DATASET_NAME, MAIN_DF = None, None

if MAIN_DF is not None:
    print("\nSelected main dataset:", MAIN_DATASET_NAME)
    print("Shape:", MAIN_DF.shape)
    display(MAIN_DF.head(3))
else:
    print("\nNo existing main annotation DataFrame was detected.")
    print("Please upload AnnotatedDataset.xlsx now.")

    uploaded_main = files.upload()
    main_files = [
        name for name in uploaded_main
        if name.lower().endswith((".xlsx", ".xls"))
    ]

    if not main_files:
        raise ValueError("Please upload AnnotatedDataset.xlsx.")

    MAIN_DATASET_NAME = main_files[0]
    MAIN_DF = pd.read_excel(MAIN_DATASET_NAME)

    if not is_annotation_dataframe(MAIN_DF):
        raise ValueError(
            "The uploaded file does not contain the expected annotation columns."
        )

print("\nMain dataset is ready.")

Potential annotation DataFrames already in memory:
zero_label: 240 rows × 13 columns
d: 478,176 rows × 16 columns
m: 21,425 rows × 19 columns
q: 478,176 rows × 19 columns

Selected main dataset: d
Shape: (478176, 16)


,no,id,year,report,namedentity,startidx,endidx,entitytype,errorlabel,relation,incidenttypes,translated_report,translated_namedentity,extracted,exact_match,entity_occurrences_in_report
0,0,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ノルアドレナリン,1,9,Drug,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,Noradrenaline,ノルアドレナリン,True,2
1,1,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,0.1,12,15,Strength_amount,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,0.1,0.1,True,1
2,2,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,3ml/h,18,23,Strength_rate,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,3ml/h,3ml/h,True,3



Main dataset is ready.


In [21]:
# CELL 3 — Load all workbook sheets without combining datasets

ZIP_BOOKS = {}
WORKBOOK_INVENTORY = []

for path in zip_excel_paths:

    try:
        xls = pd.ExcelFile(path)

        for sheet_name in xls.sheet_names:

            df = pd.read_excel(path, sheet_name=sheet_name)

            key = f"{path.name}::{sheet_name}"
            ZIP_BOOKS[key] = df

            id_col = find_column(df, ["id", "reportid"])
            text_col = find_column(df, ["report", "reports"])

            WORKBOOK_INVENTORY.append({
                "workbook": path.name,
                "sheet": sheet_name,
                "rows": len(df),
                "columns": len(df.columns),
                "unique_report_ids": (
                    df[id_col].astype(str).nunique()
                    if id_col else None
                ),
                "unique_report_texts": (
                    df[text_col].astype(str).nunique()
                    if text_col else None
                ),
                "column_names": ", ".join(map(str, df.columns))
            })

    except Exception as e:
        print("Could not read:", path.name, str(e))

WORKBOOK_INVENTORY_DF = pd.DataFrame(WORKBOOK_INVENTORY)

print("Workbook inventory:")
display(WORKBOOK_INVENTORY_DF)

print("\nLoaded sheets:")
for key, df in ZIP_BOOKS.items():
    print(f"{key}: {df.shape}")

Workbook inventory:


,workbook,sheet,rows,columns,unique_report_ids,unique_report_texts,column_names
0,CrossValid_IFMIR_522.xlsx,Goldstandardreviewed,3125,11,517,517.0,"no, id, report, namedentity, startidx, endidx,..."
1,Error_analysis_10.xlsx,freetext,10,2,10,10.0,"id, report"
2,Error_analysis_10.xlsx,manualreviewed,13,14,10,10.0,"no, id, year, report, entity_name, start_idx, ..."
3,Error_analysis_10.xlsx,machineannotated,8,10,5,5.0,"Unnamed: 0, id, reports, entity_name, start_id..."
4,ExternalValid_JQ2021_20.xlsx,freetext,20,4,20,20.0,"evid, id, year, report"
5,ExternalValid_JQ2021_20.xlsx,manualreviewed,111,14,20,20.0,"no, evid, id, year, report, namedentity, start..."
6,ExternalValid_JQ2021_20.xlsx,machineannotated,104,10,19,19.0,"no, id, report, namedentity, startidx, endidx,..."
7,ExternalValid_JQ2021_20.xlsx,Incidenttype_evl,23,4,21,NaN,"no, id, reviewed_incidenttype(gold), incident_..."
8,InternalValid_JQ2010-20_40.xlsx,freetext,40,4,40,40.0,"evid, id, year, report"
9,InternalValid_JQ2010-20_40.xlsx,manuallyreviewed,263,13,40,40.0,"no, id, year, report, namedentity, startidx, e..."



Loaded sheets:
CrossValid_IFMIR_522.xlsx::Goldstandardreviewed: (3125, 11)
Error_analysis_10.xlsx::freetext: (10, 2)
Error_analysis_10.xlsx::manualreviewed: (13, 14)
Error_analysis_10.xlsx::machineannotated: (8, 10)
ExternalValid_JQ2021_20.xlsx::freetext: (20, 4)
ExternalValid_JQ2021_20.xlsx::manualreviewed: (111, 14)
ExternalValid_JQ2021_20.xlsx::machineannotated: (104, 10)
ExternalValid_JQ2021_20.xlsx::Incidenttype_evl: (23, 4)
InternalValid_JQ2010-20_40.xlsx::freetext: (40, 4)
InternalValid_JQ2010-20_40.xlsx::manuallyreviewed: (263, 13)
InternalValid_JQ2010-20_40.xlsx::machineannotated: (356, 10)
InternalValid_JQ2010-20_40.xlsx::incidenttype_evl: (43, 4)


In [22]:
# CELL 4 — Dataset profiling

REV_RESULTS = {}

def dataset_profile(df, dataset_name):

    id_col = find_column(df, ["id", "reportid"])
    text_col = find_column(df, ["report", "reports"])
    entity_col = find_column(df, ["namedentity", "entityname"])
    type_col = find_column(df, ["entitytype", "entity", "label"])

    result = {
        "dataset": dataset_name,
        "annotation_rows": len(df),
        "number_of_columns": len(df.columns),
        "unique_report_ids": (
            df[id_col].astype(str).nunique() if id_col else None
        ),
        "unique_report_texts": (
            df[text_col].astype(str).nunique() if text_col else None
        ),
        "unique_entities": (
            df[entity_col].nunique(dropna=True) if entity_col else None
        ),
        "missing_cells": int(df.isna().sum().sum())
    }

    if type_col:
        result["entity_type_distribution"] = (
            df[type_col].fillna("[MISSING]")
            .astype(str).value_counts().to_dict()
        )

    return result


ALL_DATASETS = {
    "MAIN_ANNOTATED_DATASET": MAIN_DF
}

ALL_DATASETS.update(ZIP_BOOKS)

PROFILE_RESULTS = []

for name, df in ALL_DATASETS.items():

    result = dataset_profile(df, name)
    PROFILE_RESULTS.append(result)

    print("\n", "=" * 70)
    print(name)
    print("=" * 70)

    for key, value in result.items():
        if key != "entity_type_distribution":
            print(f"{key}: {value}")

    if "entity_type_distribution" in result:
        print("\nEntity type distribution:")
        display(
            pd.DataFrame(
                result["entity_type_distribution"].items(),
                columns=["Entity type", "Count"]
            ).sort_values("Count", ascending=False)
        )

    missing = (
        df.isna().sum()
        .reset_index()
        .rename(columns={"index": "column", 0: "missing_count"})
    )
    missing["missing_percentage"] = (
        100 * missing["missing_count"] / len(df)
    )

    display(missing.sort_values("missing_count", ascending=False).head(20))

REV_RESULTS["profiles"] = PROFILE_RESULTS


MAIN_ANNOTATED_DATASET
dataset: MAIN_ANNOTATED_DATASET
annotation_rows: 478176
number_of_columns: 16
unique_report_ids: 58389
unique_report_texts: 58340
unique_entities: 37168
missing_cells: 72685

Entity type distribution:


,Entity type,Count
0,Drug,168633
1,Timing,90986
2,Strength_amount,59493
3,Date,40617
4,Route,36496
5,Form_form,22859
6,Dosage,22731
7,Duration,15127
8,Strength_rate,8084
9,Frequency,7289


,column,missing_count,missing_percentage
8,errorlabel,71909,15.038187
9,relation,776,0.162283
0,no,0,0.000000
1,id,0,0.000000
4,namedentity,0,0.000000
5,startidx,0,0.000000
2,year,0,0.000000
3,report,0,0.000000
7,entitytype,0,0.000000
6,endidx,0,0.000000



CrossValid_IFMIR_522.xlsx::Goldstandardreviewed
dataset: CrossValid_IFMIR_522.xlsx::Goldstandardreviewed
annotation_rows: 3125
number_of_columns: 11
unique_report_ids: 517
unique_report_texts: 517
unique_entities: 1225
missing_cells: 630

Entity type distribution:


,Entity type,Count
0,Drug,1093
1,Strength_amount,645
2,Timing,313
3,Form_form,248
4,Dosage,228
5,Date,171
6,Route,139
7,Duration,93
8,Frequency,80
9,Strength_rate,71


,column,missing_count,missing_percentage
7,errorlabel,630,20.16
1,id,0,0.00
0,no,0,0.00
2,report,0,0.00
3,namedentity,0,0.00
5,endidx,0,0.00
4,startidx,0,0.00
6,entitytype,0,0.00
8,relation,0,0.00
9,translated_report,0,0.00


... (24 outputs left out)

,Entity type,Count
0,Drug,112
1,Timing,61
2,Strength_amount,39
3,Date,29
4,Dosage,25
5,Duration,24
6,Route,24
7,Form_form,23
8,Frequency,8
9,Strength_concentration,6


,column,missing_count,missing_percentage
7,errorlabel,76,21.348315
0,no,0,0.000000
1,id,0,0.000000
2,report,0,0.000000
4,startidx,0,0.000000
3,namedentity,0,0.000000
5,endidx,0,0.000000
6,entitytype,0,0.000000
8,relation,0,0.000000
9,incidenttypes,0,0.000000



InternalValid_JQ2010-20_40.xlsx::incidenttype_evl
dataset: InternalValid_JQ2010-20_40.xlsx::incidenttype_evl
annotation_rows: 43
number_of_columns: 4
unique_report_ids: 41
unique_report_texts: None
unique_entities: None
missing_cells: 11


,column,missing_count,missing_percentage
3,incident_types (predict),4,9.302326
0,no,3,6.976744
1,id,3,6.976744
2,reviewed_incidenttype(gold),1,2.325581


In [23]:
# CELL 5 — Duplicate and overlap audit

def normalize_report_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).casefold().split())

def get_report_keys(df):

    id_col = find_column(df, ["id", "reportid"])
    text_col = find_column(df, ["report", "reports"])

    if not id_col and not text_col:
        return None

    result = pd.DataFrame()

    if id_col:
        result["report_id"] = df[id_col].astype(str).str.strip()

    if text_col:
        result["normalized_text"] = df[text_col].map(
            normalize_report_text
        )

        result["text_hash"] = result["normalized_text"].map(
            lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()
            if x else None
        )

    return result


REPORT_KEYS = {
    name: get_report_keys(df)
    for name, df in ALL_DATASETS.items()
}

OVERLAP_ROWS = []

names = list(REPORT_KEYS.keys())

for i in range(len(names)):

    for j in range(i + 1, len(names)):

        name_a, name_b = names[i], names[j]
        a, b = REPORT_KEYS[name_a], REPORT_KEYS[name_b]

        if a is None or b is None:
            continue

        shared_ids = 0
        shared_texts = 0

        if "report_id" in a.columns and "report_id" in b.columns:
            shared_ids = len(
                set(a["report_id"].dropna()) &
                set(b["report_id"].dropna())
            )

        if "text_hash" in a.columns and "text_hash" in b.columns:
            shared_texts = len(
                set(a["text_hash"].dropna()) &
                set(b["text_hash"].dropna())
            )

        OVERLAP_ROWS.append({
            "dataset_A": name_a,
            "dataset_B": name_b,
            "shared_report_IDs": shared_ids,
            "shared_exact_normalized_texts": shared_texts
        })

OVERLAP_DF = pd.DataFrame(OVERLAP_ROWS)

print("Cross-dataset overlap:")
display(OVERLAP_DF)

REV_RESULTS["overlap"] = OVERLAP_DF.to_dict("records")

# Within-dataset duplicate report summary
DUPLICATE_ROWS = []

for name, keys in REPORT_KEYS.items():

    if keys is None:
        continue

    DUPLICATE_ROWS.append({
        "dataset": name,
        "rows": len(keys),
        "unique_report_IDs": (
            keys["report_id"].nunique()
            if "report_id" in keys else None
        ),
        "unique_report_texts": (
            keys["text_hash"].nunique()
            if "text_hash" in keys else None
        ),
        "duplicate_text_rows": (
            int(keys["text_hash"].duplicated(keep=False).sum())
            if "text_hash" in keys else None
        )
    })

DUPLICATE_DF = pd.DataFrame(DUPLICATE_ROWS)

print("Within-dataset duplicate report summary:")
display(DUPLICATE_DF)

REV_RESULTS["duplicates"] = DUPLICATE_DF.to_dict("records")

Cross-dataset overlap:


,dataset_A,dataset_B,shared_report_IDs,shared_exact_normalized_texts
0,MAIN_ANNOTATED_DATASET,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,514,471
1,MAIN_ANNOTATED_DATASET,Error_analysis_10.xlsx::freetext,0,0
2,MAIN_ANNOTATED_DATASET,Error_analysis_10.xlsx::manualreviewed,0,0
3,MAIN_ANNOTATED_DATASET,Error_analysis_10.xlsx::machineannotated,0,0
4,MAIN_ANNOTATED_DATASET,ExternalValid_JQ2021_20.xlsx::freetext,0,0
...,...,...,...,...
73,InternalValid_JQ2010-20_40.xlsx::freetext,InternalValid_JQ2010-20_40.xlsx::machineannotated,39,39
74,InternalValid_JQ2010-20_40.xlsx::freetext,InternalValid_JQ2010-20_40.xlsx::incidenttype_evl,40,0
75,InternalValid_JQ2010-20_40.xlsx::manuallyreviewed,InternalValid_JQ2010-20_40.xlsx::machineannotated,39,39
76,InternalValid_JQ2010-20_40.xlsx::manuallyreviewed,InternalValid_JQ2010-20_40.xlsx::incidenttype_evl,40,0


Within-dataset duplicate report summary:


,dataset,rows,unique_report_IDs,unique_report_texts,duplicate_text_rows
0,MAIN_ANNOTATED_DATASET,478176,58389,58340.0,475561.0
1,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,3125,517,517.0,3124.0
2,Error_analysis_10.xlsx::freetext,10,10,10.0,0.0
3,Error_analysis_10.xlsx::manualreviewed,13,10,10.0,5.0
4,Error_analysis_10.xlsx::machineannotated,8,5,5.0,5.0
5,ExternalValid_JQ2021_20.xlsx::freetext,20,20,20.0,0.0
6,ExternalValid_JQ2021_20.xlsx::manualreviewed,111,20,20.0,109.0
7,ExternalValid_JQ2021_20.xlsx::machineannotated,104,19,19.0,103.0
8,ExternalValid_JQ2021_20.xlsx::Incidenttype_evl,23,21,NaN,NaN
9,InternalValid_JQ2010-20_40.xlsx::freetext,40,40,40.0,0.0


In [24]:
# CELL 6 — Entity span alignment audit

def audit_entity_spans(df, dataset_name):

    id_col = find_column(df, ["id", "reportid"])
    text_col = find_column(df, ["report", "reports"])
    entity_col = find_column(df, ["namedentity", "entityname"])
    start_col = find_column(df, ["startidx", "start_idx", "start"])
    end_col = find_column(df, ["endidx", "end_idx", "end"])
    type_col = find_column(df, ["entitytype", "entity", "label"])

    if not all([text_col, entity_col, start_col, end_col]):
        return None

    temp = df[
        [c for c in [
            id_col, text_col, entity_col,
            start_col, end_col, type_col
        ] if c is not None]
    ].copy()

    valid = (
        temp[text_col].notna() &
        temp[entity_col].notna() &
        temp[start_col].notna() &
        temp[end_col].notna()
    )

    temp = temp.loc[valid].copy()

    temp["_text"] = temp[text_col].astype(str)
    temp["_entity"] = temp[entity_col].astype(str)
    temp["_start"] = pd.to_numeric(
        temp[start_col], errors="coerce"
    )
    temp["_end"] = pd.to_numeric(
        temp[end_col], errors="coerce"
    )

    temp = temp.dropna(subset=["_start", "_end"])

    # Convention: one-based start, end-exclusive end.
    temp["_a"] = temp["_start"].astype(int) - 1
    temp["_b"] = temp["_end"].astype(int) - 1

    def check(row, marker_mode):

        entity = row["_entity"]

        if marker_mode == "space":
            entity = entity.replace("▁", " ")
        elif marker_mode == "removed":
            entity = entity.replace("▁", "")

        a, b = row["_a"], row["_b"]
        text = row["_text"]

        if a < 0 or b < a or b > len(text):
            return False

        return text[a:b] == entity

    temp["exact_raw_match"] = temp.apply(
        lambda r: check(r, "raw"), axis=1
    )

    temp["exact_marker_as_space"] = temp.apply(
        lambda r: check(r, "space"), axis=1
    )

    temp["exact_marker_removed"] = temp.apply(
        lambda r: check(r, "removed"), axis=1
    )

    temp["marker_present"] = temp["_entity"].str.contains(
        "▁", regex=False
    )

    summary = {
        "dataset": dataset_name,
        "annotations_audited": len(temp),
        "exact_raw_match": int(temp["exact_raw_match"].sum()),
        "exact_raw_match_pct": round(
            100 * temp["exact_raw_match"].mean(), 3
        ),
        "exact_marker_as_space": int(
            temp["exact_marker_as_space"].sum()
        ),
        "exact_marker_as_space_pct": round(
            100 * temp["exact_marker_as_space"].mean(), 3
        ),
        "exact_marker_removed": int(
            temp["exact_marker_removed"].sum()
        ),
        "exact_marker_removed_pct": round(
            100 * temp["exact_marker_removed"].mean(), 3
        ),
        "marker_present_rows": int(temp["marker_present"].sum())
    }

    by_type = None

    if type_col:
        temp["_type"] = temp[type_col].astype(str)

        by_type = (
            temp.groupby("_type")
            .agg(
                annotations=("_type", "size"),
                raw_matches=("exact_raw_match", "sum"),
                marker_space_matches=("exact_marker_as_space", "sum")
            )
            .reset_index()
            .rename(columns={"_type": "entity_type"})
        )

        by_type["raw_match_pct"] = (
            100 * by_type["raw_matches"] / by_type["annotations"]
        )

    return summary, by_type, temp


SPAN_SUMMARIES = []
SPAN_DETAILS = {}

for name, df in ALL_DATASETS.items():

    result = audit_entity_spans(df, name)

    if result is None:
        continue

    summary, by_type, details = result

    SPAN_SUMMARIES.append(summary)
    SPAN_DETAILS[name] = details

    print("\nSpan alignment:", name)
    display(pd.DataFrame([summary]))

    if by_type is not None:
        display(by_type.sort_values("annotations", ascending=False))

SPAN_SUMMARY_DF = pd.DataFrame(SPAN_SUMMARIES)

REV_RESULTS["span_summary"] = SPAN_SUMMARY_DF.to_dict("records")

print("\nOverall span audit summary:")
display(SPAN_SUMMARY_DF)


Span alignment: MAIN_ANNOTATED_DATASET


,dataset,annotations_audited,exact_raw_match,exact_raw_match_pct,exact_marker_as_space,exact_marker_as_space_pct,exact_marker_removed,exact_marker_removed_pct,marker_present_rows
0,MAIN_ANNOTATED_DATASET,478176,456751,95.519,457779,95.734,456751,95.519,10047


,entity_type,annotations,raw_matches,marker_space_matches,raw_match_pct
2,Drug,168633,163899,163998,97.192720
11,Timing,90986,84510,84802,92.882421
8,Strength_amount,59493,57786,58061,97.130755
0,Date,40617,35619,35756,87.694808
7,Route,36496,35446,35450,97.122972
4,Form_form,22859,22219,22281,97.200227
1,Dosage,22731,22151,22211,97.448418
3,Duration,15127,14487,14508,95.769154
10,Strength_rate,8084,7898,7930,97.699159
6,Frequency,7289,7075,7095,97.064069



Span alignment: CrossValid_IFMIR_522.xlsx::Goldstandardreviewed


,dataset,annotations_audited,exact_raw_match,exact_raw_match_pct,exact_marker_as_space,exact_marker_as_space_pct,exact_marker_removed,exact_marker_removed_pct,marker_present_rows
0,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,3125,0,0.0,0,0.0,0,0.0,0


,entity_type,annotations,raw_matches,marker_space_matches,raw_match_pct
2,Drug,1093,0,0,0.0
8,Strength_amount,645,0,0,0.0
11,Timing,313,0,0,0.0
4,Form_form,248,0,0,0.0
1,Dosage,228,0,0,0.0
0,Date,171,0,0,0.0
7,Route,139,0,0,0.0
3,Duration,93,0,0,0.0
6,Frequency,80,0,0,0.0
10,Strength_rate,71,0,0,0.0


... (16 outputs left out)

,dataset,annotations_audited,exact_raw_match,exact_raw_match_pct,exact_marker_as_space,exact_marker_as_space_pct,exact_marker_removed,exact_marker_removed_pct,marker_present_rows
0,InternalValid_JQ2010-20_40.xlsx::machineannotated,356,0,0.0,0,0.0,0,0.0,8


,entity_type,annotations,raw_matches,marker_space_matches,raw_match_pct
2,Drug,112,0,0,0.0
10,Timing,61,0,0,0.0
7,Strength_amount,39,0,0,0.0
0,Date,29,0,0,0.0
1,Dosage,25,0,0,0.0
3,Duration,24,0,0,0.0
6,Route,24,0,0,0.0
4,Form_form,23,0,0,0.0
5,Frequency,8,0,0,0.0
8,Strength_concentration,6,0,0,0.0



Overall span audit summary:


,dataset,annotations_audited,exact_raw_match,exact_raw_match_pct,exact_marker_as_space,exact_marker_as_space_pct,exact_marker_removed,exact_marker_removed_pct,marker_present_rows
0,MAIN_ANNOTATED_DATASET,478176,456751,95.519,457779,95.734,456751,95.519,10047
1,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,3125,0,0.000,0,0.000,0,0.000,0
2,Error_analysis_10.xlsx::manualreviewed,13,0,0.000,0,0.000,0,0.000,0
3,Error_analysis_10.xlsx::machineannotated,8,0,0.000,0,0.000,0,0.000,0
4,ExternalValid_JQ2021_20.xlsx::manualreviewed,111,0,0.000,0,0.000,0,0.000,0
5,ExternalValid_JQ2021_20.xlsx::machineannotated,104,0,0.000,0,0.000,0,0.000,2
6,InternalValid_JQ2010-20_40.xlsx::manuallyreviewed,263,0,0.000,0,0.000,0,0.000,0
7,InternalValid_JQ2010-20_40.xlsx::machineannotated,356,0,0.000,0,0.000,0,0.000,8


In [25]:
# CELL 7 — Incident-type evaluation

def parse_labels(value):

    if pd.isna(value):
        return set()

    return {
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    }


INCIDENT_RESULTS = []

for name, df in ZIP_BOOKS.items():

    gold_col = find_column(
        df,
        ["reviewed_incidenttype(gold)",
         "reviewedincidenttypegold",
         "incidenttypegold",
         "goldincidenttype"]
    )

    pred_col = find_column(
        df,
        ["incident_types (predict)",
         "incidenttypespredict",
         "predictedincidenttypes",
         "incidenttypeprediction"]
    )

    if not gold_col or not pred_col:
        continue

    paired = df[[gold_col, pred_col]].dropna().copy()

    if len(paired) == 0:
        continue

    TP = FP = FN = 0
    exact = 0

    for _, row in paired.iterrows():

        gold = parse_labels(row[gold_col])
        pred = parse_labels(row[pred_col])

        TP += len(gold & pred)
        FP += len(pred - gold)
        FN += len(gold - pred)

        if gold == pred:
            exact += 1

    precision = TP / (TP + FP) if TP + FP else np.nan
    recall = TP / (TP + FN) if TP + FN else np.nan

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else np.nan
    )

    result = {
        "dataset_sheet": name,
        "paired_reports": len(paired),
        "exact_set_matches": exact,
        "exact_set_accuracy": exact / len(paired),
        "micro_TP": TP,
        "micro_FP": FP,
        "micro_FN": FN,
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_F1": f1
    }

    INCIDENT_RESULTS.append(result)

INCIDENT_RESULTS_DF = pd.DataFrame(INCIDENT_RESULTS)

print("Incident-type evaluation:")
if len(INCIDENT_RESULTS_DF):
    display(INCIDENT_RESULTS_DF)
else:
    print(
        "No workbook sheet with compatible gold and predicted "
        "incident-type columns was detected."
    )

REV_RESULTS["incident_evaluation"] = (
    INCIDENT_RESULTS_DF.to_dict("records")
)

Incident-type evaluation:


,dataset_sheet,paired_reports,exact_set_matches,exact_set_accuracy,micro_TP,micro_FP,micro_FN,micro_precision,micro_recall,micro_F1
0,ExternalValid_JQ2021_20.xlsx::Incidenttype_evl,19,9,0.473684,12,10,15,0.545455,0.444444,0.489796
1,InternalValid_JQ2010-20_40.xlsx::incidenttype_evl,39,23,0.589744,43,19,15,0.693548,0.741379,0.716667


In [26]:
# CELL 8 — Manual versus machine entity comparison

def entity_set_by_report(df):

    idc = find_column(df, ["id", "reportid"])
    textc = find_column(df, ["report", "reports"])
    entc = find_column(df, ["namedentity", "entityname"])
    sc = find_column(df, ["startidx", "start_idx", "start"])
    ec = find_column(df, ["endidx", "end_idx", "end"])
    lc = find_column(df, ["entitytype", "entity", "label"])

    if not all([idc, textc, entc, sc, ec, lc]):
        return None

    out = defaultdict(set)

    for _, r in df.iterrows():

        if any(pd.isna(r[c]) for c in [idc, textc, entc, sc, ec, lc]):
            continue

        try:
            a = int(r[sc]) - 1
            b = int(r[ec]) - 1
        except Exception:
            continue

        text = str(r[textc])
        entity = str(r[entc]).replace("▁", " ")

        # Retain only spans that match under this diagnostic convention.
        if 0 <= a < b <= len(text) and text[a:b] == entity:
            out[str(r[idc]).strip()].add(
                (a, b, str(r[lc]).strip())
            )

    return out


ENTITY_COMPARISONS = []

for workbook in sorted(set(k[0] for k in ZIP_BOOKS)):

    sheets = [
        (key, df) for key, df in ZIP_BOOKS.items()
        if key.startswith(workbook + "::")
    ]

    manual_sheets = [
        (key, df) for key, df in sheets
        if any(x in key.lower() for x in [
            "manual", "goldstandard", "reviewed"
        ])
    ]

    machine_sheets = [
        (key, df) for key, df in sheets
        if any(x in key.lower() for x in [
            "machine", "predicted", "prediction"
        ])
    ]

    for mkey, mdf in manual_sheets:

        for pkey, pdf in machine_sheets:

            gold = entity_set_by_report(mdf)
            pred = entity_set_by_report(pdf)

            if gold is None or pred is None:
                continue

            common_ids = set(gold) & set(pred)

            TP = FP = FN = 0

            for rid in common_ids:

                TP += len(gold[rid] & pred[rid])
                FP += len(pred[rid] - gold[rid])
                FN += len(gold[rid] - pred[rid])

            precision = TP / (TP + FP) if TP + FP else np.nan
            recall = TP / (TP + FN) if TP + FN else np.nan

            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0 else np.nan
            )

            ENTITY_COMPARISONS.append({
                "workbook": workbook,
                "manual_sheet": mkey,
                "machine_sheet": pkey,
                "common_report_ids": len(common_ids),
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "precision": precision,
                "recall": recall,
                "strict_micro_F1": f1
            })

ENTITY_COMPARISONS_DF = pd.DataFrame(ENTITY_COMPARISONS)

print("Manual versus machine annotation comparisons:")

if len(ENTITY_COMPARISONS_DF):
    display(ENTITY_COMPARISONS_DF)
else:
    print(
        "No compatible manual/machine entity sheet pairs were detected. "
        "No comparison has been invented."
    )

REV_RESULTS["entity_comparisons"] = (
    ENTITY_COMPARISONS_DF.to_dict("records")
)

Manual versus machine annotation comparisons:
No compatible manual/machine entity sheet pairs were detected. No comparison has been invented.


In [27]:
# CELL 9 — Minority category and error-label analysis

MINORITY_RESULTS = []
ERROR_LABEL_RESULTS = []

for name, df in ALL_DATASETS.items():

    type_col = find_column(df, ["entitytype", "entity", "label"])

    if type_col:

        counts = df[type_col].fillna("[MISSING]").astype(str).value_counts()

        for label, count in counts.items():

            MINORITY_RESULTS.append({
                "dataset": name,
                "entity_type": label,
                "count": int(count),
                "percentage": 100 * count / len(df)
            })

    error_col = find_column(df, ["errorlabel", "error_label"])

    if error_col:

        counts = (
            df[error_col].fillna("[MISSING]")
            .astype(str).value_counts()
        )

        for label, count in counts.items():

            ERROR_LABEL_RESULTS.append({
                "dataset": name,
                "error_label": label,
                "count": int(count),
                "percentage": 100 * count / len(df)
            })


MINORITY_DF = pd.DataFrame(MINORITY_RESULTS)
ERROR_LABEL_DF = pd.DataFrame(ERROR_LABEL_RESULTS)

print("Entity category distribution:")
display(MINORITY_DF.sort_values(
    ["dataset", "count"], ascending=[True, True]
))

print("\nObserved error-label distribution:")
if len(ERROR_LABEL_DF):
    display(ERROR_LABEL_DF)
else:
    print("No errorlabel columns were detected.")

REV_RESULTS["minority_categories"] = MINORITY_DF.to_dict("records")
REV_RESULTS["error_labels"] = ERROR_LABEL_DF.to_dict("records")

Entity category distribution:


,dataset,entity_type,count,percentage
24,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,Form_mode,17,0.544000
23,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,Strength_concentration,27,0.864000
22,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,Strength_rate,71,2.272000
21,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,Frequency,80,2.560000
20,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,Duration,93,2.976000
...,...,...,...,...
4,MAIN_ANNOTATED_DATASET,Route,36496,7.632336
3,MAIN_ANNOTATED_DATASET,Date,40617,8.494153
2,MAIN_ANNOTATED_DATASET,Strength_amount,59493,12.441653
1,MAIN_ANNOTATED_DATASET,Timing,90986,19.027722



Observed error-label distribution:


,dataset,error_label,count,percentage
0,MAIN_ANNOTATED_DATASET,IA,348088,72.794954
1,MAIN_ANNOTATED_DATASET,[MISSING],71909,15.038187
2,MAIN_ANNOTATED_DATASET,IN,57939,12.116668
3,MAIN_ANNOTATED_DATASET,0,240,0.050191
4,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,IA,1730,55.360000
5,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,[MISSING],630,20.160000
6,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,IN,602,19.264000
7,CrossValid_IFMIR_522.xlsx::Goldstandardreviewed,NN,163,5.216000
8,ExternalValid_JQ2021_20.xlsx::manualreviewed,IA,71,63.963964
9,ExternalValid_JQ2021_20.xlsx::manualreviewed,[MISSING],23,20.720721


In [28]:
# CELL 10 — Export the reviewer evidence report as one PDF

!pip -q install reportlab matplotlib

import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, PageBreak, Image as RLImage
)
from reportlab.lib.enums import TA_CENTER
from pathlib import Path
import json, os

PDF_PATH = "/content/Medication_Dataset_Reviewer_Evidence.pdf"
FIG_DIR = Path("/content/medication_revision_figures")
FIG_DIR.mkdir(exist_ok=True)

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="ReportTitle",
    parent=styles["Title"],
    fontSize=19,
    leading=23,
    alignment=TA_CENTER,
    spaceAfter=12
))

styles.add(ParagraphStyle(
    name="SectionHeading",
    parent=styles["Heading2"],
    fontSize=13,
    leading=16,
    spaceBefore=12,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="SmallText",
    parent=styles["BodyText"],
    fontSize=8,
    leading=10
))

story = []

def add_paragraph(text, style="BodyText"):
    story.append(Paragraph(str(text), styles[style]))
    story.append(Spacer(1, 5))

def safe_value(x):
    if x is None:
        return "Not available"
    if isinstance(x, float) and np.isnan(x):
        return "Not available"
    return str(x)

def add_dataframe_table(df, max_rows=35):

    if df is None or len(df) == 0:
        add_paragraph("No compatible data were available for this analysis.")
        return

    display_df = df.head(max_rows).copy()

    data = [
        [Paragraph(str(c), styles["SmallText"])
         for c in display_df.columns]
    ]

    for _, row in display_df.iterrows():
        data.append([
            Paragraph(
                safe_value(v)[:180],
                styles["SmallText"]
            )
            for v in row.tolist()
        ])

    available_width = landscape(A4)[0] - 28 * mm
    ncols = len(display_df.columns)
    col_width = available_width / max(ncols, 1)

    table = Table(
        data,
        repeatRows=1,
        colWidths=[col_width] * ncols
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4)
    ]))

    story.append(table)
    story.append(Spacer(1, 10))


# Cover
add_paragraph(
    "Medication Incident Dataset: Reviewer Evidence Report",
    "ReportTitle"
)

add_paragraph(
    "Automatically generated from the existing Colab AnnotatedDataset "
    "and the uploaded ZIP workbooks."
)

add_paragraph(
    f"Main dataset variable: {MAIN_DATASET_NAME}"
)

add_paragraph(
    f"Main dataset size: {len(MAIN_DF):,} annotation rows."
)

add_paragraph(
    f"ZIP workbooks/sheets loaded: {len(ZIP_BOOKS)}."
)

story.append(PageBreak())

# Dataset inventory
add_paragraph("1. Dataset and workbook inventory", "SectionHeading")
add_dataframe_table(WORKBOOK_INVENTORY_DF)

# Dataset profiles
add_paragraph("2. Dataset profiles", "SectionHeading")

profile_table = pd.DataFrame([
    {
        "dataset": p["dataset"],
        "annotation_rows": p["annotation_rows"],
        "unique_report_ids": p["unique_report_ids"],
        "unique_report_texts": p["unique_report_texts"],
        "unique_entities": p["unique_entities"],
        "missing_cells": p["missing_cells"]
    }
    for p in PROFILE_RESULTS
])

add_dataframe_table(profile_table)

# Overlap
add_paragraph("3. Duplicate reports and cross-dataset overlap", "SectionHeading")
add_dataframe_table(OVERLAP_DF)
add_dataframe_table(DUPLICATE_DF)

# Span audit
add_paragraph("4. Entity span and offset consistency", "SectionHeading")
add_dataframe_table(SPAN_SUMMARY_DF)

# Incident evaluation
add_paragraph("5. Incident-type classification evaluation", "SectionHeading")
add_dataframe_table(INCIDENT_RESULTS_DF)

# Entity comparisons
add_paragraph("6. Manual versus machine entity comparisons", "SectionHeading")
add_dataframe_table(ENTITY_COMPARISONS_DF)

# Minority categories
add_paragraph("7. Minority entity categories", "SectionHeading")
add_dataframe_table(
    MINORITY_DF.sort_values(
        ["dataset", "count"], ascending=[True, True]
    ) if len(MINORITY_DF) else MINORITY_DF
)

# Error labels
add_paragraph("8. Error-label distribution", "SectionHeading")
add_dataframe_table(ERROR_LABEL_DF)

# Scientific limitations
story.append(PageBreak())

add_paragraph("9. Interpretation and scientific limitations", "SectionHeading")

limitations = [
    "This report summarizes analyses performed on the uploaded workbooks "
    "and the existing main AnnotatedDataset DataFrame.",

    "Report-ID or exact-text overlap is evidence of possible dataset overlap. "
    "It does not establish patient-level independence or annotation independence.",

    "Exact span alignment measures consistency between stored offsets and "
    "text. It is not a measure of inter-annotator agreement or clinical correctness.",

    "The one-based, end-exclusive offset convention and the interpretation "
    "of the special marker must be confirmed against the original dataset documentation.",

    "Manual-versus-machine comparisons are valid only if the sheets genuinely "
    "contain paired annotations for the same reports and the annotation protocol "
    "supports the intended interpretation.",

    "Incident-type metrics are calculated only where both reviewed and predicted "
    "labels are present in the same sheet.",

    "No model baseline, LoRA ablation, ontology ablation, RAG experiment, "
    "or cross-dataset model generalization result is created by this audit.",

    "No independent inter-annotator agreement, clinical utility, or external "
    "clinical validation is inferred from these descriptive analyses.",

    "The results should be checked against the source annotation guidelines "
    "before being presented as final manuscript findings."
]

for item in limitations:
    add_paragraph("• " + item)

# Machine-readable appendix
story.append(PageBreak())
add_paragraph("10. Analysis configuration and available evidence", "SectionHeading")

add_paragraph(
    "This report was generated from the current Colab session. "
    "The source Excel files were not modified."
)

add_paragraph(
    "The report intentionally distinguishes calculated results from "
    "analyses that require additional model predictions or annotation documentation."
)

doc = SimpleDocTemplate(
    PDF_PATH,
    pagesize=landscape(A4),
    rightMargin=14*mm,
    leftMargin=14*mm,
    topMargin=14*mm,
    bottomMargin=14*mm
)

doc.build(story)

# Save the complete structured results alongside the PDF.
JSON_PATH = "/content/Medication_Dataset_Reviewer_Evidence.json"

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(REV_RESULTS, f, ensure_ascii=False, indent=2, default=str)

print("PDF successfully generated:", PDF_PATH)
print("JSON results generated:", JSON_PATH)

files.download(PDF_PATH)

PDF successfully generated: /content/Medication_Dataset_Reviewer_Evidence.pdf
JSON results generated: /content/Medication_Dataset_Reviewer_Evidence.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
# CELL 0 — Clone and inspect the existing project

import os
import subprocess
from pathlib import Path

REPO_URL = (
    "https://github.com/SharareTaheriMoghadam/"
    "Patient-Safety-Concept-Extraction-with-LoRA-tuned-LLaMA-"
    "Knowledge-Graphs-GNN-and-Causal-Modeling.git"
)

PROJECT_DIR = Path("/content/PatientSafetyConceptExtraction")

if PROJECT_DIR.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull"],
        check=False
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )

print("PROJECT DIRECTORY:", PROJECT_DIR)

# Display project files
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and ".git" not in path.parts:
        print(path.relative_to(PROJECT_DIR))

# Display README files
readme_files = list(PROJECT_DIR.glob("README*"))

for readme in readme_files:
    print("\n" + "=" * 80)
    print("README:", readme.name)
    print("=" * 80)
    print(readme.read_text(encoding="utf-8", errors="ignore")[:20000])

# Inspect dependency files
for filename in [
    "requirements.txt",
    "environment.yml",
    "pyproject.toml",
    "setup.py"
]:
    file_path = PROJECT_DIR / filename
    if file_path.exists():
        print("\nDEPENDENCIES:", filename)
        print(file_path.read_text(encoding="utf-8", errors="ignore")[:12000])

# Identify relevant source files
extensions = {".py", ".ipynb", ".yaml", ".yml", ".json", ".owl", ".ttl", ".rdf"}

source_files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
    and ".git" not in p.parts
]

print("\nTOTAL RELEVANT SOURCE FILES:", len(source_files))

for p in source_files:
    print(p.relative_to(PROJECT_DIR))

PROJECT DIRECTORY: /content/PatientSafetyConceptExtraction
README.md
annotation/auto_annotator.py
annotation/weak_labeler.py
causal/ate_estimator.py
causal/causal_pipeline.py
causal/propensity_model.py
config/model.yaml
deployment/api_service.py
deployment/fhir_adapter.py
deployment/model_monitoring.py
evaluation/decision_curve_analysis.py
evaluation/metrics.py
evaluation/ontology_similarity.py
evaluation/statistical_testing.py
gnn/graph_builder.py
gnn/temporal_gnn.py
kg/kg_builder.py
kg/ontology_fusion.py
models/llama_loader.py
models/multi_agent_reasoner.py
models/temporal_predictor.py
multi_agent_reasoner.py
pipeline/full_pipeline.py
rag/rag_engine.py
rag/vector_store.py
requirements.txt
training/lora_finetune.py

README: README.md
# Patient-Safety-Concept-Extraction-with-LoRA-tuned-LLaMA-Knowledge-Graphs-GNN-and-Causal-Modeling
SafeGraph-RAG-Temporal is a next-generation ontology-driven patient safety intelligence framework integrating: Instruction-tuned LLaMA with LoRA fine-tuning

In [30]:
# CELL 1 — Install analysis and ontology libraries

!pip -q install rdflib owlready2 pandas numpy scikit-learn \
    transformers datasets evaluate seqeval accelerate peft \
    sentencepiece matplotlib seaborn openpyxl requests

import json
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("/content/reviewer_evidence")
ONTOLOGY_DIR = OUTPUT_DIR / "ontologies"
RESULTS_DIR = OUTPUT_DIR / "results"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
FIGURES_DIR = OUTPUT_DIR / "figures"

for folder in [
    OUTPUT_DIR,
    ONTOLOGY_DIR,
    RESULTS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Evidence directory:", OUTPUT_DIR)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Evidence directory: /content/reviewer_evidence


In [32]:

# CELL 2 — Download publicly available MSCDO and ICPS
# No BioPortal API key required.

import requests
from pathlib import Path

ONTOLOGY_DIR = Path("/content/reviewer_evidence/ontologies")
ONTOLOGY_DIR.mkdir(parents=True, exist_ok=True)

ONTOLOGIES = {
    "MSCDO": {
        "page": "https://bioportal.bioontology.org/ontologies/MSCDO",
        "download": "https://data.bioontology.org/ontologies/MSCDO/download"
    },
    "ICPS": {
        "page": "https://bioportal.bioontology.org/ontologies/ICPS",
        "download": "https://data.bioontology.org/ontologies/ICPS/download"
    }
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; OntologyResearch/1.0)"
})

download_results = []

for acronym, info in ONTOLOGIES.items():

    output_file = ONTOLOGY_DIR / f"{acronym}.owl"

    try:
        response = session.get(
            info["download"],
            timeout=120,
            allow_redirects=True
        )

        response.raise_for_status()

        if len(response.content) < 100:
            raise ValueError("Downloaded content is unexpectedly small.")

        # Save the public response for inspection.
        output_file.write_bytes(response.content)

        download_results.append({
            "ontology": acronym,
            "source": info["download"],
            "status": "Downloaded",
            "file": str(output_file),
            "size_bytes": len(response.content),
            "content_type": response.headers.get("Content-Type")
        })

    except Exception as e:

        download_results.append({
            "ontology": acronym,
            "source": info["download"],
            "status": "Download failed",
            "file": "",
            "size_bytes": 0,
            "content_type": "",
            "error": str(e)
        })

download_df = pd.DataFrame(download_results)
display(download_df)

download_df.to_csv(
    "/content/reviewer_evidence/results/ontology_download_status.csv",
    index=False
)

,ontology,source,status,file,size_bytes,content_type,error
0,MSCDO,https://data.bioontology.org/ontologies/MSCDO/...,Download failed,,0,,401 Client Error: Unauthorized for url: https:...
1,ICPS,https://data.bioontology.org/ontologies/ICPS/d...,Download failed,,0,,401 Client Error: Unauthorized for url: https:...


In [33]:
# CELL 3 — Parse and profile MSCDO and ICPS

from rdflib import Graph, RDF, RDFS, OWL, SKOS, URIRef, Literal

ontology_graphs = {}
ontology_profiles = {}

for acronym in ONTOLOGIES:

    owl_file = ONTOLOGY_DIR / f"{acronym}.owl"

    if not owl_file.exists():
        print(f"{acronym}: ontology file not found; skipping.")
        continue

    graph = Graph()
    graph.parse(str(owl_file), format="xml")

    ontology_graphs[acronym] = graph

    classes = set(graph.subjects(RDF.type, OWL.Class))
    classes.update(graph.subjects(RDF.type, RDFS.Class))

    object_properties = set(
        graph.subjects(RDF.type, OWL.ObjectProperty)
    )

    datatype_properties = set(
        graph.subjects(RDF.type, OWL.DatatypeProperty)
    )

    profile = {
        "ontology": acronym,
        "triples": len(graph),
        "classes": len(classes),
        "object_properties": len(object_properties),
        "datatype_properties": len(datatype_properties),
        "classes_with_label": 0,
        "classes_with_definition": 0
    }

    for cls in classes:
        labels = list(graph.objects(cls, RDFS.label))
        definitions = (
            list(graph.objects(cls, SKOS.definition))
            + list(graph.objects(cls, URIRef(
                "http://purl.obolibrary.org/obo/IAO_0000115"
            )))
        )

        if labels:
            profile["classes_with_label"] += 1

        if definitions:
            profile["classes_with_definition"] += 1

    ontology_profiles[acronym] = profile

profile_df = pd.DataFrame(ontology_profiles.values())
display(profile_df)

profile_df.to_csv(
    RESULTS_DIR / "ontology_profile.csv",
    index=False
)

MSCDO: ontology file not found; skipping.
ICPS: ontology file not found; skipping.


""


In [34]:
# CELL 4 — Extract ontology labels, synonyms, definitions and URIs

from rdflib.namespace import SKOS

SYNONYM_PREDICATES = [
    SKOS.altLabel,
    URIRef("http://www.geneontology.org/formats/oboInOwl#hasExactSynonym"),
    URIRef("http://www.geneontology.org/formats/oboInOwl#hasRelatedSynonym"),
    URIRef("http://www.geneontology.org/formats/oboInOwl#hasBroadSynonym"),
    URIRef("http://www.geneontology.org/formats/oboInOwl#hasNarrowSynonym")
]

DEFINITION_PREDICATES = [
    SKOS.definition,
    URIRef("http://purl.obolibrary.org/obo/IAO_0000115")
]

term_records = []

for acronym, graph in ontology_graphs.items():

    classes = set(graph.subjects(RDF.type, OWL.Class))
    classes.update(graph.subjects(RDF.type, RDFS.Class))

    for cls in classes:

        labels = list(graph.objects(cls, RDFS.label))

        synonyms = []
        for predicate in SYNONYM_PREDICATES:
            synonyms.extend(graph.objects(cls, predicate))

        definitions = []
        for predicate in DEFINITION_PREDICATES:
            definitions.extend(graph.objects(cls, predicate))

        term_records.append({
            "ontology": acronym,
            "class_uri": str(cls),
            "preferred_label": str(labels[0]) if labels else "",
            "all_labels": " | ".join(str(x) for x in labels),
            "synonyms": " | ".join(sorted(set(str(x) for x in synonyms))),
            "definitions": " | ".join(sorted(set(str(x) for x in definitions)))
        })

ontology_terms = pd.DataFrame(term_records)

display(ontology_terms.head(20))
print("Total extracted ontology classes:", len(ontology_terms))

ontology_terms.to_csv(
    RESULTS_DIR / "MSCDO_ICPS_ontology_terms.csv",
    index=False
)

""


Total extracted ontology classes: 0


In [35]:
# CELL 5 — Extract ontology relationships

relationship_records = []

for acronym, graph in ontology_graphs.items():

    for subject, predicate, obj in graph:

        if (
            isinstance(subject, URIRef)
            and isinstance(obj, URIRef)
            and predicate not in [RDF.type]
        ):

            relationship_records.append({
                "ontology": acronym,
                "subject_uri": str(subject),
                "predicate_uri": str(predicate),
                "object_uri": str(obj)
            })

relationships_df = pd.DataFrame(relationship_records)

display(relationships_df.head(30))

print("Total relationships:", len(relationships_df))

relationships_df.to_csv(
    RESULTS_DIR / "MSCDO_ICPS_relationships.csv",
    index=False
)

""


Total relationships: 0


In [37]:
# DIAGNOSTIC — Inspect ontology extraction outputs

print("combined_terms type:", type(combined_terms))
print("combined_terms shape:", combined_terms.shape)
print("combined_terms columns:", combined_terms.columns.tolist())

print("\nAvailable variables:")
for name in [
    "mscdо_terms", "mscdo_terms", "icps_terms",
    "mscdо_classes", "mscdo_classes", "icps_classes",
    "mscdо_df", "mscdo_df", "icps_df"
]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj), getattr(obj, "shape", None))
        if isinstance(obj, pd.DataFrame):
            print("  Columns:", obj.columns.tolist())

combined_terms type: <class 'pandas.core.frame.DataFrame'>
combined_terms shape: (0, 0)
combined_terms columns: []

Available variables:


In [42]:
# CELL 2 — Upload MSCDO and ICPS ontology files

from google.colab import files
from pathlib import Path
import shutil

ONTOLOGY_DIR = Path("/content/reviewer_evidence/ontologies")
ONTOLOGY_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for filename, content in uploaded.items():

    name = filename.lower()

    if "mscdo" in name:
        target = ONTOLOGY_DIR / "MSCDO.owl"

    elif "icps" in name:
        target = ONTOLOGY_DIR / "ICPS.owl"

    else:
        print("Skipped unidentified file:", filename)
        continue

    target.write_bytes(content)
    print("Saved:", target, "|", target.stat().st_size, "bytes")

print("\nFinal ontology files:")

for f in ONTOLOGY_DIR.iterdir():
    print(f.name, f.stat().st_size, "bytes")

Saving PatientSafetyIncident.zip to PatientSafetyIncident.zip
Saving mscdo owl format V3-1.txt to mscdo owl format V3-1.txt
Skipped unidentified file: PatientSafetyIncident.zip
Saved: /content/reviewer_evidence/ontologies/MSCDO.owl | 41864 bytes

Final ontology files:
MSCDO.owl 41864 bytes


In [43]:

# RECOVERY CELL — Validate MSCDO and inspect the ICPS ZIP

from pathlib import Path
from zipfile import ZipFile, BadZipFile
from rdflib import Graph
import pandas as pd
import re

ONTOLOGY_DIR = Path("/content/reviewer_evidence/ontologies")
ONTOLOGY_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Validate MSCDO
# --------------------------------------------------

mscdo_file = ONTOLOGY_DIR / "MSCDO.owl"

if mscdo_file.exists():
    content = mscdo_file.read_bytes()

    print("MSCDO file size:", len(content))
    print("MSCDO preview:")
    print(content[:500].decode("utf-8", errors="replace"))

    parsed_graph = None
    parsed_format = None

    for fmt in ["xml", "turtle", "json-ld", "nt", "n3"]:
        try:
            graph = Graph()
            graph.parse(data=content, format=fmt)

            if len(graph) > 0:
                parsed_graph = graph
                parsed_format = fmt
                break
        except Exception:
            pass

    if parsed_graph is not None:
        print("\nMSCDO successfully parsed!")
        print("Format:", parsed_format)
        print("RDF triples:", len(parsed_graph))
    else:
        print("\nMSCDO could not yet be parsed as RDF.")
else:
    print("MSCDO file not found.")

# --------------------------------------------------
# 2. Recover the uploaded ZIP
# --------------------------------------------------

zip_path = Path("/content/PatientSafetyIncident.zip")

if not zip_path.exists() and "uploaded" in globals():
    if "PatientSafetyIncident.zip" in uploaded:
        zip_path.write_bytes(uploaded["PatientSafetyIncident.zip"])
        print("\nRecovered ZIP from upload variable.")

# --------------------------------------------------
# 3. Inspect ZIP contents
# --------------------------------------------------

if zip_path.exists():
    print("\nZIP size:", zip_path.stat().st_size)

    try:
        with ZipFile(zip_path, "r") as z:
            members = z.namelist()

            print("\nFiles inside PatientSafetyIncident.zip:")
            for member in members:
                info = z.getinfo(member)
                print(f"{member} | {info.file_size:,} bytes")

            # Extract safely, preventing path traversal
            for member in members:
                if member.endswith("/"):
                    continue

                destination = (ONTOLOGY_DIR / member).resolve()

                if not destination.is_relative_to(ONTOLOGY_DIR.resolve()):
                    print("Skipped unsafe archive path:", member)
                    continue

                destination.parent.mkdir(parents=True, exist_ok=True)

                with z.open(member) as source, open(destination, "wb") as target:
                    target.write(source.read())

            print("\nZIP extraction completed.")

    except BadZipFile:
        print("ERROR: The uploaded file is not a valid ZIP archive.")
else:
    print("\nZIP file is not available in the current Colab session.")

# --------------------------------------------------
# 4. List extracted files
# --------------------------------------------------

print("\nFinal ontology directory contents:")

for f in ONTOLOGY_DIR.rglob("*"):
    if f.is_file():
        print(f.relative_to(ONTOLOGY_DIR), "|", f.stat().st_size, "bytes")

MSCDO file size: 41864
MSCDO preview:
@prefix : <http://example.org/msdo#MSCDO> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix atc: <http://www.whocc.no/atc_ddd_index/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xml: <http://www.w3.org/XML/1998/namespace> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix icps: <http://www.who.int/patientsafety/taxonomy/> .
@prefix msdo: <http://example.org/msdo#> .
@prefix rdfs: <http://www.w3.org/2000

MSCDO successfully parsed!
Format: turtle
RDF triples: 488

ZIP size: 84425

Files inside PatientSafetyIncident.zip:
Anatomy.owl | 42,494 bytes
BuildingStructure.owl | 15,079 bytes
Clothing.owl | 9,159 bytes
ContributingFactors-Hazards.owl | 8,184 bytes
Countries.owl | 339,835 bytes
Detection.owl | 6,476 bytes
DetectionMechanisms.owl | 15,899 bytes
Furniture.owl | 8,088 bytes
HospitalEquipment.owl | 14,346 bytes
IncidentCharacteristic.owl | 11,452 bytes
In

In [44]:

# REPLACEMENT CELLS 3–6
# Extract ontology concepts from MSCDO and PatientSafetyIncident modules

!pip -q install rdflib pandas openpyxl

from pathlib import Path
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, SKOS
import pandas as pd
import re

BASE_DIR = Path("/content/reviewer_evidence")
ONTOLOGY_DIR = BASE_DIR / "ontologies"
RESULTS_DIR = BASE_DIR / "results"

ONTOLOGY_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Identify ontology files
# --------------------------------------------------

ontology_sources = []

mscdo_file = ONTOLOGY_DIR / "MSCDO.owl"

if mscdo_file.exists():
    ontology_sources.append(("MSCDO", mscdo_file))

# The ZIP contains multiple Patient Safety Incident modules.
# Retain module-level provenance rather than assuming all
# modules are a single independently validated ICPS release.

for path in sorted(ONTOLOGY_DIR.rglob("*.owl")):
    if path.name == "MSCDO.owl":
        continue

    ontology_sources.append(("PatientSafetyIncident", path))

print("Ontology files identified:", len(ontology_sources))

for source, path in ontology_sources:
    print(source, "|", path.name, "|", path.stat().st_size, "bytes")

# --------------------------------------------------
# 2. RDF parsing helper
# --------------------------------------------------

def parse_rdf_file(path):
    content = path.read_bytes()

    for fmt in ["xml", "turtle", "json-ld", "nt", "n3"]:
        try:
            graph = Graph()
            graph.parse(data=content, format=fmt)

            if len(graph) > 0:
                return graph, fmt

        except Exception:
            continue

    return None, None

# --------------------------------------------------
# 3. Identify label, synonym and definition properties
# --------------------------------------------------

def local_name(uri):
    text = str(uri)
    return re.split(r"[/#:] ", text)[-1] if False else re.split(r"[/#]", text)[-1]

def literal_values(graph, subject, predicate):
    return [
        str(value).strip()
        for value in graph.objects(subject, predicate)
        if isinstance(value, Literal) and str(value).strip()
    ]

def get_values_by_local_name(graph, subject, keywords):
    values = []

    for predicate in set(graph.predicates(subject, None)):
        pname = local_name(predicate).lower().replace("_", "").replace("-", "")

        if any(keyword in pname for keyword in keywords):
            values.extend(literal_values(graph, subject, predicate))

    return sorted(set(values))

# Preferred label candidates
PREFERRED_PREDICATES = [
    RDFS.label,
    SKOS.prefLabel
]

# --------------------------------------------------
# 4. Extract concepts from every ontology file
# --------------------------------------------------

all_terms = []
parse_log = []

for source_name, path in ontology_sources:

    graph, rdf_format = parse_rdf_file(path)

    if graph is None:
        parse_log.append({
            "source": source_name,
            "file": path.name,
            "status": "Parsing failed",
            "format": None,
            "triples": 0
        })
        continue

    parse_log.append({
        "source": source_name,
        "file": path.name,
        "status": "Parsed",
        "format": rdf_format,
        "triples": len(graph)
    })

    # Collect entities explicitly declared as classes,
    # plus subjects having recognised label properties.
    subjects = set()

    for cls in [OWL.Class, RDFS.Class]:
        subjects.update(graph.subjects(RDF.type, cls))

    for predicate in PREFERRED_PREDICATES:
        subjects.update(graph.subjects(predicate, None))

    # Also include labelled individuals/concepts.
    for subject in subjects:

        if not isinstance(subject, URIRef):
            continue

        labels = []

        for predicate in PREFERRED_PREDICATES:
            labels.extend(literal_values(graph, subject, predicate))

        labels = sorted(set(labels))

        # Fallback to local URI name only when no explicit label exists.
        if not labels:
            continue

        synonyms = get_values_by_local_name(
            graph, subject,
            ["synonym", "alternative label", "altlabel"]
        )

        definitions = (
            literal_values(graph, subject, SKOS.definition)
            + get_values_by_local_name(
                graph, subject,
                ["definition", "description", "scope note"]
            )
        )

        all_terms.append({
            "ontology": source_name,
            "source_file": path.name,
            "concept_uri": str(subject),
            "preferred_label": labels[0],
            "all_labels": " | ".join(labels),
            "synonyms": " | ".join(sorted(set(synonyms))),
            "definition": " | ".join(sorted(set(definitions)))
        })

# --------------------------------------------------
# 5. Create combined ontology term table
# --------------------------------------------------

combined_terms = pd.DataFrame(all_terms)

if combined_terms.empty:
    raise ValueError(
        "No labelled ontology concepts were extracted. "
        "Inspect the parsing log and RDF predicates."
    )

def normalise_term(value):
    value = str(value).lower().strip()
    value = re.sub(r"[_\-]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value

combined_terms["normalised_label"] = (
    combined_terms["preferred_label"].apply(normalise_term)
)

combined_terms = combined_terms.drop_duplicates(
    subset=["ontology", "source_file", "concept_uri"]
).reset_index(drop=True)

parse_df = pd.DataFrame(parse_log)

# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------

combined_terms.to_csv(
    RESULTS_DIR / "combined_ontology_terms.csv",
    index=False,
    encoding="utf-8-sig"
)

parse_df.to_csv(
    RESULTS_DIR / "ontology_parsing_log.csv",
    index=False
)

print("\n========== EXTRACTION SUMMARY ==========")
print("Total extracted concepts:", len(combined_terms))

print("\nConcepts by source:")
display(
    combined_terms.groupby("ontology")
    .agg(
        concepts=("concept_uri", "nunique"),
        modules=("source_file", "nunique")
    )
)

print("\nParsing summary:")
display(parse_df)

print("\nSample extracted terms:")
display(combined_terms.head(15))

print("\nFiles saved successfully.")

Ontology files identified: 26
MSCDO | MSCDO.owl | 41864 bytes
PatientSafetyIncident | ActionsToReduceRisk.owl | 46571 bytes
PatientSafetyIncident | Activity.owl | 17455 bytes
PatientSafetyIncident | AmelioratingActions.owl | 9337 bytes
PatientSafetyIncident | Anatomy.owl | 42494 bytes
PatientSafetyIncident | BuildingStructure.owl | 15079 bytes
PatientSafetyIncident | Clothing.owl | 9159 bytes
PatientSafetyIncident | ContributingFactors-Hazards.owl | 8184 bytes
PatientSafetyIncident | Countries.owl | 339835 bytes
PatientSafetyIncident | Detection.owl | 6476 bytes
PatientSafetyIncident | DetectionMechanisms.owl | 15899 bytes
PatientSafetyIncident | Furniture.owl | 8088 bytes
PatientSafetyIncident | HospitalEquipment.owl | 14346 bytes
PatientSafetyIncident | IncidentCharacteristic.owl | 11452 bytes
PatientSafetyIncident | InferredAnatomyv3.owl | 37005 bytes
PatientSafetyIncident | Medication.owl | 4005 bytes
PatientSafetyIncident | MitigatingFactors.owl | 13720 bytes
PatientSafetyIncident

,concepts,modules
ontology,,
MSCDO,6,1
PatientSafetyIncident,1317,21



Parsing summary:


,source,file,status,format,triples
0,MSCDO,MSCDO.owl,Parsed,turtle,488
1,PatientSafetyIncident,ActionsToReduceRisk.owl,Parsed,xml,456
2,PatientSafetyIncident,Activity.owl,Parsed,xml,222
3,PatientSafetyIncident,AmelioratingActions.owl,Parsed,xml,82
4,PatientSafetyIncident,Anatomy.owl,Parsed,xml,582
5,PatientSafetyIncident,BuildingStructure.owl,Parsed,xml,186
6,PatientSafetyIncident,Clothing.owl,Parsed,xml,106
7,PatientSafetyIncident,ContributingFactors-Hazards.owl,Parsed,xml,58
8,PatientSafetyIncident,Countries.owl,Parsed,xml,3732
9,PatientSafetyIncident,Detection.owl,Parsed,xml,47



Sample extracted terms:


,ontology,source_file,concept_uri,preferred_label,all_labels,synonyms,definition,normalised_label
0,MSCDO,MSCDO.owl,http://example.org/msdo#PatientContext,Patient Context,Patient Context,,,patient context
1,MSCDO,MSCDO.owl,http://example.org/msdo#MedicationRelatedRisk,Medication-Related Risk,Medication-Related Risk,,,medication related risk
2,MSCDO,MSCDO.owl,http://example.org/msdo#PatientSafetyPrevention,Patient Safety and Prevention,Patient Safety and Prevention,,,patient safety and prevention
3,MSCDO,MSCDO.owl,http://example.org/msdo#HumanAIInteraction,Human–AI Interaction,Human–AI Interaction,,,human–ai interaction
4,MSCDO,MSCDO.owl,http://example.org/msdo#MedicationEntity,Medication Entity,Medication Entity,,,medication entity
5,MSCDO,MSCDO.owl,http://example.org/msdo#DecisionArtifact,Decision Artifact,Decision Artifact,,,decision artifact
6,PatientSafetyIncident,ActionsToReduceRisk.owl,http://www.ICPS/ontologies/ActionsToReduceRisk...,Water,Water,,,water
7,PatientSafetyIncident,ActionsToReduceRisk.owl,http://www.ICPS/ontologies/ActionsToReduceRisk...,Mobility aid lifter,Mobility aid lifter,,,mobility aid lifter
8,PatientSafetyIncident,ActionsToReduceRisk.owl,http://www.ICPS/ontologies/ActionsToReduceRisk...,Nutritional support pressure ulcer minimisatio...,Nutritional support pressure ulcer minimisatio...,,,nutritional support pressure ulcer minimisatio...
9,PatientSafetyIncident,ActionsToReduceRisk.owl,http://www.ICPS/ontologies/ActionsToReduceRisk...,Innes,Innes,,,innes



Files saved successfully.


In [46]:
import os
import glob

print("Excel files found in /content:\n")

for f in glob.glob("/content/**/*.xlsx", recursive=True):
    print(f)

Excel files found in /content:

/content/AnnotatedDataset.xlsx
/content/med_error_data/AnnotatedDataset.xlsx
/content/medication_revision/zip_datasets/ExternalValid_JQ2021_20.xlsx
/content/medication_revision/zip_datasets/Error_analysis_10.xlsx
/content/medication_revision/zip_datasets/InternalValid_JQ2010-20_40.xlsx
/content/medication_revision/zip_datasets/CrossValid_IFMIR_522.xlsx


In [47]:
import pandas as pd
import os

MAIN_DATA_PATH = "/content/AnnotatedDataset.xlsx"

assert os.path.exists(MAIN_DATA_PATH), (
    f"File not found: {MAIN_DATA_PATH}"
)

main_df = pd.read_excel(MAIN_DATA_PATH, sheet_name="Sheet1")

print("Dataset loaded successfully.")
print("Shape:", main_df.shape)
print("Columns:", main_df.columns.tolist())
display(main_df.head())

Dataset loaded successfully.
Shape: (478176, 13)
Columns: ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']


,no,id,year,report,namedentity,startidx,endidx,entitytype,errorlabel,relation,incidenttypes,translated_report,translated_namedentity
0,0,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,ノルアドレナリン,1,9,Drug,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,Noradrenaline
1,1,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,0.1,12,15,Strength_amount,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,0.1
2,2,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,3ml/h,18,23,Strength_rate,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,3ml/h
3,3,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,13:55,46,51,Timing,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,13:55
4,4,A001502FDA86DD17F,2019,ノルアドレナリンを時速0.15γ(3ml/h)で投与中。日中は血圧120-130台で経過。1...,シ,52,53,Route,IA,1.0,Wrong Strength_rate,Noradrenaline is being administered at a rate ...,Si


In [48]:
required = [
    "id",
    "report",
    "namedentity",
    "startidx",
    "endidx",
    "entitytype"
]

missing = [c for c in required if c not in main_df.columns]

if missing:
    print("Still missing:", missing)
else:
    print("Excellent! All required columns are available.")

Excellent! All required columns are available.


In [50]:
import os
import glob

files = (
    glob.glob("/content/**/*.xlsx", recursive=True)
    + glob.glob("/content/**/*.xls", recursive=True)
)

print("Excel files currently available:\n")
for i, f in enumerate(files, 1):
    print(f"{i}. {f}")

print("\nCurrent main_df:")
if "main_df" in globals():
    print("Shape:", main_df.shape)
    print("Columns:", main_df.columns.tolist())
else:
    print("Not defined")

Excel files currently available:

1. /content/AnnotatedDataset.xlsx
2. /content/med_error_data/AnnotatedDataset.xlsx
3. /content/medication_revision/zip_datasets/ExternalValid_JQ2021_20.xlsx
4. /content/medication_revision/zip_datasets/Error_analysis_10.xlsx
5. /content/medication_revision/zip_datasets/InternalValid_JQ2010-20_40.xlsx
6. /content/medication_revision/zip_datasets/CrossValid_IFMIR_522.xlsx

Current main_df:
Shape: (43, 4)
Columns: ['no', 'id', 'reviewed_incidenttype(gold)', 'incident_types_(predict)']


In [51]:
import os
import pandas as pd

candidate_paths = [
    "/content/med_error_data/AnnotatedDataset.xlsx",
    "/content/AnnotatedDataset.xlsx"
]

required_columns = {
    "id", "report", "namedentity",
    "startidx", "endidx", "entitytype"
}

valid_datasets = []

for path in candidate_paths:
    if not os.path.exists(path):
        print(f"\nNot found: {path}")
        continue

    print(f"\nChecking: {path}")

    excel = pd.ExcelFile(path)
    print("Sheets:", excel.sheet_names)

    for sheet in excel.sheet_names:
        temp = pd.read_excel(path, sheet_name=sheet)
        columns = set(temp.columns.astype(str).str.strip())

        print(f"  Sheet: {sheet} | Shape: {temp.shape}")
        print("  Columns:", temp.columns.tolist())

        if required_columns.issubset(columns):
            valid_datasets.append((path, sheet, temp))

if not valid_datasets:
    raise ValueError(
        "Neither workbook contains all required entity-annotation columns. "
        "Please share the diagnostic output."
    )

# Select the matching sheet with the most annotation rows
selected = max(valid_datasets, key=lambda x: len(x[2]))

MAIN_DATA_PATH, MAIN_SHEET, main_df = selected

print("\n" + "=" * 60)
print("MAIN ENTITY DATASET LOADED")
print("File:", MAIN_DATA_PATH)
print("Sheet:", MAIN_SHEET)
print("Shape:", main_df.shape)
print("Columns:", main_df.columns.tolist())
print("=" * 60)

assert required_columns.issubset(set(main_df.columns))


Checking: /content/med_error_data/AnnotatedDataset.xlsx
Sheets: ['Sheet1']
  Sheet: Sheet1 | Shape: (478176, 13)
  Columns: ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']

Checking: /content/AnnotatedDataset.xlsx
Sheets: ['Sheet1']
  Sheet: Sheet1 | Shape: (478176, 13)
  Columns: ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']

MAIN ENTITY DATASET LOADED
File: /content/med_error_data/AnnotatedDataset.xlsx
Sheet: Sheet1
Shape: (478176, 13)
Columns: ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']


In [53]:

# CELL 7 — Dataset loading and ontology-to-annotation mapping

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

BASE_DIR = Path("/content/reviewer_evidence")
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Locate the main annotated workbook
possible_files = (
    list(Path("/content").rglob("AnnotatedDataset.xlsx"))
    + list(Path("/content").rglob("AnnotatedDataset.xls"))
)

if "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    main_df = df.copy()
elif possible_files:
    main_df = pd.read_excel(possible_files[0], sheet_name=0)
    print("Loaded:", possible_files[0])
else:
    raise FileNotFoundError(
        "AnnotatedDataset.xlsx was not found. Upload it to Colab "
        "or assign its contents to a DataFrame named df."
    )

# Standardise column names
main_df.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in main_df.columns
]

required_columns = [
    "id", "report", "namedentity", "startidx",
    "endidx", "entitytype"
]

# Accommodate standardised underscore variants
rename_candidates = {
    "named_entity": "namedentity",
    "entity_type": "entitytype",
    "start_idx": "startidx",
    "end_idx": "endidx"
}
main_df = main_df.rename(columns=rename_candidates)

missing = [c for c in required_columns if c not in main_df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Available columns: {main_df.columns.tolist()}"
    )

# Keep IDs as strings
main_df["id"] = main_df["id"].astype(str)

# Preserve original entity labels and create normalised versions
def normalise_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[_\-]+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x

main_df["entitytype_norm"] = main_df["entitytype"].apply(normalise_text)
main_df["namedentity_norm"] = main_df["namedentity"].apply(normalise_text)

# Create a dataset entity-type inventory
entity_inventory = (
    main_df.groupby("entitytype", dropna=False)
    .agg(
        annotation_count=("entitytype", "size"),
        unique_reports=("id", "nunique")
    )
    .reset_index()
    .sort_values("annotation_count", ascending=False)
)

display(entity_inventory)

entity_inventory.to_csv(
    RESULTS_DIR / "dataset_entity_inventory.csv",
    index=False
)

# Build a mapping template, not an automatic semantic equivalence claim
ontology_categories = (
    combined_terms.groupby("ontology")["preferred_label"]
    .apply(list)
    .to_dict()
)

mapping_rows = []

for entity_type in sorted(main_df["entitytype"].dropna().unique()):
    mapping_rows.append({
        "dataset_entity_type": entity_type,
        "normalised_entity_type": normalise_text(entity_type),
        "candidate_ontology": "",
        "candidate_concept_label": "",
        "concept_uri": "",
        "mapping_status": "To be reviewed by domain expert",
        "mapping_notes": ""
    })

mapping_template = pd.DataFrame(mapping_rows)

mapping_template.to_csv(
    RESULTS_DIR / "ontology_mapping_template.csv",
    index=False
)

print("\nDataset shape:", main_df.shape)
print("Unique report IDs:", main_df["id"].nunique())
print("Ontology concepts available:", len(combined_terms))
print("Mapping template saved.")

ValueError: Missing required columns: ['report', 'namedentity', 'startidx', 'endidx', 'entitytype']. Available columns: ['no', 'id', 'reviewed_incidenttype(gold)', 'incident_types_(predict)']

In [54]:
# ============================================================
# CELL 7 — LOAD MAIN DATASET, VALIDATE AND PREPARE INVENTORIES
# ============================================================

import os
import pandas as pd
import numpy as np
import re

# 1. Define paths explicitly
BASE_DIR = "/content/med_error_data"
RESULTS_DIR = "/content/reviewer_evidence/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

MAIN_DATA_PATH = os.path.join(BASE_DIR, "AnnotatedDataset.xlsx")

if not os.path.exists(MAIN_DATA_PATH):
    raise FileNotFoundError(
        f"Main annotated dataset not found: {MAIN_DATA_PATH}"
    )

# 2. Always reload the correct main entity dataset.
# This prevents an old incident-type DataFrame from being reused.
main_df = pd.read_excel(
    MAIN_DATA_PATH,
    sheet_name="Sheet1"
)

# 3. Standardise column names
main_df.columns = (
    main_df.columns.astype(str).str.strip().str.lower()
)

# 4. Validate required columns
required_columns = [
    "id",
    "report",
    "namedentity",
    "startidx",
    "endidx",
    "entitytype"
]

missing = [
    col for col in required_columns
    if col not in main_df.columns
]

if missing:
    raise ValueError(
        f"Main dataset is missing columns: {missing}\n"
        f"Available columns: {main_df.columns.tolist()}"
    )

# 5. Confirm that this is the expected main dataset
if len(main_df) < 100000:
    raise ValueError(
        f"Unexpectedly small dataset: {len(main_df)} rows. "
        "Please check the input file."
    )

print("MAIN DATASET LOADED AND VALIDATED")
print("File:", MAIN_DATA_PATH)
print("Shape:", main_df.shape)
print("Columns:", main_df.columns.tolist())

# 6. Text normalisation function (also used by later cells)
def normalise_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    return value

# 7. Standardise annotation fields
main_df["id"] = main_df["id"].astype(str).str.strip()
main_df["entitytype_norm"] = (
    main_df["entitytype"].fillna("").astype(str).str.strip()
)
main_df["namedentity_norm"] = (
    main_df["namedentity"].apply(normalise_text)
)

# 8. Create entity category inventory
entity_inventory = (
    main_df.groupby("entitytype_norm", dropna=False)
    .agg(
        annotation_count=("id", "size"),
        unique_reports=("id", "nunique")
    )
    .reset_index()
    .rename(columns={"entitytype_norm": "entity_type"})
    .sort_values("annotation_count", ascending=False)
)

entity_inventory.to_csv(
    os.path.join(RESULTS_DIR, "dataset_entity_inventory.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 9. Create ontology mapping template
mapping_columns = [
    "ontology",
    "source_file",
    "concept_uri",
    "preferred_label",
    "all_labels",
    "synonyms",
    "definition",
    "normalised_label"
]

if "combined_terms" in globals() and isinstance(combined_terms, pd.DataFrame):
    available_mapping_columns = [
        col for col in mapping_columns
        if col in combined_terms.columns
    ]

    mapping_template = combined_terms[
        available_mapping_columns
    ].copy()

    for col in mapping_columns:
        if col not in mapping_template.columns:
            mapping_template[col] = ""

    mapping_template = mapping_template[mapping_columns]

    mapping_template["manual_mapping_status"] = "Not reviewed"
    mapping_template["matched_dataset_entity_type"] = ""
    mapping_template["expert_notes"] = ""

else:
    print(
        "WARNING: combined_terms is not available. "
        "Rerun ontology extraction Cells 2–6 before ontology mapping."
    )

    mapping_template = pd.DataFrame(
        columns=mapping_columns + [
            "manual_mapping_status",
            "matched_dataset_entity_type",
            "expert_notes"
        ]
    )

mapping_template.to_csv(
    os.path.join(RESULTS_DIR, "ontology_mapping_template.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 10. Save a small verification report
print("\nENTITY INVENTORY")
display(entity_inventory)

print("\nDATASET VERIFICATION")
print("Total annotation rows:", len(main_df))
print("Unique report IDs:", main_df["id"].nunique())
print("Unique entity categories:", main_df["entitytype_norm"].nunique())
print("Ontology terms available:", len(mapping_template))

print("\nCell 7 completed successfully.")
print("Results directory:", RESULTS_DIR)

MAIN DATASET LOADED AND VALIDATED
File: /content/med_error_data/AnnotatedDataset.xlsx
Shape: (478176, 13)
Columns: ['no', 'id', 'year', 'report', 'namedentity', 'startidx', 'endidx', 'entitytype', 'errorlabel', 'relation', 'incidenttypes', 'translated_report', 'translated_namedentity']

ENTITY INVENTORY


,entity_type,annotation_count,unique_reports
2,Drug,168633,53703
11,Timing,90986,35374
8,Strength_amount,59493,23537
0,Date,40617,20907
7,Route,36496,17551
4,Form_form,22859,12252
1,Dosage,22731,11804
3,Duration,15127,10316
10,Strength_rate,8084,3988
6,Frequency,7289,4889



DATASET VERIFICATION
Total annotation rows: 478176
Unique report IDs: 58389
Unique entity categories: 13
Ontology terms available: 1323

Cell 7 completed successfully.
Results directory: /content/reviewer_evidence/results


In [56]:
# ============================================================
# CELL 8 — FOCUSED ONTOLOGY EXTRACTION
# Classes, hierarchy, object properties and OWL restrictions
# ============================================================

import os, re, glob
import pandas as pd
from pathlib import Path
from rdflib import Graph, URIRef, BNode, RDF, RDFS, OWL, SKOS

RESULTS_DIR = "/content/reviewer_evidence/results"
ONTOLOGY_DIR = "/content/reviewer_evidence/ontologies"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Locate ontology files
# ------------------------------------------------------------

candidate_files = (
    glob.glob("/content/**/*.owl", recursive=True)
    + glob.glob("/content/**/*.ttl", recursive=True)
)

ontology_files = []

for f in candidate_files:
    name = os.path.basename(f).lower()
    if (
        "reviewer_evidence" in f.lower()
        and (
            "ontology" in f.lower()
            or "ms cdo" in name
            or "mscdo" in name
        )
    ):
        ontology_files.append(f)

ontology_files = sorted(set(ontology_files))

if not ontology_files:
    raise FileNotFoundError(
        "No ontology files found. Check /content/reviewer_evidence/"
    )

print("Ontology files found:", len(ontology_files))

# ------------------------------------------------------------
# 2. Parse ontology files
# ------------------------------------------------------------

graphs = {}
parse_log = []

for path in ontology_files:
    g = Graph()
    parsed = False
    errors = []

    formats = ["xml", "turtle", "nt"]

    for fmt in formats:
        try:
            g.parse(path, format=fmt)
            parsed = True
            break
        except Exception as e:
            errors.append(str(e)[:150])
            g = Graph()

    if parsed:
        graphs[path] = g
        parse_log.append({
            "file": os.path.basename(path),
            "path": path,
            "triples": len(g),
            "status": "Parsed"
        })
    else:
        parse_log.append({
            "file": os.path.basename(path),
            "path": path,
            "triples": 0,
            "status": "Failed: " + " | ".join(errors)
        })

parse_log_df = pd.DataFrame(parse_log)
parse_log_df.to_csv(
    f"{RESULTS_DIR}/focused_ontology_parse_log.csv",
    index=False
)

if not graphs:
    raise ValueError("No ontology files could be parsed.")

print("Successfully parsed:", len(graphs))

# ------------------------------------------------------------
# 3. Merge parsed graphs
# ------------------------------------------------------------

combined_graph = Graph()

for g in graphs.values():
    for triple in g:
        combined_graph.add(triple)

print("Combined triples:", len(combined_graph))

# ------------------------------------------------------------
# 4. Extract class labels
# ------------------------------------------------------------

def get_labels(graph, uri):
    labels = set()

    for predicate in [RDFS.label, SKOS.prefLabel, SKOS.altLabel]:
        for value in graph.objects(uri, predicate):
            labels.add(str(value))

    return sorted(labels)


classes = set(combined_graph.subjects(RDF.type, OWL.Class))
classes.update(combined_graph.subjects(RDF.type, RDFS.Class))

# Include subjects in subclass relationships
for child, parent in combined_graph.subject_objects(RDFS.subClassOf):
    if isinstance(child, URIRef):
        classes.add(child)
    if isinstance(parent, URIRef):
        classes.add(parent)

class_records = []

for uri in classes:
    labels = get_labels(combined_graph, uri)
    if not labels:
        continue

    class_records.append({
        "concept_uri": str(uri),
        "preferred_label": labels[0],
        "all_labels": " | ".join(labels)
    })

class_df = pd.DataFrame(class_records)

print("Labelled classes:", len(class_df))

# ------------------------------------------------------------
# 5. Identify clinically relevant seed concepts
# ------------------------------------------------------------

seed_keywords = {
    "medication": [
        "medication", "medicine", "drug", "pharmaceutical",
        "medicinal product", "prescription"
    ],
    "dose_strength": [
        "dose", "dosage", "strength", "concentration",
        "amount", "quantity", "rate"
    ],
    "administration": [
        "administration", "route", "frequency",
        "form", "infusion", "injection"
    ],
    "time": [
        "time", "date", "duration", "timing",
        "interval", "frequency"
    ],
    "patient": [
        "patient", "person", "recipient", "wrong patient"
    ],
    "incident": [
        "incident", "patient safety", "medication error",
        "medication incident", "adverse event"
    ],
    "contributing_factor": [
        "contributing factor", "hazard", "risk factor",
        "cause", "human factor"
    ],
    "detection": [
        "detection", "detecting", "recognition",
        "identification"
    ],
    "outcome": [
        "outcome", "harm", "injury", "consequence",
        "patient outcome"
    ],
    "mitigation": [
        "mitigation", "mitigating", "preventive action",
        "risk reduction", "ameliorating action"
    ]
}

def normalise_label(x):
    return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()

seed_records = []

for _, row in class_df.iterrows():
    label = normalise_label(row["all_labels"])
    matched_groups = []

    for group, keywords in seed_keywords.items():
        if any(
            re.search(
                r"(?<!\w)" + re.escape(normalise_label(k)) + r"(?!\w)",
                label
            )
            for k in keywords
        ):
            matched_groups.append(group)

    if matched_groups:
        seed_records.append({
            **row.to_dict(),
            "seed_groups": " | ".join(matched_groups)
        })

seed_df = pd.DataFrame(seed_records)

if seed_df.empty:
    raise ValueError(
        "No seed concepts matched. Inspect class_df and revise seed_keywords."
    )

seed_uris = set(seed_df["concept_uri"])

print("Seed concepts identified:", len(seed_uris))

# ------------------------------------------------------------
# 6. Build hierarchy: seeds + ancestors + descendants
# ------------------------------------------------------------

parents = {}
children = {}

for child, parent in combined_graph.subject_objects(RDFS.subClassOf):
    if isinstance(child, URIRef) and isinstance(parent, URIRef):
        parents.setdefault(str(child), set()).add(str(parent))
        children.setdefault(str(parent), set()).add(str(child))

selected_depth = {}

# Seed concepts at relative depth 0
for uri in seed_uris:
    selected_depth[uri] = 0

# Ancestors, up to 2 levels
for uri in seed_uris:
    current = {uri}

    for depth in range(1, 3):
        next_nodes = set()

        for node in current:
            next_nodes.update(parents.get(node, set()))

        for node in next_nodes:
            if node not in selected_depth:
                selected_depth[node] = -depth

        current = next_nodes

# Descendants, up to 2 levels
for uri in seed_uris:
    current = {uri}

    for depth in range(1, 3):
        next_nodes = set()

        for node in current:
            next_nodes.update(children.get(node, set()))

        for node in next_nodes:
            if node not in selected_depth:
                selected_depth[node] = depth

        current = next_nodes

# Keep only classes with labels
class_lookup = {
    row["concept_uri"]: row
    for row in class_records
}

selected_records = []

for uri, depth in selected_depth.items():
    if uri in class_lookup:
        selected_records.append({
            **class_lookup[uri],
            "relative_hierarchy_depth": depth,
            "selection_role": (
                "Seed" if depth == 0
                else "Ancestor" if depth < 0
                else "Descendant"
            )
        })

focused_classes = pd.DataFrame(selected_records).drop_duplicates(
    subset=["concept_uri"]
)

focused_uris = set(focused_classes["concept_uri"])

# ------------------------------------------------------------
# 7. Extract relevant object properties
# ------------------------------------------------------------

object_properties = set(
    combined_graph.subjects(RDF.type, OWL.ObjectProperty)
)

property_records = []

for prop in object_properties:

    domains = [
        str(x) for x in combined_graph.objects(prop, RDFS.domain)
    ]
    ranges = [
        str(x) for x in combined_graph.objects(prop, RDFS.range)
    ]

    relevant = (
        bool(set(domains) & focused_uris)
        or bool(set(ranges) & focused_uris)
    )

    if relevant:
        property_records.append({
            "property_uri": str(prop),
            "labels": " | ".join(get_labels(combined_graph, prop)),
            "domains": " | ".join(domains),
            "ranges": " | ".join(ranges)
        })

focused_properties = pd.DataFrame(property_records)

# ------------------------------------------------------------
# 8. Extract hierarchy relationships
# ------------------------------------------------------------

hierarchy_records = []

for child, parent in combined_graph.subject_objects(RDFS.subClassOf):
    if str(child) in focused_uris or str(parent) in focused_uris:
        hierarchy_records.append({
            "subject": str(child),
            "relationship": "rdfs:subClassOf",
            "object": str(parent)
        })

for predicate in [
    OWL.equivalentClass,
    OWL.disjointWith
]:
    for subject, obj in combined_graph.subject_objects(predicate):
        if str(subject) in focused_uris or str(obj) in focused_uris:
            hierarchy_records.append({
                "subject": str(subject),
                "relationship": str(predicate),
                "object": str(obj)
            })

focused_hierarchy = pd.DataFrame(hierarchy_records)

# ------------------------------------------------------------
# 9. Extract relevant OWL restrictions
# ------------------------------------------------------------

restriction_records = []

restriction_predicates = [
    OWL.someValuesFrom,
    OWL.allValuesFrom,
    OWL.hasValue,
    OWL.cardinality,
    OWL.minCardinality,
    OWL.maxCardinality,
    OWL.qualifiedCardinality,
    OWL.minQualifiedCardinality,
    OWL.maxQualifiedCardinality
]

for cls in focused_uris:

    for restriction in combined_graph.objects(
        URIRef(cls), RDFS.subClassOf
    ):

        if not isinstance(restriction, BNode):
            continue

        prop = next(
            combined_graph.objects(restriction, OWL.onProperty),
            None
        )

        if prop is None:
            continue

        for predicate in restriction_predicates:
            for value in combined_graph.objects(
                restriction, predicate
            ):
                restriction_records.append({
                    "class_uri": cls,
                    "property_uri": str(prop),
                    "restriction_type": str(predicate),
                    "restriction_value": str(value)
                })

focused_restrictions = pd.DataFrame(restriction_records)

# ------------------------------------------------------------
# 10. Save compact outputs
# ------------------------------------------------------------

focused_classes.to_csv(
    f"{RESULTS_DIR}/focused_ontology_classes.csv",
    index=False,
    encoding="utf-8-sig"
)

focused_properties.to_csv(
    f"{RESULTS_DIR}/focused_ontology_properties.csv",
    index=False,
    encoding="utf-8-sig"
)

focused_hierarchy.to_csv(
    f"{RESULTS_DIR}/focused_ontology_hierarchy.csv",
    index=False,
    encoding="utf-8-sig"
)

focused_restrictions.to_csv(
    f"{RESULTS_DIR}/focused_ontology_restrictions.csv",
    index=False,
    encoding="utf-8-sig"
)

seed_df.to_csv(
    f"{RESULTS_DIR}/ontology_seed_concepts.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFOCUSED ONTOLOGY EXTRACTION COMPLETED")
print("Seed concepts:", len(seed_df))
print("Selected classes:", len(focused_classes))
print("Selected properties:", len(focused_properties))
print("Hierarchy relationships:", len(focused_hierarchy))
print("OWL restrictions:", len(focused_restrictions))

display(focused_classes.head(30))
display(focused_properties.head(20))

Ontology files found: 1
Successfully parsed: 1
Combined triples: 488
Labelled classes: 6
Seed concepts identified: 4

FOCUSED ONTOLOGY EXTRACTION COMPLETED
Seed concepts: 4
Selected classes: 4
Selected properties: 14
Hierarchy relationships: 7
OWL restrictions: 0


,concept_uri,preferred_label,all_labels,relative_hierarchy_depth,selection_role
0,http://example.org/msdo#PatientContext,Patient Context,Patient Context,0,Seed
1,http://example.org/msdo#PatientSafetyPrevention,Patient Safety and Prevention,Patient Safety and Prevention,0,Seed
2,http://example.org/msdo#MedicationRelatedRisk,Medication-Related Risk,Medication-Related Risk,0,Seed
3,http://example.org/msdo#MedicationEntity,Medication Entity,Medication Entity,0,Seed


,property_uri,labels,domains,ranges
0,http://example.org/msdo#hasRiskProfile,,http://example.org/msdo#Medication | http://rx...,http://example.org/msdo#RiskProfile
1,http://example.org/msdo#associatedWith,,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDr...,http://example.org/msdo#MedicationRelatedRisk ...
2,http://example.org/msdo#hasSeverity,,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDr...,http://example.org/msdo#Severity
3,http://example.org/msdo#interactsWith,,http://example.org/msdo#Medication | http://rx...,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDr...
4,http://example.org/msdo#hasActiveIngredient,,http://example.org/msdo#Medication | http://rx...,http://example.org/msdo#ActiveIngredient
5,http://example.org/msdo#hasOrganFunction,,http://example.org/msdo#Patient | http://examp...,http://example.org/msdo#OrganFunction
6,http://example.org/msdo#hasGeneticRisk,,http://example.org/msdo#Patient | http://examp...,http://example.org/msdo#PharmacogenomicProfile
7,http://example.org/msdo#mitigatedBy,,http://example.org/msdo#MedicationRelatedRisk,http://example.org/msdo#MitigatingAction
8,http://example.org/msdo#hasClinicalCondition,,http://example.org/msdo#Patient | http://examp...,http://snomed.info/id/404684003 | http://examp...
9,http://example.org/msdo#causesAdverseEvent,,http://example.org/msdo#Medication | http://rx...,http://purl.bioontology.org/ontology/MEDDRA/10...


In [57]:
# ============================================================
# CELL 9 — FOCUSED ONTOLOGY-TO-DATASET MAPPING
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/reviewer_evidence/results"

category_keywords = {
    "Drug": ["drug", "medication", "medicine", "pharmaceutical"],
    "Dosage": ["dose", "dosage"],
    "Strength_amount": ["strength", "amount", "quantity"],
    "Strength_rate": ["rate"],
    "Strength_concentration": ["concentration"],
    "Frequency": ["frequency", "repetition"],
    "Duration": ["duration", "interval"],
    "Timing": ["timing", "time"],
    "Date": ["date"],
    "Route": ["route", "administration"],
    "Form_form": ["form", "dosage form"],
    "Form_mode": ["mode", "administration"],
    "Wrong_patient": ["patient", "person", "recipient"]
}

mapping_rows = []

for _, row in focused_classes.iterrows():

    label = str(row["all_labels"]).lower()

    suggestions = []

    for category, keywords in category_keywords.items():
        if any(k in label for k in keywords):
            suggestions.append(category)

    mapping_rows.append({
        "concept_uri": row["concept_uri"],
        "preferred_label": row["preferred_label"],
        "ontology_labels": row["all_labels"],
        "relative_hierarchy_depth": row["relative_hierarchy_depth"],
        "selection_role": row["selection_role"],
        "suggested_dataset_categories": " | ".join(suggestions),
        "expert_confirmed_category": "",
        "mapping_status": "Pending expert review",
        "notes": ""
    })

focused_mapping = pd.DataFrame(mapping_rows)

focused_mapping.to_csv(
    f"{RESULTS_DIR}/focused_ontology_mapping_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Focused mapping template created.")
print("Concepts requiring review:", len(focused_mapping))
display(focused_mapping.head(30))

Focused mapping template created.
Concepts requiring review: 4


,concept_uri,preferred_label,ontology_labels,relative_hierarchy_depth,selection_role,suggested_dataset_categories,expert_confirmed_category,mapping_status,notes
0,http://example.org/msdo#PatientContext,Patient Context,Patient Context,0,Seed,Wrong_patient,,Pending expert review,
1,http://example.org/msdo#PatientSafetyPrevention,Patient Safety and Prevention,Patient Safety and Prevention,0,Seed,Wrong_patient,,Pending expert review,
2,http://example.org/msdo#MedicationRelatedRisk,Medication-Related Risk,Medication-Related Risk,0,Seed,Drug,,Pending expert review,
3,http://example.org/msdo#MedicationEntity,Medication Entity,Medication Entity,0,Seed,Drug,,Pending expert review,


In [58]:
# ============================================================
# CELL 10 — FOCUSED RELATIONSHIP TABLE
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/reviewer_evidence/results"

relationship_rows = []

for _, row in focused_properties.iterrows():

    domains = [
        x.strip() for x in str(row["domains"]).split("|")
        if x.strip()
    ]

    ranges = [
        x.strip() for x in str(row["ranges"]).split("|")
        if x.strip()
    ]

    for domain in domains:
        for range_value in ranges:

            relationship_rows.append({
                "domain_uri": domain,
                "property_uri": row["property_uri"],
                "property_label": row["labels"],
                "range_uri": range_value
            })

relationship_table = pd.DataFrame(relationship_rows)

relationship_table.to_csv(
    f"{RESULTS_DIR}/focused_relationships.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Focused object-property relationships:", len(relationship_table))
display(relationship_table.head(30))

Focused object-property relationships: 70


,domain_uri,property_uri,property_label,range_uri
0,http://example.org/msdo#Medication,http://example.org/msdo#hasRiskProfile,,http://example.org/msdo#RiskProfile
1,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,http://example.org/msdo#hasRiskProfile,,http://example.org/msdo#RiskProfile
2,http://example.org/msdo#MedicationEntity,http://example.org/msdo#hasRiskProfile,,http://example.org/msdo#RiskProfile
3,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,http://example.org/msdo#associatedWith,,http://example.org/msdo#MedicationRelatedRisk
4,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,http://example.org/msdo#associatedWith,,http://purl.bioontology.org/ontology/MEDDRA/10...
5,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,http://example.org/msdo#associatedWith,,http://example.org/msdo#AdverseDrugEvent
6,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,http://example.org/msdo#associatedWith,,http://example.org/msdo#AdverseDrugReaction
7,http://example.org/msdo#MedicationEntity,http://example.org/msdo#associatedWith,,http://example.org/msdo#MedicationRelatedRisk
8,http://example.org/msdo#MedicationEntity,http://example.org/msdo#associatedWith,,http://purl.bioontology.org/ontology/MEDDRA/10...
9,http://example.org/msdo#MedicationEntity,http://example.org/msdo#associatedWith,,http://example.org/msdo#AdverseDrugEvent


In [59]:
# ============================================================
# CELL 11 — ONTOLOGY RULE AND AXIOM INVENTORY
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/reviewer_evidence/results"

rule_rows = []

for _, row in focused_restrictions.iterrows():
    rule_rows.append({
        "rule_type": "OWL restriction",
        "subject": row["class_uri"],
        "property": row["property_uri"],
        "constraint": row["restriction_type"],
        "value": row["restriction_value"],
        "execution_status": "Extracted; not reasoned over"
    })

for _, row in focused_hierarchy.iterrows():

    if "equivalentClass" in row["relationship"]:
        rule_type = "Class equivalence"
    elif "disjointWith" in row["relationship"]:
        rule_type = "Class disjointness"
    else:
        rule_type = "Class hierarchy"

    rule_rows.append({
        "rule_type": rule_type,
        "subject": row["subject"],
        "property": row["relationship"],
        "constraint": "",
        "value": row["object"],
        "execution_status": "Extracted; not reasoned over"
    })

rule_inventory = pd.DataFrame(rule_rows)

rule_inventory.to_csv(
    f"{RESULTS_DIR}/focused_ontology_rule_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Extracted ontology rules/axioms:", len(rule_inventory))
display(rule_inventory.head(30))

Extracted ontology rules/axioms: 7


,rule_type,subject,property,constraint,value,execution_status
0,Class hierarchy,http://example.org/msdo#AdverseDrugEvent,rdfs:subClassOf,,http://example.org/msdo#MedicationRelatedRisk,Extracted; not reasoned over
1,Class hierarchy,http://www.who.int/patientsafety/taxonomy/Medi...,rdfs:subClassOf,,http://example.org/msdo#MedicationRelatedRisk,Extracted; not reasoned over
2,Class hierarchy,http://purl.bioontology.org/ontology/MEDDRA/10...,rdfs:subClassOf,,http://example.org/msdo#MedicationRelatedRisk,Extracted; not reasoned over
3,Class hierarchy,http://example.org/msdo#MedicationError,rdfs:subClassOf,,http://example.org/msdo#MedicationRelatedRisk,Extracted; not reasoned over
4,Class hierarchy,http://rxnav.nlm.nih.gov/REST/rxcui/ClinicalDrug,rdfs:subClassOf,,http://example.org/msdo#MedicationEntity,Extracted; not reasoned over
5,Class hierarchy,http://example.org/msdo#Medication,rdfs:subClassOf,,http://example.org/msdo#MedicationEntity,Extracted; not reasoned over
6,Class hierarchy,http://example.org/msdo#Patient,rdfs:subClassOf,,http://example.org/msdo#PatientContext,Extracted; not reasoned over


In [60]:
# ============================================================
# CELL 12 — GROUPED REPORT-LEVEL SPLIT
# ============================================================

import pandas as pd
import os
from sklearn.model_selection import GroupShuffleSplit

RESULTS_DIR = "/content/reviewer_evidence/results"

report_table = main_df[["id", "report"]].drop_duplicates().copy()

report_table["id"] = report_table["id"].astype(str).str.strip()

report_table["normalised_report"] = (
    report_table["report"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Check IDs with multiple distinct report texts
id_text_counts = report_table.groupby("id")[
    "normalised_report"
].nunique()

problem_ids = id_text_counts[id_text_counts > 1]

if len(problem_ids):
    problem_table = report_table[
        report_table["id"].isin(problem_ids.index)
    ]

    problem_table.to_csv(
        f"{RESULTS_DIR}/ids_requiring_text_review.csv",
        index=False,
        encoding="utf-8-sig"
    )

    raise ValueError(
        f"{len(problem_ids)} IDs have multiple distinct report texts. "
        "Review ids_requiring_text_review.csv before splitting."
    )

report_table = report_table[
    report_table["normalised_report"] != ""
].drop_duplicates(subset=["id"])

split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(
        report_table,
        groups=report_table["normalised_report"]
    )
)

train = report_table.iloc[train_idx].copy()
temp = report_table.iloc[temp_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    split2.split(temp, groups=temp["normalised_report"])
)

validation = temp.iloc[val_idx].copy()
test = temp.iloc[test_idx].copy()

train["split"] = "train"
validation["split"] = "validation"
test["split"] = "test"

split_assignments = pd.concat(
    [train, validation, test],
    ignore_index=True
)

assert (
    split_assignments.groupby("normalised_report")["split"].nunique().max()
    == 1
)

split_assignments.to_csv(
    f"{RESULTS_DIR}/report_level_split_assignments.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Grouped split completed.")
print(split_assignments["split"].value_counts())
print("Unique reports:", len(split_assignments))
print("Identical texts across splits: none")

print(
    "This is report-level grouping, not patient-level or "
    "institution-level independent validation."
)

Grouped split completed.
split
train         46705
validation     5842
test           5842
Name: count, dtype: int64
Unique reports: 58389
Identical texts across splits: none
This is report-level grouping, not patient-level or institution-level independent validation.


In [61]:
# ============================================================
# CELL 13 — STRICT AND RELAXED ENTITY EVALUATION
# ============================================================

def entity_tuple(entity):
    return (
        str(entity["report_id"]),
        int(entity["start"]),
        int(entity["end"]),
        str(entity["entity_type"])
    )


def calculate_metrics(tp, fp, fn):

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "micro_f1": f1
    }


def strict_entity_evaluation(gold, predicted):

    gold_set = set(entity_tuple(x) for x in gold)
    pred_set = set(entity_tuple(x) for x in predicted)

    return calculate_metrics(
        len(gold_set & pred_set),
        len(pred_set - gold_set),
        len(gold_set - pred_set)
    )


def relaxed_entity_evaluation(gold, predicted):

    gold_items = [entity_tuple(x) for x in gold]
    pred_items = [entity_tuple(x) for x in predicted]

    candidates = []

    for pi, p in enumerate(pred_items):
        for gi, g in enumerate(gold_items):

            if p[0] != g[0] or p[3] != g[3]:
                continue

            overlap = max(
                0,
                min(p[2], g[2]) - max(p[1], g[1])
            )

            if overlap > 0:
                candidates.append((overlap, pi, gi))

    candidates.sort(reverse=True)

    matched_pred = set()
    matched_gold = set()

    for _, pi, gi in candidates:
        if pi not in matched_pred and gi not in matched_gold:
            matched_pred.add(pi)
            matched_gold.add(gi)

    return calculate_metrics(
        len(matched_gold),
        len(pred_items) - len(matched_pred),
        len(gold_items) - len(matched_gold)
    )


print("Strict and relaxed evaluation functions defined.")
print("No model predictions evaluated.")

Strict and relaxed evaluation functions defined.
No model predictions evaluated.


In [62]:
# ============================================================
# CELL 14 — GOLD EXPORT AND MODEL TEMPLATES
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/reviewer_evidence/results"

split_lookup = split_assignments[["id", "split"]].copy()

gold_export = main_df.copy()
gold_export["id"] = gold_export["id"].astype(str).str.strip()

gold_export = gold_export.merge(
    split_lookup,
    on="id",
    how="inner",
    validate="many_to_one"
)

gold_export.to_csv(
    f"{RESULTS_DIR}/gold_entity_annotations_with_splits.csv",
    index=False,
    encoding="utf-8-sig"
)

prediction_template = pd.DataFrame(columns=[
    "model", "report_id", "start", "end",
    "entity_type", "predicted_text", "seed", "split"
])

prediction_template.to_csv(
    f"{RESULTS_DIR}/model_predictions_template.csv",
    index=False,
    encoding="utf-8-sig"
)

results_template = pd.DataFrame(columns=[
    "model", "split", "seed", "entity_type",
    "TP", "FP", "FN", "precision", "recall",
    "f1", "evaluation_type"
])

results_template.to_csv(
    f"{RESULTS_DIR}/model_results_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Gold annotations exported:", len(gold_export))
print("Reports:", gold_export["id"].nunique())
print("Prediction and result templates created.")

Gold annotations exported: 478176
Reports: 58389
Prediction and result templates created.


In [63]:
# ============================================================
# CELL 15 — BOOTSTRAP AND PAIRED MODEL COMPARISON
# ============================================================

import numpy as np
from statsmodels.stats.contingency_tables import mcnemar


def report_level_bootstrap_f1(
    gold_by_report,
    pred_by_report,
    n_bootstrap=1000,
    random_state=42
):

    rng = np.random.default_rng(random_state)
    report_ids = list(gold_by_report.keys())

    if not report_ids:
        raise ValueError("No reports supplied.")

    scores = []

    for b in range(n_bootstrap):

        sampled = rng.choice(
            report_ids,
            size=len(report_ids),
            replace=True
        )

        gold_sample = []
        pred_sample = []

        for i, rid in enumerate(sampled):

            new_id = f"bootstrap_{i}"

            for item in gold_by_report[rid]:
                gold_sample.append({
                    **item,
                    "report_id": new_id
                })

            for item in pred_by_report.get(rid, []):
                pred_sample.append({
                    **item,
                    "report_id": new_id
                })

        result = strict_entity_evaluation(
            gold_sample,
            pred_sample
        )

        scores.append(result["micro_f1"])

    return {
        "mean_f1": float(np.mean(scores)),
        "ci_lower": float(np.percentile(scores, 2.5)),
        "ci_upper": float(np.percentile(scores, 97.5)),
        "bootstrap_iterations": n_bootstrap
    }


def paired_report_error_test(
    gold_by_report,
    predictions_a,
    predictions_b
):

    report_ids = list(gold_by_report.keys())

    both_correct = 0
    a_only = 0
    b_only = 0
    both_wrong = 0

    for rid in report_ids:

        gold_set = {
            entity_tuple(x)[1:]
            for x in gold_by_report[rid]
        }

        a_set = {
            entity_tuple(x)[1:]
            for x in predictions_a.get(rid, [])
        }

        b_set = {
            entity_tuple(x)[1:]
            for x in predictions_b.get(rid, [])
        }

        a_correct = a_set == gold_set
        b_correct = b_set == gold_set

        if a_correct and b_correct:
            both_correct += 1
        elif a_correct:
            a_only += 1
        elif b_correct:
            b_only += 1
        else:
            both_wrong += 1

    result = mcnemar(
        [[both_correct, a_only], [b_only, both_wrong]],
        exact=True
    )

    return {
        "both_correct": both_correct,
        "model_a_only_correct": a_only,
        "model_b_only_correct": b_only,
        "both_wrong": both_wrong,
        "p_value": float(result.pvalue)
    }


print("Statistical functions defined.")
print("No significance tests have been run.")

Statistical functions defined.
No significance tests have been run.


In [64]:
# ============================================================
# CELL 16 — VALIDATION AUDIT AND HUMAN EVALUATION TEMPLATES
# ============================================================

import pandas as pd
import os
import glob

RESULTS_DIR = "/content/reviewer_evidence/results"
VALIDATION_DIR = "/content/medication_revision/zip_datasets"

validation_files = glob.glob(
    os.path.join(VALIDATION_DIR, "*.xlsx")
)

main_ids = set(main_df["id"].dropna().astype(str).str.strip())

main_texts = set(
    main_df["report"].dropna().astype(str)
    .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

audit_rows = []

for path in validation_files:

    xls = pd.ExcelFile(path)

    for sheet in xls.sheet_names:

        temp = pd.read_excel(path, sheet_name=sheet)

        cols = {
            str(c).lower().strip(): c
            for c in temp.columns
        }

        id_col = next(
            (cols[x] for x in ["id", "report_id", "no"] if x in cols),
            None
        )

        text_col = next(
            (
                cols[x] for x in
                ["report", "freetext", "free_text", "translated_report"]
                if x in cols
            ),
            None
        )

        ids = (
            set(temp[id_col].dropna().astype(str).str.strip())
            if id_col else set()
        )

        texts = (
            set(
                temp[text_col].dropna().astype(str)
                .str.lower().str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )
            if text_col else set()
        )

        audit_rows.append({
            "file": os.path.basename(path),
            "sheet": sheet,
            "rows": len(temp),
            "id_column": id_col,
            "text_column": text_col,
            "unique_ids": len(ids),
            "overlap_main_ids": len(ids & main_ids),
            "unique_texts": len(texts),
            "overlap_main_texts": len(texts & main_texts)
        })

validation_audit = pd.DataFrame(audit_rows)

validation_audit.to_csv(
    f"{RESULTS_DIR}/validation_workbook_overlap_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

iaa_template = pd.DataFrame(columns=[
    "report_id",
    "annotator_1_start", "annotator_1_end", "annotator_1_type",
    "annotator_2_start", "annotator_2_end", "annotator_2_type",
    "adjudicated_start", "adjudicated_end", "adjudicated_type"
])

iaa_template.to_csv(
    f"{RESULTS_DIR}/interannotator_agreement_template.csv",
    index=False,
    encoding="utf-8-sig"
)

expert_template = pd.DataFrame(columns=[
    "report_id",
    "model_prediction",
    "expert_correctness",
    "clinical_relevance",
    "severity",
    "missed_safety_information",
    "expert_comments"
])

expert_template.to_csv(
    f"{RESULTS_DIR}/clinical_expert_evaluation_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Validation workbook audit completed.")
display(validation_audit)
print("IAA and clinical expert templates created.")
print("Workbook overlap does not establish independent validation.")

Validation workbook audit completed.


,file,sheet,rows,id_column,text_column,unique_ids,overlap_main_ids,unique_texts,overlap_main_texts
0,ExternalValid_JQ2021_20.xlsx,freetext,20,id,report,20,0,20,0
1,ExternalValid_JQ2021_20.xlsx,manualreviewed,111,id,report,20,0,20,0
2,ExternalValid_JQ2021_20.xlsx,machineannotated,104,id,report,19,0,19,0
3,ExternalValid_JQ2021_20.xlsx,Incidenttype_evl,23,id,None,20,0,0,0
4,Error_analysis_10.xlsx,freetext,10,id,report,10,0,10,0
5,Error_analysis_10.xlsx,manualreviewed,13,id,report,10,0,10,0
6,Error_analysis_10.xlsx,machineannotated,8,id,None,5,0,0,0
7,InternalValid_JQ2010-20_40.xlsx,freetext,40,id,report,40,39,40,39
8,InternalValid_JQ2010-20_40.xlsx,manuallyreviewed,263,id,report,40,39,40,39
9,InternalValid_JQ2010-20_40.xlsx,machineannotated,356,id,report,47,47,47,46


IAA and clinical expert templates created.
Workbook overlap does not establish independent validation.


In [65]:
# ============================================================
# CELL 17 — REVIEWER EVIDENCE INVENTORY
# ============================================================

import pandas as pd
import os

RESULTS_DIR = "/content/reviewer_evidence/results"

evidence = [
    ("Main dataset", "478,176 annotation rows loaded", "Completed"),
    ("Entity distribution", "13 entity categories summarised", "Completed"),
    ("Ontology parsing", "Available ontology files parsed", "Completed computationally"),
    ("Focused ontology extraction", "Selected seed concepts and hierarchy", "Completed computationally if Cell 8 succeeds"),
    ("Object properties", "Selected properties with domains and ranges", "Completed computationally if Cell 8 succeeds"),
    ("Ontology restrictions", "Selected OWL restrictions extracted", "Completed computationally if Cell 8 succeeds"),
    ("Ontology provenance", "Versions and official source provenance", "Requires verification"),
    ("Ontology mapping", "Dataset category mapping template", "Pending expert review"),
    ("Ontology reasoning", "No reasoner or rule execution performed", "Not run"),
    ("Ontology ablation", "Conditions documented", "Not run"),
    ("RAG", "No retrieval model evaluated", "Not run"),
    ("Duplicate audit", "ID and report text audit", "Completed if Cell 11/12 succeeds"),
    ("Data split", "Grouped report-level split", "Completed if Cell 12 succeeds"),
    ("Entity evaluation", "Evaluation functions and templates", "Pending predictions"),
    ("Statistical testing", "Bootstrap and paired test functions", "Pending predictions"),
    ("Independent annotation", "IAA template prepared", "Pending annotators"),
    ("Clinical evaluation", "Expert evaluation template prepared", "Pending experts"),
    ("External validation", "Workbook structure and overlap audit", "Audit only")
]

evidence_df = pd.DataFrame(
    evidence,
    columns=["area", "evidence", "status"]
)

evidence_df.to_csv(
    f"{RESULTS_DIR}/reviewer_evidence_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

print("REVIEWER EVIDENCE INVENTORY")
display(evidence_df)

print("\nOutput files:")
for filename in sorted(os.listdir(RESULTS_DIR)):
    print("-", filename)

REVIEWER EVIDENCE INVENTORY


,area,evidence,status
0,Main dataset,"478,176 annotation rows loaded",Completed
1,Entity distribution,13 entity categories summarised,Completed
2,Ontology parsing,Available ontology files parsed,Completed computationally
3,Focused ontology extraction,Selected seed concepts and hierarchy,Completed computationally if Cell 8 succeeds
4,Object properties,Selected properties with domains and ranges,Completed computationally if Cell 8 succeeds
5,Ontology restrictions,Selected OWL restrictions extracted,Completed computationally if Cell 8 succeeds
6,Ontology provenance,Versions and official source provenance,Requires verification
7,Ontology mapping,Dataset category mapping template,Pending expert review
8,Ontology reasoning,No reasoner or rule execution performed,Not run
9,Ontology ablation,Conditions documented,Not run



Output files:
- MSCDO_ICPS_ontology_terms.csv
- MSCDO_ICPS_relationships.csv
- clinical_expert_evaluation_template.csv
- combined_ontology_terms.csv
- dataset_entity_inventory.csv
- focused_ontology_classes.csv
- focused_ontology_hierarchy.csv
- focused_ontology_mapping_template.csv
- focused_ontology_parse_log.csv
- focused_ontology_properties.csv
- focused_ontology_restrictions.csv
- focused_ontology_rule_inventory.csv
- focused_relationships.csv
- gold_entity_annotations_with_splits.csv
- interannotator_agreement_template.csv
- model_predictions_template.csv
- model_results_template.csv
- ontology_download_status.csv
- ontology_mapping_template.csv
- ontology_parsing_log.csv
- ontology_profile.csv
- ontology_seed_concepts.csv
- report_level_split_assignments.csv
- reviewer_evidence_inventory.csv
- validation_workbook_overlap_audit.csv


In [66]:

# ============================================================
# CELL 18 — CONSOLIDATED PDF EVIDENCE REPORT
# Medication Safety NLP / Ontology Reviewer Response
# ============================================================

import os
import pandas as pd
import numpy as np
from datetime import datetime
from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, PageBreak, LongTable, KeepTogether
)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

RESULTS_DIR = "/content/reviewer_evidence/results"
PDF_PATH = os.path.join(
    RESULTS_DIR,
    "Medication_Safety_Ontology_Reviewer_Evidence_Report.pdf"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Load actual output files
# ------------------------------------------------------------

def load_csv(filename):
    path = os.path.join(RESULTS_DIR, filename)

    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"Could not read {filename}: {e}")

    print(f"Not available: {filename}")
    return pd.DataFrame()


entity_inventory = load_csv("dataset_entity_inventory.csv")
focused_classes = load_csv("focused_ontology_classes.csv")
seed_concepts = load_csv("ontology_seed_concepts.csv")
focused_properties = load_csv("focused_ontology_properties.csv")
focused_hierarchy = load_csv("focused_ontology_hierarchy.csv")
focused_restrictions = load_csv("focused_ontology_restrictions.csv")
focused_mapping = load_csv("focused_ontology_mapping_template.csv")
relationships = load_csv("focused_relationships.csv")
rule_inventory = load_csv("focused_ontology_rule_inventory.csv")
split_assignments = load_csv("report_level_split_assignments.csv")
validation_audit = load_csv("validation_workbook_overlap_audit.csv")
evidence_inventory = load_csv("reviewer_evidence_inventory.csv")
parse_log = load_csv("focused_ontology_parse_log.csv")

# ------------------------------------------------------------
# 3. Create consolidated reviewer modification matrix
# ------------------------------------------------------------

reviewer_matrix = pd.DataFrame([
    {
        "Reviewer concern": "Clarify research aim, objectives, motivation, scope and practical gap",
        "Manuscript section": "Introduction",
        "Required modification": "Rewrite the aim and objectives; explain the clinical and methodological gap and define the scope of medication-safety concept extraction.",
        "Evidence currently available": "Japanese dataset structure and 13 entity categories are available.",
        "Remaining work": "Revise the introduction and ensure the stated research scope matches the actual experiments.",
        "Status": "Manuscript revision required"
    },
    {
        "Reviewer concern": "Strengthen and critically compare related work",
        "Manuscript section": "Related work",
        "Required modification": "Compare the cited heart-disease DNN, cancer platform, ICU monitoring AI and hybrid autoencoder-GAT approaches in terms of task, data, architecture, explainability and limitations.",
        "Evidence currently available": "No comparative literature analysis generated by Cells 7–17.",
        "Remaining work": "Complete a source-verified literature comparison and explain the distinction from this study.",
        "Status": "Pending"
    },
    {
        "Reviewer concern": "Explain architecture, hyperparameters, training and rationale",
        "Manuscript section": "Methods",
        "Required modification": "Describe the actual implemented pipeline, model architecture, training settings, preprocessing, data splits and hardware.",
        "Evidence currently available": "Dataset and report-level split preparation; ontology subset extraction.",
        "Remaining work": "Document actual model implementation and training logs. Do not describe planned components as implemented.",
        "Status": "Pending model evidence"
    },
    {
        "Reviewer concern": "Make the novelty explicit",
        "Manuscript section": "Introduction / Discussion",
        "Required modification": "Add a dedicated comparison of the proposed approach with existing ontology-guided, LLM-based and clinical NLP approaches.",
        "Evidence currently available": "Focused ontology concepts, properties and restrictions can inform the proposed knowledge representation.",
        "Remaining work": "Establish novelty against verified literature and actual implemented methods.",
        "Status": "Pending"
    },
    {
        "Reviewer concern": "Resolve overall F1 versus precision, recall, micro-F1 and macro-F1 inconsistencies",
        "Manuscript section": "Results",
        "Required modification": "Recalculate all metrics from the same prediction files and define averaging and matching conventions.",
        "Evidence currently available": "Evaluation functions prepared; no new model predictions evaluated in Cells 7–17.",
        "Remaining work": "Recompute metrics from reproducible predictions and reconcile the reported F1=0.89, P=0.81, R=0.79, micro-F1=0.80 and macro-F1=0.73.",
        "Status": "Unresolved"
    },
    {
        "Reviewer concern": "Resolve category count discrepancy (7,886 versus 8,742)",
        "Manuscript section": "Dataset / Results",
        "Required modification": "Reconcile annotation counts, filtering, duplicates and category definitions across tables.",
        "Evidence currently available": "Current Japanese main dataset has 478,176 annotation rows and 13 entity categories.",
        "Remaining work": "Trace the two earlier counts to their source files and processing steps.",
        "Status": "Unresolved"
    },
    {
        "Reviewer concern": "Establish annotation reliability and independent gold standard",
        "Manuscript section": "Dataset / Annotation protocol",
        "Required modification": "Describe annotators, independent annotation, adjudication and inter-annotator agreement.",
        "Evidence currently available": "IAA template created; span-alignment audit was previously conducted.",
        "Remaining work": "Obtain independent annotations and calculate appropriate agreement statistics. Span alignment is not IAA.",
        "Status": "Pending human annotation"
    },
    {
        "Reviewer concern": "Address data leakage and independence of evaluation splits",
        "Manuscript section": "Methods / Validation",
        "Required modification": "Prevent duplicate reports crossing splits and clarify the level of independence.",
        "Evidence currently available": "Grouped report-level split and duplicate-text checks.",
        "Remaining work": "Verify split output; patient-level and institution-level independence are not established by report grouping.",
        "Status": "Partially addressed"
    },
    {
        "Reviewer concern": "Reconcile entity categories and label mapping across tables",
        "Manuscript section": "Dataset / Results",
        "Required modification": "Provide a consistent label inventory and explain mappings between source annotation labels and model output labels.",
        "Evidence currently available": "13-category inventory and focused ontology mapping template.",
        "Remaining work": "Review and approve mappings; reconcile all manuscript tables.",
        "Status": "Partially addressed"
    },
    {
        "Reviewer concern": "Define strict, relaxed and concept-level matching",
        "Manuscript section": "Evaluation",
        "Required modification": "Define exact-span, overlap-based and concept-level matching criteria and apply them consistently.",
        "Evidence currently available": "Strict and relaxed evaluation functions prepared.",
        "Remaining work": "Verify offset conventions, implement concept-level evaluation if applicable, and evaluate actual predictions.",
        "Status": "Partially addressed"
    },
    {
        "Reviewer concern": "Add suitable language-model and rule-based baselines",
        "Manuscript section": "Experiments",
        "Required modification": "Compare against appropriate Japanese-language pretrained models and rule/ontology-based baselines.",
        "Evidence currently available": "No baseline predictions generated by Cells 7–17.",
        "Remaining work": "Select language-appropriate models, implement baselines and evaluate on identical splits.",
        "Status": "Not run"
    },
    {
        "Reviewer concern": "Isolate the contribution of terminology, ontology constraints and LoRA",
        "Manuscript section": "Ablation study",
        "Required modification": "Run controlled ablations with clearly defined conditions.",
        "Evidence currently available": "Focused ontology subset and ablation planning can inform experiment design.",
        "Remaining work": "Implement and run ablations; report seeds, uncertainty and effect sizes.",
        "Status": "Not run"
    },
    {
        "Reviewer concern": "Verify weak-supervision quality and label proportions",
        "Manuscript section": "Methods / Results",
        "Required modification": "Report weak-label sources, proportions, noise checks and verification against gold annotations.",
        "Evidence currently available": "Main dataset contains errorlabel values; their meanings require a data dictionary.",
        "Remaining work": "Verify label definitions and measure weak-label quality against reviewed annotations.",
        "Status": "Pending"
    },
    {
        "Reviewer concern": "Explain LoRA contribution and statistical significance",
        "Manuscript section": "Results / Discussion",
        "Required modification": "Report LoRA settings, multiple seeds, efficiency and statistical comparisons; acknowledge nonsignificant effects if confirmed.",
        "Evidence currently available": "No new LoRA experiment or statistical comparison in Cells 7–17.",
        "Remaining work": "Run controlled comparisons and report actual results without assuming improvement.",
        "Status": "Not run"
    },
    {
        "Reviewer concern": "Clarify RAG design and its contribution",
        "Manuscript section": "Methods / Ablation",
        "Required modification": "Describe retrieval corpus, retrieval strategy, ranking and evaluation; compare with no-RAG conditions if RAG is claimed.",
        "Evidence currently available": "Ontology retrieval corpus preparation was previously performed; focused ontology subset now available.",
        "Remaining work": "Implement and evaluate RAG, or remove unsupported RAG claims.",
        "Status": "Preparation only"
    },
    {
        "Reviewer concern": "Provide confidence intervals and appropriate paired tests",
        "Manuscript section": "Statistical analysis",
        "Required modification": "Report uncertainty and use statistical tests suited to the paired evaluation design.",
        "Evidence currently available": "Bootstrap and paired report-level test functions prepared.",
        "Remaining work": "Run tests on actual predictions; justify the statistical unit and primary endpoint.",
        "Status": "Pending predictions"
    },
    {
        "Reviewer concern": "Assess clinical usefulness and expert validity",
        "Manuscript section": "Results / Discussion",
        "Required modification": "Evaluate clinical relevance, missed safety information and expert-rated correctness.",
        "Evidence currently available": "Clinical expert evaluation template prepared.",
        "Remaining work": "Conduct blinded expert evaluation and report protocol and findings.",
        "Status": "Pending experts"
    },
    {
        "Reviewer concern": "Address rare and minority categories",
        "Manuscript section": "Results / Discussion",
        "Required modification": "Report per-class performance and discuss class imbalance, especially very rare categories.",
        "Evidence currently available": "Current inventory identifies 14 Wrong_patient annotations.",
        "Remaining work": "Report per-class support and metrics; assess whether rare-category claims are statistically meaningful.",
        "Status": "Partially addressed"
    },
    {
        "Reviewer concern": "Demonstrate cross-dataset generalisability",
        "Manuscript section": "Validation / Discussion",
        "Required modification": "Clarify the external validation design and distinguish independent data from overlapping records.",
        "Evidence currently available": "Validation workbook structure and overlap audit.",
        "Remaining work": "Verify source independence and perform external evaluation using compatible annotations.",
        "Status": "Audit only"
    },
    {
        "Reviewer concern": "Provide complete reproducibility details",
        "Manuscript section": "Methods / Supplementary material",
        "Required modification": "Report preprocessing, software versions, hardware, model settings, seeds, data splits and evaluation code.",
        "Evidence currently available": "Dataset, ontology and split artifacts prepared.",
        "Remaining work": "Collect actual training configurations, logs and software environment.",
        "Status": "Pending"
    },
    {
        "Reviewer concern": "Remove placeholders and incomplete project questionnaire material",
        "Manuscript section": "Entire manuscript",
        "Required modification": "Remove unresolved placeholders, questionnaire remnants and unsupported statements.",
        "Evidence currently available": "Not assessed by the computational cells.",
        "Remaining work": "Perform a full manuscript editorial audit.",
        "Status": "Manuscript revision required"
    },
    {
        "Reviewer concern": "Clarify the change from MIMIC to Japanese incident reports",
        "Manuscript section": "Methods / Limitations",
        "Required modification": "Clearly state the actual dataset used, its language, source, annotation structure and implications for generalisability.",
        "Evidence currently available": "Japanese dataset with 478,176 entity annotations and 58,389 report IDs.",
        "Remaining work": "Document dataset provenance, permissions, translation process and the methodological consequences of the dataset change.",
        "Status": "Manuscript revision required"
    },
    {
        "Reviewer concern": "Verify ontology provenance, hierarchy and selected axioms",
        "Manuscript section": "Ontology / Methods",
        "Required modification": "Report source versions, selection criteria, hierarchy depth, object properties and relevant logical constraints.",
        "Evidence currently available": "Focused ontology extraction, hierarchy, properties and restrictions.",
        "Remaining work": "Verify official ontology provenance and manually review clinical relevance and mapping.",
        "Status": "Partially addressed"
    }
])

reviewer_matrix_path = os.path.join(
    RESULTS_DIR,
    "consolidated_reviewer_modification_matrix.csv"
)

reviewer_matrix.to_csv(
    reviewer_matrix_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. ReportLab styles and document
# ------------------------------------------------------------

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="ReportTitle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=19,
    leading=23,
    alignment=TA_CENTER,
    spaceAfter=10
))

styles.add(ParagraphStyle(
    name="SectionHeading",
    parent=styles["Heading1"],
    fontName="Helvetica-Bold",
    fontSize=13,
    leading=16,
    spaceBefore=10,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="SubHeading",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=10,
    leading=13,
    spaceBefore=7,
    spaceAfter=5
))

styles.add(ParagraphStyle(
    name="ReportBody",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=8,
    leading=11,
    spaceAfter=5
))

styles.add(ParagraphStyle(
    name="SmallCell",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=6.5,
    leading=8,
    wordWrap="CJK"
))

styles.add(ParagraphStyle(
    name="SmallHeader",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=6.5,
    leading=8,
    textColor=colors.white
))

doc = SimpleDocTemplate(
    PDF_PATH,
    pagesize=landscape(A4),
    rightMargin=13*mm,
    leftMargin=13*mm,
    topMargin=15*mm,
    bottomMargin=15*mm,
    title="Medication Safety Ontology and Reviewer Evidence Report",
    author="Research evidence report"
)

story = []

def para(value, style="SmallCell"):
    if pd.isna(value):
        value = ""
    value = escape(str(value))
    return Paragraph(value.replace("\n", "<br/>"), styles[style])


def add_table(title, df, columns=None, max_rows=None):
    story.append(Paragraph(escape(title), styles["SectionHeading"]))

    if df is None or df.empty:
        story.append(Paragraph(
            "No records available in the corresponding output file.",
            styles["ReportBody"]
        ))
        return

    view = df.copy()

    if columns:
        available = [c for c in columns if c in view.columns]
        view = view[available]

    total_rows = len(view)

    if max_rows is not None and total_rows > max_rows:
        view = view.head(max_rows)

    data = [
        [para(c, "SmallHeader") for c in view.columns]
    ]

    for _, row in view.iterrows():
        data.append([
            para(row[c]) for c in view.columns
        ])

    available_width = landscape(A4)[0] - 26*mm
    col_width = available_width / max(1, len(view.columns))
    widths = [col_width] * len(view.columns)

    table = LongTable(
        data,
        colWidths=widths,
        repeatRows=1,
        hAlign="LEFT"
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#34495E")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#B8C2CC")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
        ("TOPPADDING", (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [
            colors.white, colors.HexColor("#F2F5F7")
        ])
    ]))

    story.append(table)
    story.append(Spacer(1, 7))

    if max_rows is not None and total_rows > max_rows:
        story.append(Paragraph(
            f"Showing {max_rows} of {total_rows} rows. "
            "The complete table is retained in its corresponding CSV file.",
            styles["ReportBody"]
        ))


def footer(canvas, document):
    canvas.saveState()
    canvas.setFont("Helvetica", 7)
    canvas.setFillColor(colors.HexColor("#555555"))
    canvas.drawString(
        13*mm, 8*mm,
        "Medication Safety NLP — Reviewer Evidence Report"
    )
    canvas.drawRightString(
        landscape(A4)[0] - 13*mm,
        8*mm,
        f"Page {document.page}"
    )
    canvas.restoreState()


# ------------------------------------------------------------
# 5. Cover and scope
# ------------------------------------------------------------

story.append(Spacer(1, 22*mm))

story.append(Paragraph(
    "Medication Safety NLP and Focused Ontology Analysis",
    styles["ReportTitle"]
))

story.append(Paragraph(
    "Consolidated Computational Evidence and Manuscript "
    "Reviewer Modification Report",
    styles["Heading2"]
))

story.append(Spacer(1, 8*mm))

story.append(Paragraph(
    f"Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    styles["ReportBody"]
))

story.append(Paragraph(
    "Purpose: consolidate the outputs of Colab Cells 7–17 and "
    "organise the reviewer-requested manuscript modifications. "
    "This report distinguishes completed computational audits, "
    "preparatory artifacts, and experiments or human evaluations "
    "that have not yet been conducted.",
    styles["ReportBody"]
))

story.append(Paragraph(
    "<b>Important methodological qualification:</b> successful "
    "ontology extraction, data auditing, split preparation and "
    "evaluation-function definition do not constitute model "
    "training, baseline evaluation, ablation experiments, "
    "inter-annotator agreement or clinical validation.",
    styles["ReportBody"]
))

story.append(PageBreak())

# ------------------------------------------------------------
# 6. Dataset summary
# ------------------------------------------------------------

story.append(Paragraph("1. Main dataset and annotation summary",
                       styles["SectionHeading"]))

if "main_df" in globals():
    dataset_summary = pd.DataFrame([
        ["Dataset file", "/content/med_error_data/AnnotatedDataset.xlsx"],
        ["Annotation rows", len(main_df)],
        ["Unique report IDs", main_df["id"].nunique()],
        ["Entity categories", main_df["entitytype"].nunique()],
        ["Language/source context", "Japanese medication-incident reports"]
    ], columns=["Measure", "Value"])

    add_table("Main dataset summary", dataset_summary)

add_table(
    "Entity category distribution",
    entity_inventory,
    max_rows=30
)

# ------------------------------------------------------------
# 7. Ontology evidence
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("2. Focused ontology extraction",
                       styles["SectionHeading"]))

ontology_summary = pd.DataFrame([
    ["Parsed ontology files", len(parse_log)],
    ["Seed concepts", len(seed_concepts)],
    ["Selected classes", len(focused_classes)],
    ["Selected object properties", len(focused_properties)],
    ["Hierarchy relationships", len(focused_hierarchy)],
    ["OWL restrictions", len(focused_restrictions)],
    ["Explicit relationship rows", len(relationships)]
], columns=["Measure", "Count"])

add_table("Ontology extraction summary", ontology_summary)
add_table("Ontology parsing log", parse_log, max_rows=50)

add_table(
    "Seed concepts",
    seed_concepts,
    columns=[
        "preferred_label", "all_labels",
        "concept_uri", "seed_groups"
    ],
    max_rows=100
)

add_table(
    "Selected ontology classes and relative hierarchy depth",
    focused_classes,
    columns=[
        "preferred_label", "all_labels",
        "relative_hierarchy_depth",
        "selection_role", "concept_uri"
    ],
    max_rows=150
)

add_table(
    "Selected object properties",
    focused_properties,
    columns=[
        "labels", "property_uri", "domains", "ranges"
    ],
    max_rows=100
)

add_table(
    "Focused hierarchy relationships",
    focused_hierarchy,
    max_rows=150
)

add_table(
    "Explicit domain-property-range relationships",
    relationships,
    max_rows=150
)

add_table(
    "Extracted OWL restrictions",
    focused_restrictions,
    max_rows=100
)

add_table(
    "Ontology rule and axiom inventory",
    rule_inventory,
    max_rows=150
)

# ------------------------------------------------------------
# 8. Mapping
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("3. Ontology-to-dataset mapping",
                       styles["SectionHeading"]))

add_table(
    "Focused ontology mapping template",
    focused_mapping,
    columns=[
        "preferred_label",
        "relative_hierarchy_depth",
        "suggested_dataset_categories",
        "expert_confirmed_category",
        "mapping_status",
        "notes"
    ],
    max_rows=150
)

story.append(Paragraph(
    "Mapping status must remain pending until the proposed "
    "ontology-to-annotation correspondences have been reviewed. "
    "Keyword-based suggestions are not expert validation.",
    styles["ReportBody"]
))

# ------------------------------------------------------------
# 9. Split and validation
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("4. Data split and validation audit",
                       styles["SectionHeading"]))

if not split_assignments.empty and "split" in split_assignments.columns:
    split_summary = (
        split_assignments["split"]
        .value_counts()
        .rename_axis("split")
        .reset_index(name="report_count")
    )
    split_summary["proportion"] = (
        split_summary["report_count"] /
        split_summary["report_count"].sum()
    ).round(4)

    add_table("Report-level split distribution", split_summary)

add_table(
    "Validation workbook overlap audit",
    validation_audit,
    max_rows=100
)

story.append(Paragraph(
    "The split is grouped by normalised report text. It does not "
    "establish patient-level or institution-level independence. "
    "Workbook overlap checks do not prove that a dataset is "
    "independent or externally representative.",
    styles["ReportBody"]
))

# ------------------------------------------------------------
# 10. Evidence inventory
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("5. Reviewer evidence inventory",
                       styles["SectionHeading"]))

add_table(
    "Computational evidence and outstanding work",
    evidence_inventory,
    max_rows=100
)

# ------------------------------------------------------------
# 11. Reviewer modification matrix
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph(
    "6. Consolidated manuscript reviewer modification matrix",
    styles["SectionHeading"]
))

story.append(Paragraph(
    "This matrix consolidates the reviewer concerns previously "
    "shared during manuscript revision discussions. It is a "
    "working response plan, not a claim that all requested "
    "revisions or experiments have been completed. The original "
    "reviewer reports should be checked before final submission.",
    styles["ReportBody"]
))

add_table(
    "Reviewer requests, manuscript changes and remaining evidence",
    reviewer_matrix,
    max_rows=100
)

# ------------------------------------------------------------
# 12. Output file inventory
# ------------------------------------------------------------

story.append(PageBreak())
story.append(Paragraph("7. Supporting output files",
                       styles["SectionHeading"]))

output_files = []

for filename in sorted(os.listdir(RESULTS_DIR)):
    if filename.lower().endswith((".csv", ".jsonl", ".txt")):
        path = os.path.join(RESULTS_DIR, filename)
        output_files.append({
            "filename": filename,
            "size_kb": round(os.path.getsize(path) / 1024, 2)
        })

output_inventory = pd.DataFrame(output_files)

add_table("Supporting files available in results directory",
          output_inventory)

story.append(Paragraph(
    "The corresponding CSV files preserve the machine-readable "
    "tables. Where the PDF limits the number of displayed rows, "
    "the complete records remain in those CSV files.",
    styles["ReportBody"]
))

# ------------------------------------------------------------
# 13. Build PDF
# ------------------------------------------------------------

doc.build(
    story,
    onFirstPage=footer,
    onLaterPages=footer
)

print("\nPDF REPORT SUCCESSFULLY CREATED")
print("Path:", PDF_PATH)
print("Size (KB):", round(os.path.getsize(PDF_PATH) / 1024, 2))
print("\nReviewer modification matrix also saved:")
print(reviewer_matrix_path)

# Verify file
assert os.path.exists(PDF_PATH)
assert os.path.getsize(PDF_PATH) > 1000

print("\nReady to download and share.")

Could not read focused_ontology_restrictions.csv: No columns to parse from file
Not available: focused_ontology_restrictions.csv

PDF REPORT SUCCESSFULLY CREATED
Path: /content/reviewer_evidence/results/Medication_Safety_Ontology_Reviewer_Evidence_Report.pdf
Size (KB): 31.83

Reviewer modification matrix also saved:
/content/reviewer_evidence/results/consolidated_reviewer_modification_matrix.csv

Ready to download and share.


In [67]:
from google.colab import files

files.download(
    "/content/reviewer_evidence/results/Medication_Safety_Ontology_Reviewer_Evidence_Report.pdf"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:

# DIAGNOSTIC CELL — Verify MSCDO and ICPS ontology files

from pathlib import Path
from rdflib import Graph
import pandas as pd

ONTOLOGY_DIR = Path("/content/reviewer_evidence/ontologies")

print("Ontology directory exists:", ONTOLOGY_DIR.exists())

if ONTOLOGY_DIR.exists():
    print("\nFiles found:")
    for f in ONTOLOGY_DIR.iterdir():
        if f.is_file():
            print(f" - {f.name} | {f.stat().st_size:,} bytes")

ontology_files = {
    "MSCDO": ONTOLOGY_DIR / "MSCDO.owl",
    "ICPS": ONTOLOGY_DIR / "ICPS.owl"
}

ontology_diagnostics = []

for name, path in ontology_files.items():

    print(f"\n{'='*55}")
    print("ONTOLOGY:", name)

    if not path.exists():
        print("STATUS: FILE NOT FOUND")
        ontology_diagnostics.append({
            "ontology": name,
            "status": "File not found",
            "triples": 0,
            "format": None
        })
        continue

    content = path.read_bytes()

    print("File size:", len(content), "bytes")
    print("First 300 bytes:")
    print(content[:300].decode("utf-8", errors="replace"))

    parsed = False

    # Try common RDF serialization formats
    for rdf_format in ["xml", "turtle", "json-ld", "nt", "n3"]:

        try:
            graph = Graph()
            graph.parse(data=content, format=rdf_format)

            triple_count = len(graph)

            if triple_count > 0:
                print(f"\nSUCCESS: Parsed as {rdf_format}")
                print("Number of RDF triples:", triple_count)

                ontology_diagnostics.append({
                    "ontology": name,
                    "status": "Successfully parsed",
                    "triples": triple_count,
                    "format": rdf_format
                })

                parsed = True
                break

        except Exception:
            continue

    if not parsed:
        print("\nFAILED: No RDF format could parse this file.")

        ontology_diagnostics.append({
            "ontology": name,
            "status": "Not valid or unsupported RDF",
            "triples": 0,
            "format": None
        })

display(pd.DataFrame(ontology_diagnostics))

Ontology directory exists: True

Files found:

ONTOLOGY: MSCDO
STATUS: FILE NOT FOUND

ONTOLOGY: ICPS
STATUS: FILE NOT FOUND


,ontology,status,triples,format
0,MSCDO,File not found,0,None
1,ICPS,File not found,0,None


In [41]:

# CELL 6 — Combine and normalise MSCDO and ICPS ontology terms

import pandas as pd
import re

# Inspect the existing ontology DataFrames
print("Input DataFrame:", combined_terms.shape)
print("Input columns:", combined_terms.columns.tolist())

# Identify the preferred-label column
possible_label_columns = [
    "preferred_label",
    "label",
    "name",
    "class_label",
    "class_name",
    "term",
    "rdfs_label"
]

label_column = next(
    (col for col in possible_label_columns if col in combined_terms.columns),
    None
)

if combined_terms.empty:
    raise ValueError(
        "combined_terms is empty. Please check Cells 3–5: "
        "the ontology terms were not extracted."
    )

if label_column is None:
    raise ValueError(
        "No recognised label column was found. "
        f"Available columns: {combined_terms.columns.tolist()}. "
        "Please share the output of the diagnostic cell."
    )

# Standardise the label column
combined_terms = combined_terms.copy()
combined_terms["preferred_label"] = (
    combined_terms[label_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Normalise labels for comparison
def normalise_term(value):
    value = str(value).lower().strip()
    value = re.sub(r"[_\-]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value

combined_terms["normalised_label"] = (
    combined_terms["preferred_label"].apply(normalise_term)
)

# Remove empty labels and duplicate terms
combined_terms = combined_terms[
    combined_terms["normalised_label"] != ""
].drop_duplicates(
    subset=["ontology", "normalised_label"]
).reset_index(drop=True)

print("\nCombined ontology terms:", len(combined_terms))
print("\nTerms by ontology:")

if "ontology" in combined_terms.columns:
    display(combined_terms.groupby("ontology").size().to_frame("term_count"))

display(combined_terms.head(10))

# Save the extracted terms
output_path = (
    "/content/reviewer_evidence/results/"
    "combined_ontology_terms.csv"
)

combined_terms.to_csv(output_path, index=False)
print("\nSaved:", output_path)

Input DataFrame: (0, 0)
Input columns: []


ValueError: combined_terms is empty. Please check Cells 3–5: the ontology terms were not extracted.

## 6. Validate offsets against original and translated text

In [ ]:
checks=[]
for version,tc,ec in [("original",C["text"],C["entity"]),("translated",C["translation"],C["translation_entity"] or C["entity"])]:
    if tc is None: continue
    n=ok=0
    for _,r in raw.iterrows():
        vals=[r.get(C["start"]),r.get(C["end"]),r.get(ec),r.get(tc)]
        if any(pd.isna(v) for v in vals): continue
        try: a,b=int(vals[0]),int(vals[1])
        except: continue
        n+=1; ok+=int(0<=a<b<=len(str(vals[3])) and str(vals[3])[a:b]==str(vals[2]))
    checks.append({"text_version":version,"checked":n,"exact_matches":ok,"match_rate":ok/n if n else np.nan})
offset_audit=pd.DataFrame(checks); display(offset_audit)
best=offset_audit.sort_values("match_rate",ascending=False).iloc[0]["text_version"]
TEXT=C["text"] if best=="original" else C["translation"]
ENTITY=C["entity"] if best=="original" else (C["translation_entity"] or C["entity"])
print("Best offset alignment:",best)
if offset_audit.iloc[0]["match_rate"] < 0.8:
    print("WARNING: Low alignment. Do not report exact-span metrics until the offset convention is resolved.")

,text_version,checked,exact_matches,match_rate
0,original,3125,3125,1.0000
1,translated,3125,10,0.0032


Best offset alignment: original


## 7. Build one record per report using only exact verified spans

In [ ]:
def make_records(df):
    out={}; stat=Counter()
    for _,r in df.iterrows():
        if any(pd.isna(r[x]) for x in [C["id"],TEXT,ENTITY,C["start"],C["end"],C["type"]]):
            stat["missing"]+=1; continue
        rid=str(r[C["id"]]).strip(); txt=str(r[TEXT]); ent=str(r[ENTITY])
        try: a,b=int(r[C["start"]]),int(r[C["end"]])
        except: stat["bad_offset"]+=1; continue
        if not (0<=a<b<=len(txt) and txt[a:b]==ent):
            stat["offset_mismatch"]+=1; continue
        if rid not in out: out[rid]={"doc_id":rid,"text":txt,"entities":[]}
        elif out[rid]["text"]!=txt: stat["id_text_conflict"]+=1; continue
        out[rid]["entities"].append({"start":a,"end":b,"label":str(r[C["type"]]).strip()})
        stat["valid_span_rows"]+=1
    recs=[]
    for x in out.values():
        x["entities"]=list({(e["start"],e["end"],e["label"]):e for e in x["entities"]}.values())
        recs.append(x)
    return recs,stat
records,stats=make_records(raw)
print(dict(stats),"\nReports:",len(records),"Unique spans:",sum(len(x["entities"]) for x in records))
if len(records)<2: raise ValueError("Insufficient valid reports after offset checking.")

{'valid_span_rows': 3125} 
Reports: 517 Unique spans: 3124


## 8. Duplicate-text audit and export JSONL

In [ ]:
hashes=defaultdict(list)
for r in records: hashes[hashlib.sha256(r["text"].encode()).hexdigest()].append(r["doc_id"])
dup=[v for v in hashes.values() if len(v)>1]
OUT=Path("/content/med_error_outputs"); OUT.mkdir(exist_ok=True)
def write_jsonl(path,rows):
    with open(path,"w",encoding="utf-8") as f:
        for x in rows: f.write(json.dumps(x,ensure_ascii=False)+"\n")
write_jsonl(OUT/"audited_records.jsonl",records)
label_counts.to_csv(OUT/"entity_label_counts.csv",index=False)
offset_audit.to_csv(OUT/"offset_audit.csv",index=False)
audit={"workbook_sheet":str(CROSS_KEY),"annotation_rows":len(raw),"unique_report_ids":int(ids.nunique()),
       "offset_alignment":best,"offset_audit":checks,"record_build_stats":dict(stats),
       "retained_reports":len(records),"retained_spans":sum(len(x["entities"]) for x in records),
       "exact_duplicate_text_groups":len(dup),"label_counts":label_counts.to_dict("records")}
(OUT/"dataset_audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2))
print("Duplicate text groups:",len(dup),"Outputs:",list(OUT.iterdir()))

Duplicate text groups: 0 Outputs: [PosixPath('/content/med_error_outputs/offset_audit.csv'), PosixPath('/content/med_error_outputs/entity_label_counts.csv'), PosixPath('/content/med_error_outputs/dataset_audit.json'), PosixPath('/content/med_error_outputs/audited_records.jsonl')]


## 9. Exploratory report-level split — not an official test set

In [ ]:
arr=np.arange(len(records)); splitter=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42)
tr,te=next(splitter.split(arr,groups=[r["doc_id"] for r in records]))
train=[records[i] for i in tr]; test=[records[i] for i in te]
assert not ({x["doc_id"] for x in train}&{x["doc_id"] for x in test})
write_jsonl(OUT/"exploratory_train.jsonl",train); write_jsonl(OUT/"exploratory_test.jsonl",test)
print("Train reports:",len(train),"Test reports:",len(test))
print("CAUTION: use source-defined folds/official manual gold samples if specified by the dataset documentation.")

Train reports: 413 Test reports: 104
CAUTION: use source-defined folds/official manual gold samples if specified by the dataset documentation.


## 10. Optional TF-IDF + logistic-regression baseline for report-level `errorlabel`

In [ ]:
print("C dictionary:")
print(C)

print("\nColumns in q:")
print(q.columns.tolist())

print("\nFirst 5 rows:")
display(q.head())

print("\nData types:")
print(q.dtypes)

C dictionary:
{'id': 'id', 'text': 'report', 'translation': 'translated_report', 'entity': 'namedentity', 'translation_entity': 'translated_namedentity', 'start': 'startidx', 'end': 'endidx', 'type': 'entitytype', 'error': 'errorlabel'}

Columns in q:
['id', 'report', 'errorlabel']

First 5 rows:


,id,report,errorlabel
111,H073FE2A14A46DC1C,ベタニス50mgを調剤しなければならないところを、ベシケア5mgを調剤してしまった。,IN
174,H0C3757CC72AAA193,カルボシステインを調剤するところを、カロナールを調剤してしまった。,IN
313,H19DF716D12A1C963,KCL20mEqのところ、アスパラギン酸K10mEqで調剤してしまった。病棟にて、投与前にN...,IN
317,H1A1FEDE43D660E0C,退院時処方のリバスタッチを、間違えてニュープロを渡したと薬剤部より報告あり。実際に使用し、患...,IN
381,H23C96D8EF92654EC,アキネトンと間違い、フルマゼニル(先発品アネキセート)を指示し投与した。,IN



Data types:
id            object
report        object
errorlabel    object
dtype: object


In [ ]:
print("C mapping:")
print(C)

print("\nActual q columns:")
print(q.columns.tolist())

print("\nSample records:")
for _, row in q.head(5).iterrows():
    print("ID:", row["id"])
    print("REPORT:", repr(row["report"]))
    print("ERROR LABEL:", repr(row["errorlabel"]))
    print("-" * 70)

C mapping:
{'id': 'id', 'text': 'report', 'translation': 'translated_report', 'entity': 'namedentity', 'translation_entity': 'translated_namedentity', 'start': 'startidx', 'end': 'endidx', 'type': 'entitytype', 'error': 'errorlabel'}

Actual q columns:
['id', 'report', 'errorlabel']

Sample records:
ID: H073FE2A14A46DC1C
REPORT: 'ベタニス50mgを調剤しなければならないところを、ベシケア5mgを調剤してしまった。'
ERROR LABEL: 'IN'
----------------------------------------------------------------------
ID: H0C3757CC72AAA193
REPORT: 'カルボシステインを調剤するところを、カロナールを調剤してしまった。'
ERROR LABEL: 'IN'
----------------------------------------------------------------------
ID: H19DF716D12A1C963
REPORT: 'KCL20mEqのところ、アスパラギン酸K10mEqで調剤してしまった。病棟にて、投与前にNsが気付いた。'
ERROR LABEL: 'IN'
----------------------------------------------------------------------
ID: H1A1FEDE43D660E0C
REPORT: '退院時処方のリバスタッチを、間違えてニュープロを渡したと薬剤部より報告あり。実際に使用し、患者が傾眠になり心配になった家族からの連絡で発覚。'
ERROR LABEL: 'IN'
----------------------------------------------------------------------
ID: H23C96D8E

In [ ]:

# Cell 10 — TF-IDF + Logistic Regression baseline

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Use the actual column names from q
text_col = C["text"]
label_col = C["error"]
id_col = C["id"]

print("Text column:", text_col)
print("Label column:", label_col)
print("ID column:", id_col)

# Prepare data
baseline_df = q[[id_col, text_col, label_col]].copy()

baseline_df[text_col] = baseline_df[text_col].fillna("").astype(str).str.strip()
baseline_df[label_col] = baseline_df[label_col].fillna("").astype(str).str.strip()

baseline_df = baseline_df[
    (baseline_df[text_col] != "") &
    (baseline_df[label_col] != "")
].drop_duplicates(subset=[id_col]).reset_index(drop=True)

X = baseline_df[text_col]
y = baseline_df[label_col]
groups = baseline_df[id_col]

print("\nTotal reports:", len(baseline_df))
print("\nLabel distribution:")
print(y.value_counts())

if y.nunique() < 2:
    raise ValueError("Only one class is present. Classification is not possible.")

# Group-aware split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("\nTraining reports:", len(X_train))
print("Test reports:", len(X_test))

print("\nTraining labels:")
print(y_train.value_counts())

# Model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        max_df=1.0,
        max_features=30000,
        sublinear_tf=True,
        strip_accents="unicode",
        token_pattern=r"(?u)\b\w+\b"
    )),
    ("lr", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train and evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("\nAccuracy:", round(accuracy_score(y_test, pred), 4))

print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))

# Save predictions
baseline_results = pd.DataFrame({
    "report_id": groups.iloc[test_idx].values,
    "true_label": y_test.values,
    "predicted_label": pred
})

baseline_results.to_csv(
    "/content/tfidf_logistic_regression_predictions.csv",
    index=False
)

print("\nPredictions saved.")

Text column: report
Label column: errorlabel
ID column: id

Total reports: 46

Label distribution:
errorlabel
IN    40
IA     6
Name: count, dtype: int64

Training reports: 36
Test reports: 10

Training labels:
errorlabel
IN    33
IA     3
Name: count, dtype: int64

Accuracy: 0.7

Classification report:
              precision    recall  f1-score   support

          IA       0.00      0.00      0.00         3
          IN       0.70      1.00      0.82         7

    accuracy                           0.70        10
   macro avg       0.35      0.50      0.41        10
weighted avg       0.49      0.70      0.58        10


Predictions saved.


## 11. Create label-mapping worksheet and download outputs

In [ ]:
mapping=pd.DataFrame({"source_entity_type":sorted({e["label"] for r in records for e in r["entities"]}),
                        "normalized_label_to_agree":"","definition":"","mapping_decision_notes":""})
mapping.to_csv(OUT/"label_mapping_template.csv",index=False)
bundle=Path("/content/med_error_outputs.zip")
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file(): z.write(f,arcname=f.name)
files.download(str(bundle))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Stop conditions before model training or manuscript claims
- Verify dataset paper/codebook and whether CrossValid is development data or a prescribed validation fold.
- Inspect all manual-review workbooks and establish whether their reports overlap with CrossValid.
- Use manually adjudicated annotations as gold evaluation only if annotation provenance and adjudication are documented.
- The current dataset sample does not automatically provide a large weak-supervision training corpus.
- Do not claim LoRA, ontology, knowledge-graph, BioBERT, ClinicalBERT, RAG, causal, or cross-dataset results until each is implemented and run on a common valid split.
- The random split and TF-IDF result are exploratory pipeline checks, not a substitute for the dataset's official validation design.

In [ ]:
# ============================================================
# CELL 19 — AUTHORITATIVE DATASET, CATEGORY AND OFFSET RECONCILIATION
# ============================================================
import os, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if "main_df" not in globals() or not isinstance(main_df, pd.DataFrame):
    candidates = ["/content/med_error_data/AnnotatedDataset.xlsx", "/content/AnnotatedDataset.xlsx"]
    source = next((x for x in candidates if os.path.exists(x)), None)
    if source is None:
        raise FileNotFoundError("Main dataset not found. Run the existing dataset-loading cells first.")
    main_df = pd.read_excel(source, sheet_name=0)
    main_df.columns = main_df.columns.astype(str).str.strip().str.lower()

required = ["id", "report", "namedentity", "startidx", "endidx", "entitytype"]
missing = [c for c in required if c not in main_df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}. Available: {main_df.columns.tolist()}")

d19 = main_df.copy()
d19["id"] = d19["id"].astype(str).str.strip()
d19["entitytype"] = d19["entitytype"].fillna("").astype(str).str.strip()
d19["report_norm"] = d19["report"].fillna("").astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
d19["entity_norm"] = d19["namedentity"].fillna("").astype(str).str.strip()

category_counts_19 = (
    d19.groupby("entitytype", dropna=False)
       .agg(annotation_rows=("id", "size"), unique_reports=("id", "nunique"),
            unique_surface_forms=("entity_norm", "nunique"))
       .reset_index()
       .sort_values("annotation_rows", ascending=False)
)
category_counts_19.to_csv(RESULTS_DIR / "cell19_authoritative_category_counts.csv", index=False, encoding="utf-8-sig")

duplicate_rows = int(d19.duplicated(subset=required, keep="first").sum())
unique_ids = int(d19["id"].nunique())
unique_texts = int(d19.loc[d19["report_norm"] != "", "report_norm"].nunique())
id_text_conflicts = int((d19.groupby("id")["report_norm"].nunique() > 1).sum())
text_id_groups = int((d19.groupby("report_norm")["id"].nunique() > 1).sum())

offset_rows = []
for convention in ["zero_based_end_exclusive", "one_based_end_exclusive"]:
    checked = matched = 0
    for row in d19[["report", "namedentity", "startidx", "endidx"]].itertuples(index=False, name=None):
        text, entity, start, end = row
        if pd.isna(text) or pd.isna(entity) or pd.isna(start) or pd.isna(end):
            continue
        try:
            a, b = int(start), int(end)
        except Exception:
            continue
        if convention == "one_based_end_exclusive":
            a, b = a - 1, b - 1
        txt, ent = str(text), str(entity)
        checked += 1
        if 0 <= a < b <= len(txt) and txt[a:b] == ent:
            matched += 1
    offset_rows.append({"convention": convention, "checked_rows": checked,
                        "exact_span_matches": matched,
                        "match_rate": matched / checked if checked else np.nan})
offset_audit_19 = pd.DataFrame(offset_rows)
offset_audit_19.to_csv(RESULTS_DIR / "cell19_offset_convention_audit.csv", index=False)

summary_19 = pd.DataFrame([
    ["annotation_rows", len(d19)],
    ["unique_report_ids", unique_ids],
    ["unique_normalized_report_texts", unique_texts],
    ["entity_categories_including_blank", int(d19["entitytype"].nunique(dropna=False))],
    ["exact_duplicate_annotation_rows", duplicate_rows],
    ["report_ids_with_multiple_texts", id_text_conflicts],
    ["normalized_texts_associated_with_multiple_ids", text_id_groups],
    ["historical_count_7886", "Not independently attributable without its source/filter definition"],
    ["historical_count_8742", "Not independently attributable without its source/filter definition"]
], columns=["measure", "value"])
summary_19.to_csv(RESULTS_DIR / "cell19_dataset_reconciliation_summary.csv", index=False, encoding="utf-8-sig")

print("CELL 19 COMPLETE")
display(summary_19)
display(category_counts_19)
display(offset_audit_19)
print("The 7,886 and 8,742 historical values are not assumed to be comparable counts; their original derivations are needed to reconcile them.")


In [ ]:
# ============================================================
# CELL 20 — ANNOTATOR INDEPENDENCE AND IAA DATA AVAILABILITY AUDIT
# ============================================================
import os, glob, re
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _norm_col20(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

candidate_files20 = set(glob.glob("/content/**/*.xlsx", recursive=True) + glob.glob("/content/**/*.xls", recursive=True))
candidate_files20.update(str(x) for x in globals().get("zip_excel_paths", []) if os.path.isfile(str(x)))
candidate_files20.update(str(x) for x in globals().get("validation_files", []) if os.path.isfile(str(x)))

annotator_terms20 = ("annotator", "reviewer", "coder", "rater", "annotatedby", "annotatorid", "reviewerid")
audit_rows20 = []
for fp in sorted(candidate_files20):
    try:
        xls = pd.ExcelFile(fp)
        for sh in xls.sheet_names:
            try:
                head = pd.read_excel(fp, sheet_name=sh, nrows=5)
                cols = list(head.columns)
                possible = [str(c) for c in cols if any(t in _norm_col20(c) for t in annotator_terms20)]
                audit_rows20.append({
                    "file": os.path.basename(fp), "sheet": sh,
                    "columns": " | ".join(map(str, cols)),
                    "possible_annotator_identity_columns": " | ".join(possible),
                    "has_explicit_annotator_identity": bool(possible),
                    "rows_sampled_for_schema": len(head)
                })
            except Exception as e:
                audit_rows20.append({"file": os.path.basename(fp), "sheet": sh, "columns": "",
                                     "possible_annotator_identity_columns": "", "has_explicit_annotator_identity": False,
                                     "rows_sampled_for_schema": 0, "read_error": str(e)[:250]})
    except Exception:
        continue

iaa_schema_audit20 = pd.DataFrame(audit_rows20)
iaa_schema_audit20.to_csv(RESULTS_DIR / "cell20_annotator_schema_audit.csv", index=False, encoding="utf-8-sig")

iaa_template20 = pd.DataFrame(columns=[
    "report_id", "text", "annotator_id", "start", "end", "entity_type",
    "annotation_round", "adjudicated", "adjudicated_start", "adjudicated_end",
    "adjudicated_entity_type", "notes"
])
iaa_template20.to_excel(RESULTS_DIR / "cell20_independent_annotation_collection_template.xlsx", index=False)

explicit_identity = bool(len(iaa_schema_audit20) and iaa_schema_audit20["has_explicit_annotator_identity"].any())
status20 = "Potential annotator identity field found; inspect source workbook and verify independent annotation protocol." if explicit_identity else "IAA NOT CALCULABLE from detected workbook structure: no explicit annotator identity field was found."
pd.DataFrame([{
    "status": status20,
    "workbooks_sheets_inspected": len(iaa_schema_audit20),
    "sheets_with_possible_annotator_identity": int(iaa_schema_audit20["has_explicit_annotator_identity"].sum()) if len(iaa_schema_audit20) else 0,
    "interpretation": "ManualReviewed versus MachineAnnotated is not inter-annotator agreement. Independent human labels and annotator identities are required."
}]).to_csv(RESULTS_DIR / "cell20_IAA_status.csv", index=False, encoding="utf-8-sig")

print("CELL 20 COMPLETE")
print(status20)
display(iaa_schema_audit20.head(30))
print("Template saved:", RESULTS_DIR / "cell20_independent_annotation_collection_template.xlsx")


In [ ]:
# ============================================================
# CELL 21 — JAPANESE RULE-BASED / CHARACTER-STRING ENTITY BASELINE
# Uses training reports only; exact-span evaluation on held-out reports.
# ============================================================
import os, re, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if "main_df" not in globals() or "split_assignments" not in globals():
    raise RuntimeError("Run the existing dataset-loading and report-level split cells before Cell 21.")

d21 = main_df.copy()
d21.columns = d21.columns.astype(str).str.strip().str.lower()
d21["id"] = d21["id"].astype(str).str.strip()
split21 = split_assignments[["id", "split"]].copy()
split21["id"] = split21["id"].astype(str).str.strip()
d21 = d21.merge(split21.drop_duplicates("id"), on="id", how="inner", validate="many_to_one")

if "offset_audit_19" not in globals():
    raise RuntimeError("Run Cell 19 first.")
best_conv21 = offset_audit_19.sort_values("match_rate", ascending=False).iloc[0]["convention"]
offset_shift21 = 1 if best_conv21 == "one_based_end_exclusive" else 0

def _valid_entity21(row):
    try:
        a, b = int(row["startidx"]) - offset_shift21, int(row["endidx"]) - offset_shift21
        txt, ent = str(row["report"]), str(row["namedentity"])
        return bool(ent.strip()) and 0 <= a < b <= len(txt) and txt[a:b] == ent
    except Exception:
        return False

valid21 = d21.apply(_valid_entity21, axis=1)
gold21 = d21.loc[valid21].copy()
gold21["start"] = gold21["startidx"].astype(int) - offset_shift21
gold21["end"] = gold21["endidx"].astype(int) - offset_shift21
gold21["entity_type"] = gold21["entitytype"].astype(str).str.strip()
gold21 = gold21[(gold21["entity_type"] != "") & (gold21["start"] >= 0) & (gold21["end"] > gold21["start"])]

train21 = gold21[gold21["split"] == "train"]
test21 = gold21[gold21["split"] == "test"]
if train21.empty or test21.empty:
    raise RuntimeError("No valid training or test spans after offset validation. Inspect Cell 19 offset audit.")

term_label_counts21 = train21.groupby(["namedentity", "entity_type"]).size().reset_index(name="count")
term_label_counts21["term"] = term_label_counts21["namedentity"].astype(str).str.strip()
term_label_counts21 = term_label_counts21[(term_label_counts21["term"].str.len() >= 2) & (term_label_counts21["term"] != "")]
term_label_counts21 = term_label_counts21.sort_values(["term", "count"], ascending=[True, False])
term_to_label21 = {}
for term, grp in term_label_counts21.groupby("term", sort=False):
    term_to_label21[str(term)] = str(grp.iloc[0]["entity_type"])

freq21 = train21["namedentity"].astype(str).value_counts()
ordered_terms21 = sorted(
    [t for t in term_to_label21 if t in freq21.index],
    key=lambda t: (int(freq21.get(t, 0)), len(t)),
    reverse=True
)[:20000]
term_to_label21 = {t: term_to_label21[t] for t in ordered_terms21}
if not term_to_label21:
    raise RuntimeError("Training split produced no usable entity terms for the dictionary baseline.")

pattern21 = re.compile("|".join(re.escape(t) for t in sorted(term_to_label21, key=len, reverse=True)))
report_text21 = d21[["id", "report", "split"]].drop_duplicates("id").set_index("id")

pred_rows21 = []
for rid, row in report_text21[report_text21["split"] == "test"].iterrows():
    text = "" if pd.isna(row["report"]) else str(row["report"])
    for match in pattern21.finditer(text):
        pred_rows21.append({
            "model": "Japanese_training_lexicon_rule_baseline",
            "report_id": str(rid), "start": int(match.start()), "end": int(match.end()),
            "entity_type": term_to_label21[match.group(0)],
            "predicted_text": match.group(0), "split": "test"
        })

pred21 = pd.DataFrame(pred_rows21, columns=[
    "model", "report_id", "start", "end", "entity_type", "predicted_text", "split"
])
gold_rows21 = gold21[gold21["split"] == "test"][["id", "start", "end", "entity_type", "namedentity"]].copy()
gold_rows21 = gold_rows21.rename(columns={"id": "report_id", "namedentity": "gold_text"})
pred21.to_csv(RESULTS_DIR / "cell21_japanese_rule_baseline_predictions.csv", index=False, encoding="utf-8-sig")
gold_rows21.to_csv(RESULTS_DIR / "cell21_test_gold_spans.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "baseline": "Japanese-compatible exact surface-form dictionary",
    "training_reports": int(report_text21[report_text21["split"] == "train"].shape[0]),
    "test_reports": int(report_text21[report_text21["split"] == "test"].shape[0]),
    "training_unique_terms": len(term_to_label21),
    "test_gold_spans": len(gold_rows21),
    "test_predicted_spans": len(pred21),
    "offset_convention": best_conv21,
    "limitations": "Lexicon matching baseline only; not a neural Japanese NER baseline. Terms are learned from training annotations only."
}]).to_csv(RESULTS_DIR / "cell21_baseline_run_summary.csv", index=False, encoding="utf-8-sig")

print("CELL 21 COMPLETE")
display(pd.read_csv(RESULTS_DIR / "cell21_baseline_run_summary.csv"))
print("Predictions:", RESULTS_DIR / "cell21_japanese_rule_baseline_predictions.csv")


In [ ]:
# ============================================================
# CELL 22 — ONTOLOGY / TERMINOLOGY / LoRA / RAG ABLATION READINESS
# Does not fabricate ablation results when model pipelines are absent.
# ============================================================
import os, glob
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_patterns22 = ["*.safetensors", "*.bin", "*.pt", "*.pth", "*.ckpt"]
checkpoint_files22 = []
for pat in checkpoint_patterns22:
    checkpoint_files22.extend(glob.glob("/content/**/" + pat, recursive=True))
checkpoint_files22 = sorted(set(checkpoint_files22))

prediction_files22 = sorted(set(
    glob.glob(str(RESULTS_DIR / "*prediction*.csv")) +
    glob.glob("/content/*prediction*.csv")
))
candidate_functions22 = [
    name for name, obj in globals().items()
    if callable(obj) and any(k in name.lower() for k in ["train", "lora", "rag", "retriev", "generate", "predict"])
]
ablation_rows22 = [
    ("A0", "Base model without LoRA, ontology or terminology", "Not run", "No verified base-model training/inference function identified in the notebook runtime."),
    ("A1", "Base model + LoRA", "Not run", "No verified LoRA training/inference pipeline identified."),
    ("A2", "Base model + terminology", "Not run", "Terminology ablation requires a model pipeline and controlled terminology injection."),
    ("A3", "Base model + ontology constraints", "Not run", "Ontology extraction exists; constraint integration into model inference is not implemented here."),
    ("A4", "LoRA + terminology", "Not run", "Requires reproducible model training and controlled configuration."),
    ("A5", "LoRA + ontology", "Not run", "Requires reproducible model training and ontology integration."),
    ("A6", "Full model: LoRA + terminology + ontology", "Not run", "No verified full-model inference pipeline identified."),
    ("A7", "Full model without RAG", "Not run", "Requires an implemented RAG-capable system and a controlled no-RAG condition."),
    ("A8", "Full model with RAG", "Not run", "No retrieval/indexing/generation pipeline identified.")
]
ablation_matrix22 = pd.DataFrame(ablation_rows22, columns=["experiment_id", "condition", "status", "reason_or_requirement"])
ablation_matrix22.to_csv(RESULTS_DIR / "cell22_ablation_status_matrix.csv", index=False, encoding="utf-8-sig")

ablation_template22 = pd.DataFrame(columns=[
    "experiment_id", "model_checkpoint", "base_model", "tokenizer",
    "lora_config", "terminology_source", "ontology_version",
    "rag_enabled", "retrieval_corpus", "seed", "split",
    "precision", "recall", "micro_f1", "macro_f1", "run_log"
])
ablation_template22.to_csv(RESULTS_DIR / "cell22_ablation_results_template.csv", index=False, encoding="utf-8-sig")

print("CELL 22 COMPLETE — readiness audit, not a model ablation.")
print("Candidate checkpoint files:", len(checkpoint_files22))
print("Prediction CSV files:", len(prediction_files22))
print("Candidate training/prediction functions:", candidate_functions22)
display(ablation_matrix22)


In [ ]:
# ============================================================
# CELL 23 — STRICT METRICS, REPORT-LEVEL BOOTSTRAP CI AND PAIRED TESTS
# Uses the held-out test gold and prediction CSVs available in RESULTS_DIR.
# ============================================================
import os, glob, json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy.stats import binomtest

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

gold_path23 = RESULTS_DIR / "cell21_test_gold_spans.csv"
if not gold_path23.exists():
    raise RuntimeError("Run Cell 21 first; test gold span file is missing.")
gold23 = pd.read_csv(gold_path23)
gold23["report_id"] = gold23["report_id"].astype(str)
gold_by23 = defaultdict(set)
for r in gold23.itertuples(index=False):
    gold_by23[str(r.report_id)].add((int(r.start), int(r.end), str(r.entity_type)))

def _score_sets23(gold_map, pred_map, ids):
    tp = fp = fn = 0
    for rid in ids:
        g, p = gold_map.get(rid, set()), pred_map.get(rid, set())
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall, "micro_f1": f1}

def _load_pred23(path):
    p = pd.read_csv(path)
    required_p = {"report_id", "start", "end", "entity_type"}
    if not required_p.issubset(p.columns):
        return None
    out = defaultdict(set)
    for r in p.itertuples(index=False):
        try:
            out[str(r.report_id)].add((int(r.start), int(r.end), str(r.entity_type)))
        except Exception:
            continue
    return out

prediction_paths23 = sorted(set(
    glob.glob(str(RESULTS_DIR / "*prediction*.csv")) +
    glob.glob("/content/*prediction*.csv")
))
prediction_paths23 = [x for x in prediction_paths23 if "template" not in x.lower()]
all_ids23 = sorted(set(gold_by23))
metric_rows23, prediction_maps23 = [], {}

for path in prediction_paths23:
    pred_map = _load_pred23(path)
    if pred_map is None:
        continue
    model_name = Path(path).stem
    prediction_maps23[model_name] = pred_map
    m = _score_sets23(gold_by23, pred_map, all_ids23)
    rng = np.random.default_rng(42)
    boot = []
    for _ in range(1000):
        sampled = rng.choice(all_ids23, size=len(all_ids23), replace=True)
        bg, bp = {}, {}
        for j, rid in enumerate(sampled):
            new_id = f"b{j}"
            bg[new_id] = gold_by23.get(str(rid), set())
            bp[new_id] = pred_map.get(str(rid), set())
        boot.append(_score_sets23(bg, bp, list(bg))["micro_f1"])
    metric_rows23.append({
        "model": model_name, **m,
        "bootstrap_ci_lower": float(np.percentile(boot, 2.5)),
        "bootstrap_ci_upper": float(np.percentile(boot, 97.5)),
        "bootstrap_iterations": 1000,
        "reports_evaluated": len(all_ids23),
        "prediction_file": path
    })

metrics23 = pd.DataFrame(metric_rows23)
metrics23.to_csv(RESULTS_DIR / "cell23_model_metrics_with_95CI.csv", index=False, encoding="utf-8-sig")

paired_rows23 = []
model_names23 = list(prediction_maps23)
for i in range(len(model_names23)):
    for j in range(i + 1, len(model_names23)):
        a_name, b_name = model_names23[i], model_names23[j]
        a, b = prediction_maps23[a_name], prediction_maps23[b_name]
        a_only = b_only = both_correct = both_wrong = 0
        for rid in all_ids23:
            g = gold_by23.get(rid, set())
            ac, bc = a.get(rid, set()) == g, b.get(rid, set()) == g
            if ac and bc: both_correct += 1
            elif ac: a_only += 1
            elif bc: b_only += 1
            else: both_wrong += 1
        discordant = a_only + b_only
        pvalue = float(binomtest(min(a_only, b_only), discordant, 0.5).pvalue) if discordant else 1.0
        paired_rows23.append({
            "model_a": a_name, "model_b": b_name, "both_exact_correct": both_correct,
            "a_only_exact_correct": a_only, "b_only_exact_correct": b_only,
            "both_not_exact_correct": both_wrong, "exact_binomial_p_value": pvalue,
            "test": "Exact paired test on report-level exact-set correctness; exploratory"
        })
paired23 = pd.DataFrame(paired_rows23)
paired23.to_csv(RESULTS_DIR / "cell23_paired_report_comparisons.csv", index=False, encoding="utf-8-sig")

print("CELL 23 COMPLETE")
if metrics23.empty:
    print("No usable prediction files were found; metric and CI tables are empty by design.")
else:
    display(metrics23)
display(paired23)


In [ ]:
# ============================================================
# CELL 24 — EXTERNAL WORKBOOK AUDIT AND ELIGIBILITY CHECK
# Does not call an external workbook validated unless gold labels and predictions exist.
# ============================================================
import os, glob, re
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _key24(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def _find24(cols, aliases):
    lookup = {_key24(c): c for c in cols}
    for a in aliases:
        if _key24(a) in lookup:
            return lookup[_key24(a)]
    return None

files24 = set(glob.glob("/content/**/*.xlsx", recursive=True) + glob.glob("/content/**/*.xls", recursive=True))
files24.update(str(x) for x in globals().get("zip_excel_paths", []) if os.path.isfile(str(x)))
main_ids24 = set(main_df["id"].dropna().astype(str).str.strip()) if "main_df" in globals() else set()
main_texts24 = set(main_df["report"].dropna().astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()) if "main_df" in globals() else set()

external_rows24 = []
for fp in sorted(files24):
    base = os.path.basename(fp).lower()
    if not any(k in base for k in ["external", "crossvalid", "internalvalid", "error_analysis"]):
        continue
    try:
        xls = pd.ExcelFile(fp)
        for sh in xls.sheet_names:
            try:
                tmp = pd.read_excel(fp, sheet_name=sh)
                idc = _find24(tmp.columns, ["id", "report_id", "reportid", "no"])
                textc = _find24(tmp.columns, ["report", "freetext", "free_text", "translated_report"])
                entc = _find24(tmp.columns, ["namedentity", "entity", "entityname"])
                startc = _find24(tmp.columns, ["startidx", "start", "start_index"])
                endc = _find24(tmp.columns, ["endidx", "end", "end_index"])
                typec = _find24(tmp.columns, ["entitytype", "entity_type", "type", "label"])
                ids = set(tmp[idc].dropna().astype(str).str.strip()) if idc else set()
                texts = set(tmp[textc].dropna().astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()) if textc else set()
                complete = all([idc, textc, entc, startc, endc, typec])
                external_rows24.append({
                    "file": os.path.basename(fp), "sheet": sh, "rows": len(tmp),
                    "id_column": idc, "text_column": textc, "entity_column": entc,
                    "start_column": startc, "end_column": endc, "type_column": typec,
                    "unique_ids": len(ids), "overlap_main_ids": len(ids & main_ids24),
                    "unique_texts": len(texts), "overlap_main_texts": len(texts & main_texts24),
                    "has_complete_span_gold_schema": complete,
                    "eligibility": "Candidate for external evaluation; verify provenance and gold annotation protocol" if complete else "Audit only: required gold span columns not all present"
                })
            except Exception as e:
                external_rows24.append({"file": os.path.basename(fp), "sheet": sh, "eligibility": "Read error: " + str(e)[:180]})
    except Exception:
        continue

external_audit24 = pd.DataFrame(external_rows24)
external_audit24.to_csv(RESULTS_DIR / "cell24_external_validation_eligibility.csv", index=False, encoding="utf-8-sig")
print("CELL 24 COMPLETE")
if external_audit24.empty:
    print("No candidate validation workbooks found in the current Colab filesystem.")
else:
    display(external_audit24)
print("An overlap audit alone is not external validation; frozen model predictions on eligible external gold reports are required.")


In [ ]:
# ============================================================
# CELL 25 — CLINICAL EXPERT REVIEW PACKET
# Prepares a sample and scoring sheet; does not substitute for clinician review.
# ============================================================
import os, json
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if "main_df" not in globals() or "split_assignments" not in globals():
    raise RuntimeError("Run the dataset and split cells before Cell 25.")

d25 = main_df.copy()
d25["id"] = d25["id"].astype(str).str.strip()
sp25 = split_assignments[["id", "split"]].copy()
sp25["id"] = sp25["id"].astype(str).str.strip()
test_ids25 = set(sp25.loc[sp25["split"] == "test", "id"])
test_df25 = d25[d25["id"].isin(test_ids25)].copy()

reports25 = test_df25[["id", "report"]].drop_duplicates("id").copy()
reports25 = reports25[reports25["report"].notna()]
sample_n25 = min(100, len(reports25))
if sample_n25:
    sample25 = reports25.sample(n=sample_n25, random_state=42).sort_values("id")
else:
    sample25 = reports25

gold_group25 = (
    test_df25.groupby("id")
    .apply(lambda x: json.dumps([
        {"text": str(r.namedentity), "type": str(r.entitytype),
         "start": str(r.startidx), "end": str(r.endidx)}
        for r in x.itertuples(index=False)
        if pd.notna(r.namedentity) and pd.notna(r.entitytype)
    ], ensure_ascii=False))
    .rename("gold_annotations_json")
    .reset_index()
)
expert25 = sample25.merge(gold_group25, on="id", how="left")
expert25 = expert25.rename(columns={"id": "report_id", "report": "report_text"})
expert25["model_predictions_json"] = ""
expert25["expert_correctness_1_5"] = ""
expert25["clinical_relevance_1_5"] = ""
expert25["potential_patient_safety_impact"] = ""
expert25["missed_safety_information"] = ""
expert25["expert_id"] = ""
expert25["expert_comments"] = ""
expert25["review_date"] = ""
expert25.to_excel(RESULTS_DIR / "cell25_clinical_expert_review_packet.xlsx", index=False)

print("CELL 25 COMPLETE")
print("Expert review packet reports:", len(expert25))
print("Saved:", RESULTS_DIR / "cell25_clinical_expert_review_packet.xlsx")
print("Clinical evaluation remains pending until qualified experts complete and return the scoring sheet.")


In [ ]:
# ============================================================
# CELL 26 — REPRODUCIBILITY PACKAGE AND CONSOLIDATED REVIEWER EVIDENCE
# ============================================================
import os, sys, glob, json, hashlib, platform, zipfile, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/content/reviewer_evidence/results"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _sha256_26(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

dataset_path26 = "/content/med_error_data/AnnotatedDataset.xlsx"
if not os.path.exists(dataset_path26):
    dataset_path26 = "/content/AnnotatedDataset.xlsx"
dataset_info26 = {
    "dataset_path": dataset_path26 if os.path.exists(dataset_path26) else "Not found",
    "sha256": _sha256_26(dataset_path26) if os.path.exists(dataset_path26) else "",
    "python_version": sys.version,
    "platform": platform.platform(),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed_used_in_existing_split": 42,
    "split_unit": "normalized report text grouped at report level",
    "model_training_configuration": "Not recoverable from the inspected notebook: no verified LoRA training run/configuration was present."
}
packages26 = {}
for pkg in ["pandas", "numpy", "scikit-learn", "transformers", "torch", "peft", "datasets", "rdflib", "openpyxl", "scipy", "statsmodels"]:
    try: packages26[pkg] = importlib.metadata.version(pkg)
    except Exception: packages26[pkg] = "Not installed / version unavailable"
dataset_info26["package_versions"] = packages26
with open(RESULTS_DIR / "cell26_reproducibility_manifest.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info26, f, ensure_ascii=False, indent=2)

def _read_csv26(name):
    p = RESULTS_DIR / name
    try: return pd.read_csv(p) if p.exists() else pd.DataFrame()
    except Exception: return pd.DataFrame()

sheets26 = {
    "Dataset_summary": _read_csv26("cell19_dataset_reconciliation_summary.csv"),
    "Category_counts": _read_csv26("cell19_authoritative_category_counts.csv"),
    "Offset_audit": _read_csv26("cell19_offset_convention_audit.csv"),
    "IAA_status": _read_csv26("cell20_IAA_status.csv"),
    "Annotator_schema": _read_csv26("cell20_annotator_schema_audit.csv"),
    "Baseline_summary": _read_csv26("cell21_baseline_run_summary.csv"),
    "Baseline_metrics_CI": _read_csv26("cell23_model_metrics_with_95CI.csv"),
    "Paired_tests": _read_csv26("cell23_paired_report_comparisons.csv"),
    "Ablation_status": _read_csv26("cell22_ablation_status_matrix.csv"),
    "External_audit": _read_csv26("cell24_external_validation_eligibility.csv")
}
review_status26 = pd.DataFrame([
    ["Dataset/category counts and offset audit", "Completed computationally", "Cell 19 outputs"],
    ["Historical 7,886 vs 8,742 discrepancy", "Not attributable from current notebook alone", "Original derivation/filter definitions required"],
    ["Independent annotation / IAA", "Not established", "Requires at least two independent human annotators and identities"],
    ["Japanese-language rule-based baseline", "Run if Cell 21 completes", "Cell 21 predictions and summary"],
    ["Neural Japanese NER baseline", "Not run", "Requires a separately implemented and justified model"],
    ["Ontology / terminology / LoRA / RAG ablations", "Not run", "Cell 22 readiness matrix; actual model pipeline absent"],
    ["Confidence intervals / paired comparisons", "Computed only for available prediction files", "Cell 23 output; interpretation limited to evaluated predictions"],
    ["External generalisability", "Audit / eligibility assessment", "Frozen model predictions and independent gold required"],
    ["Clinical expert evaluation", "Preparation only", "Cell 25 review packet; expert responses required"],
    ["Reproducibility details", "Partially documented", "Environment and dataset fingerprint exported; missing training logs remain unavailable"],
    ["Introduction / related work / novelty / discussion", "Manual manuscript revision required", "Not a Colab computational task"]
], columns=["reviewer_concern", "evidence_status", "evidence_or_remaining_requirement"])
review_status26.to_csv(RESULTS_DIR / "cell26_final_reviewer_evidence_status.csv", index=False, encoding="utf-8-sig")
sheets26["Reviewer_status"] = review_status26

xlsx26 = RESULTS_DIR / "FINAL_Reviewer_Evidence_Package.xlsx"
with pd.ExcelWriter(xlsx26, engine="openpyxl") as writer:
    for name, frame in sheets26.items():
        frame.to_excel(writer, sheet_name=name[:31], index=False)
    pd.DataFrame([dataset_info26]).to_excel(writer, sheet_name="Reproducibility", index=False)

zip26 = Path("/content/FINAL_Reviewer_Evidence_Package.zip")
with zipfile.ZipFile(zip26, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in RESULTS_DIR.rglob("*"):
        if fp.is_file() and fp != xlsx26:
            z.write(fp, arcname=str(fp.relative_to(RESULTS_DIR)))
    if xlsx26.exists():
        z.write(xlsx26, arcname=xlsx26.name)

print("CELL 26 COMPLETE")
display(review_status26)
print("Excel evidence package:", xlsx26)
print("ZIP evidence package:", zip26)
print("Please download both files from the Colab Files panel. The package intentionally preserves unresolved human/model requirements instead of marking them complete.")
